# Short notebook 2 — `submission_nb2_v12c.csv`


**Email:** lians@stud.ntnu.no

The naming convention is shared with the report: **Pipeline A** supplies **A-LightGBM** and **A-TCN**. **Pipeline B (v1)** supplies **B1-XGBoost** and **B1-LightGBM**, combined equally as the **B1 ensemble**. A-LightGBM includes both pooled and per-unit LightGBM models; a TCN seed number denotes a separate training run.

This notebook trains everything from the raw competition files and writes the submission. No external data and no saved models are used.
The prediction is a rank average of three models:

| Part | Weight | Model |
|---|---|---|
| A-LightGBM | 0.55 | LightGBM (pooled + one per unit) on hydro features and three LP feature sets |
| B1 ensemble | 0.35 | XGBoost + LightGBM per unit on a 24-channel LP |
| A-TCN | 0.10 | One TCN shared by all units |

**Where the weights come from.** They were chosen on two validation years (2021 and 2022), with models trained only on cases that end before each year.
The result was stable for Pipeline B (v1) weights between 0.3 and 0.5. Validation AUC: 0.99574 (2021) and 0.99540 (2022).



## How to run
1. **Environment.** Python 3.13 with the packages in environment.yml. Create the environment with `conda env create -f environment.yml` and activate it with conda activate tdt4173. The file uses the conda-forge channel, so LightGBM, XGBoost and PyTorch share a single OpenMP runtime (llvm-openmp) and no extra libomp installation is needed on macOS. Jupyter is included in the environment.
2. **Files.** Put this notebook in one folder together with `data/` (with the sub-folders `kernel/` and `extended/`), `sample_submission.csv` and `prediction_mapping.csv`. No other file is needed.
3. **Run all cells in order.** In Jupyter there is no time limit. From the command line use
   `jupyter nbconvert --to notebook --execute "Short notebook 2.ipynb" --ExecutePreprocessor.timeout=-1`
   (the default limit of 30 seconds per cell would stop the run).
4. **Result.** The file `submission_nb2_v12c.csv` is written next to the notebook. Intermediate files go to `work_nb2/`.
5. **Time and memory.** About 2.6 hours on 2 CPU cores. The run was tested on a machine with 7 GB RAM.
6. **Restart.** A stage that has finished is skipped when the notebook is run again. For a clean reproduction delete `work_nb2/` first. Run the two short notebooks one after the other, not at the same time.

In [ ]:
import os, sys, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
PROJECT = Path(".").resolve()          # folder with data/, sample_submission.csv, prediction_mapping.csv
assert (PROJECT / "data/kernel/Unit_commitment_decisions.csv").exists(), "Run this notebook next to the data/ folder"
WORK = PROJECT / "work_nb2"; WORK.mkdir(exist_ok=True)     # all intermediate files go here

# Thread budget: use every logical CPU reported by the OS.
THREADS = str(max(1, os.cpu_count() or 1))
ENV = dict(os.environ, V12C_WORK=str(WORK), TDT4173_PROJECT=str(PROJECT), TDT4173_THREADS=THREADS, PYTHONPATH=str(WORK),
           OMP_NUM_THREADS=THREADS, MKL_NUM_THREADS=THREADS, OPENBLAS_NUM_THREADS=THREADS, TCN_SEEDS="4173,4174,4175",
           SINTEF_DATA_DIR=str(PROJECT), SINTEF_WORK_DIR=str(WORK / "legacy"), SINTEF_OUTPUT_DIR=str(WORK / "legacy_out"))

def run_stage(outputs, *commands):
    """Run the given scripts (in parallel if several). A stage is skipped only if its marker file exists:
    the marker is written after every script has finished with exit code 0 and all expected outputs exist.
    A file left behind by an interrupted run is therefore never taken as a finished result.
    Each stage is a separate process: this keeps memory low and keeps PyTorch and LightGBM apart.
    If one script fails, the others are stopped at once."""
    marker = WORK / (".done_" + "_".join("_".join(c).replace(".py", "") for c in commands))
    if marker.exists() and all((WORK / o).exists() for o in outputs):
        print("already done:", outputs); return
    marker.unlink(missing_ok=True)
    t0 = time.time(); running = []
    # Split the thread budget across parallel commands so we do not oversubscribe.
    stage_env = ENV.copy()
    if len(commands) > 1:
        per_process = str(max(1, int(THREADS) // len(commands)))
        stage_env.update(OMP_NUM_THREADS=per_process, MKL_NUM_THREADS=per_process,
                         OPENBLAS_NUM_THREADS=per_process, TDT4173_THREADS=per_process)
    for cmd in commands:
        log = open(WORK / (cmd[0].replace(".py", "") + "_" + "_".join(cmd[1:]) + ".log"), "w")
        running.append((cmd, subprocess.Popen([sys.executable, str(WORK / cmd[0]), *cmd[1:]], cwd=PROJECT, env=stage_env, stdout=log, stderr=subprocess.STDOUT), log))
    failed = None
    while running and failed is None:
        for item in list(running):
            cmd, p, log = item
            status = p.poll()
            if status is None: continue
            running.remove(item); log.close(); text = Path(log.name).read_text()
            print(f"--- {' '.join(cmd)} (exit {status})\n" + (text if len(text) < 3000 else text[:800] + "\n...\n" + text[-2000:]))
            if status != 0: failed = (cmd, log.name)
        if running and failed is None: time.sleep(2)
    if failed:
        for cmd, p, log in running: p.terminate()
        for cmd, p, log in running: p.wait(); log.close()
        raise AssertionError(f"stage {failed[0]} failed, see {failed[1]}")
    missing = [o for o in outputs if not (WORK / o).exists()]
    assert not missing, f"stage finished but outputs are missing: {missing}"
    marker.write_text("ok")
    print(f"stage time {time.time() - t0:.0f}s")


## 1. Data loading, hydro features and tree helpers
Loads the raw files (all times in UTC) and reads the river system from `Tokke_Vinje_topology.yaml`.
It builds one row per (case, unit, hour) with features such as price and its weekly context, water values of the reservoirs above and below each plant,
start storage, inflow, heads from the volume–head curves, water budgets and daily profit compared with the start cost.
Only information known at the start of a case is used. No future storage and no labels are used as inputs.

In [ ]:
%%writefile work_nb2/hydro_base.py
display = print
import os, sys, time, gc, json, subprocess, platform, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# The notebook expects to sit next to data/ (data/kernel, data/extended), sample_submission.csv
# and prediction_mapping.csv. Set PROJECT if the data lives somewhere else.
PROJECT = Path(os.environ.get("TDT4173_PROJECT", "."))
assert (PROJECT / "data/kernel/Unit_commitment_decisions.csv").exists(), \
    "Put this notebook next to data/, or set the TDT4173_PROJECT environment variable"
WORK = Path(os.environ["V12C_WORK"])
WORK.mkdir(exist_ok=True)
SEED = 4173
H = 168
THREADS = int(os.environ.get("TDT4173_THREADS", max(1, os.cpu_count() or 1)))
SMOKE = os.environ.get("TDT4173_SMOKE") == "1"   # tiny run that only checks the code end to end
print("Project:", PROJECT.resolve(), "| threads:", THREADS, "| smoke test:", SMOKE)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| lightgbm", lgb.__version__)

def load_inputs(project=PROJECT):
    uc = pd.read_csv(project / "data/kernel/Unit_commitment_decisions.csv")
    uc["starttime"] = pd.to_datetime(uc.starttime, utc=True)
    sample = pd.read_csv(project / "sample_submission.csv")
    mapping = pd.read_csv(project / "prediction_mapping.csv")
    mapping["starttime"] = pd.to_datetime(mapping.starttime, utc=True)
    labels = list(sample.columns.drop("Run No"))
    units = list(dict.fromkeys(c.removeprefix("result_committed_").rsplit("_t", 1)[0] for c in labels))
    ku = np.array([units.index(c.removeprefix("result_committed_").rsplit("_t", 1)[0]) for c in labels])
    ht = np.array([int(c.rsplit("_t", 1)[1]) for c in labels])
    assert len(labels) == len(units) * H == len(set(zip(ku, ht))) == 2352
    assert uc[labels].isin([0, 1]).all().all()
    series = {}
    for key, rel in {"price": "kernel/Historical_day_ahead_price_2015_2025.csv",
                     "inflow": "kernel/Historical_inflow_1958_2025.csv",
                     "volume": "kernel/Historical_volume_2015_2024.csv",
                     "wv": "kernel/Synthetic_water_value_2015_2024.csv",
                     "flow": "extended/Constraint_min_flow.csv",
                     "minvol": "extended/Constraint_min_volume.csv"}.items():
        df = pd.read_csv(project / "data" / rel)
        df["date"] = pd.to_datetime(df.date, utc=True)
        assert not df.date.duplicated().any(), key
        df = df.set_index("date").sort_index()
        if key == "price":
            df = df.resample("h").mean()       # the price switches to 15-minute resolution in late 2025
        series[key] = df
    config = yaml.safe_load((project / "data/extended/Tokke_Vinje_topology.yaml").read_text())
    return uc, sample, mapping, labels, units, ku, ht, series, config

uc, sample, mapping, labels, units, ku, ht, series, config = load_inputs()
Y = uc[labels].to_numpy(dtype=np.uint8)
print("labelled cases:", len(uc), uc.starttime.min().date(), "-", uc.starttime.max().date())
print("test cases:", len(mapping), mapping.starttime.min().date(), "-", mapping.starttime.max().date())
print("units:", units)

class CaseFeatures:
    """Base features for one set of cases. Rows are addressed by id = case * 2352 + label column,
    so features are computed on demand for any subset of (case, unit, hour) rows."""
    def __init__(self, starts, series, config, units, ku, ht):
        self.units, self.ku, self.ht = units, ku, ht
        self.L = len(ku); self.n = len(starts)
        self.starts = pd.DatetimeIndex(starts)
        self.reservoirs = list(series["volume"].columns)
        self.rid = {r: i for i, r in enumerate(self.reservoirs)}
        idx = pd.to_datetime((self.starts.tz_convert("UTC").tz_localize(None).to_numpy()[:, None] +
                              np.arange(H)[None, :] * np.timedelta64(1, "h")).ravel(), utc=True)
        def window(key):
            a = series[key].reindex(idx).to_numpy(dtype=np.float32)
            assert np.isfinite(a).all(), f"missing hourly data: {key}"
            return a.reshape(self.n, H, -1)
        p = window("price")[:, :, 0]
        self.p = p
        self.mean = p.mean(axis=1); self.std = np.maximum(p.std(axis=1), 1.0)
        self.pscale = np.maximum(np.abs(self.mean), 10.0)
        # initial storage = start day; terminal water value = start day + 7 (dataset documentation)
        self.vol = series["volume"].reindex(self.starts)[self.reservoirs].to_numpy(dtype=np.float32)
        self.wv = series["wv"].reindex(self.starts + pd.Timedelta(days=7))[self.reservoirs].to_numpy(dtype=np.float32)
        self.wv0 = series["wv"].reindex(self.starts)[self.reservoirs].to_numpy(dtype=np.float32)
        assert np.isfinite(self.vol).all() and np.isfinite(self.wv).all()
        self.caps = np.array([config["model"]["reservoir"][r]["max_vol"] for r in self.reservoirs], dtype=np.float32)
        inflow = window("inflow"); inf_cols = list(series["inflow"].columns)
        inflow = inflow[:, :, [inf_cols.index(r) for r in self.reservoirs]]
        self.inf = inflow
        self.inf_total = inflow.sum(axis=1) * 3600 / 1e6           # Mm3 per week
        self.inf_cum = np.cumsum(inflow, axis=1) * 3600 / 1e6
        self.flow = window("flow"); self.minvol = window("minvol")
        self.mv_all = np.zeros((self.n, H, len(self.reservoirs)), dtype=np.float32)
        for j, r in enumerate(series["minvol"].columns):
            self.mv_all[:, :, self.rid[r]] = self.minvol[:, :, j]
        self.week_minvol = self.mv_all.max(axis=1)
        # topology from the YAML: nearest reservoirs upstream / downstream of each plant
        self.connections = config["connections"]
        forward, backward = {}, {}
        for c in self.connections:
            a, b = (c["from_type"], c["from"]), (c["to_type"], c["to"])
            forward.setdefault(a, []).append(b); backward.setdefault(b, []).append(a)
        def nearest(node, graph):
            result, todo, seen = set(), list(graph.get(node, [])), set()
            while todo:
                typ, name = todo.pop()
                if (typ, name) in seen: continue
                seen.add((typ, name))
                if typ == "reservoir": result.add(name)
                elif typ in ["river", "tunnel"]: todo.extend(graph.get((typ, name), []))
            return sorted(result)
        self.topology = []
        self.upweights = np.zeros((len(units), len(self.reservoirs)), dtype=np.float32)
        self.downweights = self.upweights.copy(); self.feedweights = self.upweights.copy()
        self.qmax, self.pmax, self.pmin, self.startcost = [], [], [], []
        for k, u in enumerate(units):
            plant = next(c["to"] for c in self.connections
                         if c["from_type"] == "generator" and c["from"] == u and c["to_type"] == "plant")
            up = nearest(("plant", plant), backward); down = nearest(("plant", plant), forward)
            assert up, (u, "no upstream reservoir")
            feed = set(up)
            # a tiny buffer reservoir (Vatjern, 0.3 Mm3) is fed by a natural upstream reservoir (Langeidvatn):
            # count those volumes as available water (heuristic, not a mass balance)
            for r in up:
                if config["model"]["reservoir"][r]["max_vol"] < 1:
                    feed.update(nearest(("reservoir", r), backward))
            for r in up: self.upweights[k, self.rid[r]] = 1 / len(up)
            for r in down: self.downweights[k, self.rid[r]] = 1 / max(len(down), 1)
            for r in feed: self.feedweights[k, self.rid[r]] = 1
            g = config["model"]["generator"][u]
            self.qmax.append(max(max(curve["x"]) for curve in g["turb_eff_curves"]))
            self.pmax.append(g["p_max"]); self.pmin.append(g["p_min"]); self.startcost.append(g.get("startcost_const", 0))
            self.topology.append(dict(unit=u, plant=plant, upstream=", ".join(up), downstream=", ".join(down),
                                      storage_feeders=", ".join(sorted(feed))))
        self.qmax = np.array(self.qmax, dtype=np.float32); self.pmax = np.array(self.pmax, dtype=np.float32)
        self.pmin = np.array(self.pmin, dtype=np.float32); self.startcost = np.array(self.startcost, dtype=np.float32)
        self.feedcap = self.feedweights @ self.caps
        self.upwv = self.wv @ self.upweights.T; self.downwv = self.wv @ self.downweights.T
        self.netwv = self.upwv - self.downwv
        self.upwv0 = self.wv0 @ self.upweights.T; self.downwv0 = self.wv0 @ self.downweights.T
        self.feedvol = self.vol @ self.feedweights.T
        self.downvol = (self.vol / self.caps) @ self.downweights.T
        self.infweek = self.inf_total @ self.feedweights.T
        self.slack = (self.vol - self.week_minvol) @ self.feedweights.T
        self.profitable = (p[:, :, None] > self.upwv[:, None, :]).mean(axis=1).astype(np.float32)
        self.profitable_net = (p[:, :, None] > self.netwv[:, None, :]).mean(axis=1).astype(np.float32)
        rank = pd.DataFrame(p).rank(axis=1, pct=True).to_numpy(dtype=np.float32)
        dayrank = pd.DataFrame(p.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy(dtype=np.float32).reshape(self.n, H)
        self.static_hour = {
            "price_week_rank": rank, "price_day_rank": dayrank,
            "price_z": (p - self.mean[:, None]) / self.std[:, None],
            "price_relative_mean": p / self.pscale[:, None],
            "price_delta_prev": np.diff(p, axis=1, prepend=p[:, :1]) / self.std[:, None],
            "price_delta_next": np.diff(p, axis=1, append=p[:, -1:]) / self.std[:, None],
            "price_rolling_6h_z": (pd.DataFrame(p).T.rolling(6, min_periods=1, center=True).mean().T.to_numpy()
                                   - self.mean[:, None]) / self.std[:, None],
        }
        self.local = idx.tz_convert("Europe/Oslo")
        self.mon = self.starts.month.to_numpy()
        self.hour = np.array(self.local.hour).reshape(self.n, H)
        self.dow = np.array(self.local.dayofweek).reshape(self.n, H)

    def X(self, ids, mode="domain"):
        ids = np.asarray(ids); c, l = ids // self.L, ids % self.L
        u, t = self.ku[l], self.ht[l]
        p = self.p[c, t]; scale = self.pscale[c]; std = self.std[c]
        names, values = [], []
        def put(n, a): names.append(n); values.append(np.asarray(a, dtype=np.float32))
        put("unit", u); put("t", t); put("hour", self.hour[c, t]); put("weekday", self.dow[c, t]); put("month", self.mon[c])
        doy = self.starts.dayofyear.to_numpy()[c]
        put("season_sin", np.sin(2 * np.pi * doy / 365.25)); put("season_cos", np.cos(2 * np.pi * doy / 365.25))
        for n, a in self.static_hour.items(): put(n, a[c, t])
        put("week_volatility", self.std[c] / scale)
        for j, r in enumerate(self.reservoirs):
            put("storage_" + r, self.vol[c, j] / self.caps[j]); put("wv_relative_" + r, self.wv[c, j] / scale)
        for j in range(self.flow.shape[2]): put("minflow_" + str(j), self.flow[c, t, j])
        for j, r in enumerate(self.reservoirs):
            if np.any(self.week_minvol[:, j] > 0):
                put("storage_slack_" + r, (self.vol[c, j] - self.week_minvol[c, j]) / self.caps[j])
        up = self.upwv[c, u]; down = self.downwv[c, u]; net = self.netwv[c, u]
        put("price_minus_upwv_z", (p - up) / std); put("price_over_upwv", p / np.maximum(np.abs(up), 5.0))
        put("price_minus_netwv_z", (p - net) / std); put("price_over_netwv", p / np.maximum(np.abs(net), 5.0))
        put("price_minus_netwv0_z", (p - self.upwv0[c, u] + self.downwv0[c, u]) / std)
        put("upwv_relative", up / scale); put("downwv_relative", down / scale)
        put("wv_change_relative", (up - self.upwv0[c, u]) / scale)
        put("profitable_week_share", self.profitable[c, u]); put("above_net_week_share", self.profitable_net[c, u])
        put("feed_storage_fraction", self.feedvol[c, u] / self.feedcap[u])
        put("down_storage_fraction", self.downvol[c, u])
        put("week_inflow_capacity", self.infweek[c, u] / self.feedcap[u])
        put("storage_slack_fraction", self.slack[c, u] / self.feedcap[u])
        put("water_budget_heuristic", (self.slack[c, u] + self.infweek[c, u]) * 1e6 / (self.qmax[u] * H * 3600))
        put("stored_budget_heuristic", self.slack[c, u] * 1e6 / (self.qmax[u] * H * 3600))
        put("startcost_relative", self.startcost[u] / np.maximum(self.pmax[u] * scale, 1))
        put("minload_fraction", self.pmin[u] / self.pmax[u])
        a = np.column_stack(values); assert np.isfinite(a).all()
        self.names = names
        return a

class RichHydroFeatures(CaseFeatures):
    """Adds hourly/cumulative inflow, whole-catchment state, reservoir heads (from vol_head curves),
    price lags/leads and daily profitability relative to the start cost. Used by the tree models."""
    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        index = pd.to_datetime((self.starts.tz_convert("UTC").tz_localize(None).to_numpy()[:, None] +
                                np.arange(H)[None, :] * np.timedelta64(1, "h")).ravel(), utc=True)
        inflow = series["inflow"].reindex(index)
        rivers = [c for c in inflow if c not in self.reservoirs]
        self.river_inflow = inflow[rivers].sum(axis=1).to_numpy(dtype=np.float32).reshape(self.n, H)
        self.catchment = np.zeros_like(self.feedweights)          # every reservoir upstream of the plant
        backward = {}
        for edge in config["connections"]:
            backward.setdefault((edge["to_type"], edge["to"]), []).append((edge["from_type"], edge["from"]))
        for k, row in enumerate(self.topology):
            todo = [("reservoir", r) for r in row["upstream"].split(", ")]; seen = set()
            while todo:
                node = todo.pop()
                if node in seen: continue
                seen.add(node)
                if node[0] == "reservoir": self.catchment[k, self.rid[node[1]]] = 1
                if node[0] != "generator": todo.extend(backward.get(node, []))
        self.catchcap = self.catchment @ self.caps
        self.catchvol = self.vol @ self.catchment.T
        self.catchinf = self.inf_total @ self.catchment.T
        self.week_flow_mean = self.flow.mean(axis=1); self.week_flow_max = self.flow.max(axis=1)
        self.heads = np.empty_like(self.vol)
        for j, r in enumerate(self.reservoirs):
            curve = config["model"]["reservoir"][r]["vol_head"]
            self.heads[:, j] = np.interp(self.vol[:, j], curve["x"], curve["y"])
        self.uphead = self.heads @ self.upweights.T; self.downhead = self.heads @ self.downweights.T
        self.outlet = np.array([config["model"]["plant"][r["plant"]].get("outlet_line", 0) for r in self.topology],
                               dtype=np.float32)
        for u in range(len(units)):
            if not self.downweights[u].any(): self.downhead[:, u] = self.outlet[u]
        self.nethead = np.maximum(self.uphead - self.downhead, 1.0)
        self.day_profitable = np.zeros((self.n, 7, len(units)), dtype=np.float32)
        self.day_positive_margin = np.zeros_like(self.day_profitable)
        for k in range(len(units)):
            margin = self.p - self.upwv[:, k, None]
            self.day_profitable[:, :, k] = (margin.reshape(self.n, 7, 24) > 0).mean(axis=2)
            self.day_positive_margin[:, :, k] = np.maximum(margin, 0).reshape(self.n, 7, 24).sum(axis=2)
        self.daily_max = self.p.reshape(self.n, 7, 24).max(axis=2)
        self.daily_mean = self.p.reshape(self.n, 7, 24).mean(axis=2)
        self.daily_min = self.p.reshape(self.n, 7, 24).min(axis=2)
        self.small_reservoir_ids = np.flatnonzero(self.caps < 30)

    def X(self, ids, mode="rich"):
        base = super().X(ids); names = self.names.copy()
        ids = np.asarray(ids); c, l = ids // self.L, ids % self.L; u, t = self.ku[l], self.ht[l]
        extra = []
        def put(name, value): names.append(name); extra.append(np.asarray(value, dtype=np.float32))
        for j, r in enumerate(self.reservoirs):
            put("week_natural_inflow_capacity_" + r, self.inf_total[c, j] / self.caps[j])
        hourly = np.einsum("ij,ij->i", self.inf[c, t, :], self.feedweights[u])
        cumulative = np.einsum("ij,ij->i", self.inf_cum[c, t, :], self.feedweights[u])
        down_inflow = np.einsum("ij,ij->i", self.inf[c, t, :], self.downweights[u])
        put("hour_inflow_fraction_qmax", hourly / self.qmax[u]); put("hour_downstream_inflow", down_inflow)
        put("cumulative_natural_inflow_fraction", cumulative / self.feedcap[u])
        put("remaining_natural_inflow_fraction", (self.infweek[c, u] - cumulative) / self.feedcap[u])
        put("volume_without_release_fraction", (self.feedvol[c, u] + cumulative) / self.feedcap[u])
        put("river_natural_inflow", self.river_inflow[c, t]); put("river_week_natural_mean", self.river_inflow.mean(axis=1)[c])
        put("catchment_storage_fraction", self.catchvol[c, u] / self.catchcap[u])
        put("catchment_week_inflow_fraction", self.catchinf[c, u] / self.catchcap[u])
        put("catchment_budget_heuristic", (self.catchvol[c, u] + self.catchinf[c, u]) * 1e6 / (self.qmax[u] * H * 3600))
        put("initial_upstream_head", self.uphead[c, u]); put("initial_net_head", self.nethead[c, u])
        put("head_implied_max_power", 9.81 * self.qmax[u] * self.nethead[c, u] / 1000)
        for j in range(self.flow.shape[2]):
            put("minflow_week_mean_" + str(j), self.week_flow_mean[c, j]); put("minflow_week_max_" + str(j), self.week_flow_max[c, j])
        for lag in [2, 6, 12, 24]:
            put("price_prev_" + str(lag) + "_z", (self.p[c, np.maximum(t - lag, 0)] - self.mean[c]) / self.std[c])
            put("price_next_" + str(lag) + "_z", (self.p[c, np.minimum(t + lag, H - 1)] - self.mean[c]) / self.std[c])
        day = t // 24
        put("day_profitable_fraction", self.day_profitable[c, day, u])
        put("day_positive_margin_over_startcost", self.day_positive_margin[c, day, u] * self.pmax[u] / np.maximum(self.startcost[u], 1))
        put("day_max_price_z", (self.daily_max[c, day] - self.mean[c]) / self.std[c])
        put("day_mean_price_z", (self.daily_mean[c, day] - self.mean[c]) / self.std[c])
        put("day_min_price_z", (self.daily_min[c, day] - self.mean[c]) / self.std[c])
        for j in self.small_reservoir_ids:
            put("small_res_cum_inflow_" + self.reservoirs[j], self.inf_cum[c, t, j] / self.caps[j])
        result = np.column_stack([base, *extra]); assert np.isfinite(result).all()
        self.names = names
        return result

class DispatchFeatures(RichHydroFeatures):
    """Adds penstock groups, head losses, turbine efficiency at the initial head and
    'how many hours can the available water run' price thresholds. Used by the temporal network."""
    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        plants = list(dict.fromkeys(row["plant"] for row in self.topology))
        self.plant_code = np.array([plants.index(row["plant"]) for row in self.topology])
        self.penstock = np.array([config["model"]["generator"][u]["penstock"] for u in units])
        self.pipe_group = np.array([100 * p + pipe for p, pipe in zip(self.plant_code, self.penstock)])
        self.pipe_position = np.empty(len(units), dtype=int)
        self.plant_q = np.empty(len(units), dtype=np.float32); self.pipe_q = np.empty(len(units), dtype=np.float32)
        self.main_loss = np.empty(len(units), dtype=np.float32); self.pipe_loss = np.empty(len(units), dtype=np.float32)
        self.eta_best = np.empty((self.n, len(units)), dtype=np.float32)
        self.eta_low = self.eta_best.copy(); self.qbest = self.eta_best.copy()
        for k, u in enumerate(units):
            same_plant = self.plant_code == self.plant_code[k]; same_pipe = self.pipe_group == self.pipe_group[k]
            self.pipe_position[k] = list(np.flatnonzero(same_pipe)).index(k)
            self.plant_q[k] = self.qmax[same_plant].sum(); self.pipe_q[k] = self.qmax[same_pipe].sum()
            pc = config["model"]["plant"][self.topology[k]["plant"]]
            assert len(pc["main_loss"]) == 1
            self.main_loss[k] = float(pc["main_loss"][0]); self.pipe_loss[k] = float(pc["penstock_loss"][self.penstock[k] - 1])
            curves = sorted(config["model"]["generator"][u]["turb_eff_curves"], key=lambda a: a["ref"])
            refs = [a["ref"] for a in curves]
            self.eta_best[:, k] = np.interp(self.nethead[:, k], refs, [max(a["y"]) / 100 for a in curves])
            self.eta_low[:, k] = np.interp(self.nethead[:, k], refs, [a["y"][0] / 100 for a in curves])
            self.qbest[:, k] = np.interp(self.nethead[:, k], refs, [a["x"][int(np.argmax(a["y"]))] for a in curves])
        available = self.slack + self.infweek
        self.unit_budget = available * 1e6 / (self.qmax[None, :] * H * 3600)
        self.plant_budget = available * 1e6 / (self.plant_q[None, :] * H * 3600)
        sorted_price = np.sort(self.p, axis=1)
        def threshold(budget):          # price of the k-th most expensive hour, k = hours the water can run
            pos = (1 - np.clip(budget, 0, 1)) * (H - 1)
            low = np.floor(pos).astype(int); high = np.minimum(low + 1, H - 1); cases = np.arange(self.n)[:, None]
            return sorted_price[cases, low] + (pos - low) * (sorted_price[cases, high] - sorted_price[cases, low])
        self.unit_threshold = threshold(self.unit_budget); self.plant_threshold = threshold(self.plant_budget)
        self.positive_context = {}
        for width in [6, 12, 24]:
            arr = np.empty((self.n, H, len(units)), dtype=np.float32)
            for k in range(len(units)):
                margin = np.maximum(self.p - self.upwv[:, k, None], 0) / self.std[:, None]
                arr[:, :, k] = pd.DataFrame(margin).T.rolling(width, min_periods=1, center=True).mean().T.to_numpy()
            self.positive_context[width] = arr

    def X(self, ids, mode="dispatch"):
        base = RichHydroFeatures.X(self, ids); names = self.names.copy()
        ids = np.asarray(ids); c, l = ids // self.L, ids % self.L; u, t = self.ku[l], self.ht[l]
        extra = []
        def put(name, value): names.append(name); extra.append(np.asarray(value, dtype=np.float32))
        put("plant_id", self.plant_code[u]); put("penstock_group", self.pipe_group[u]); put("position_in_pipe", self.pipe_position[u])
        put("pipe_share_of_plant_capacity", self.pipe_q[u] / self.plant_q[u])
        put("single_full_load_head_loss", self.main_loss[u] * self.qmax[u] ** 2 + self.pipe_loss[u] * self.qmax[u] ** 2)
        put("plant_full_load_head_loss", self.main_loss[u] * self.plant_q[u] ** 2 + self.pipe_loss[u] * self.pipe_q[u] ** 2)
        put("single_loss_fraction_head", (self.main_loss[u] + self.pipe_loss[u]) * self.qmax[u] ** 2 / self.nethead[c, u])
        put("best_turbine_efficiency", self.eta_best[c, u]); put("best_flow_fraction_max", self.qbest[c, u] / self.qmax[u])
        put("low_load_water_value_proxy_z",
            (self.p[c, t] - self.upwv[c, u] * self.eta_best[c, u] / np.maximum(self.eta_low[c, u], 0.1)) / self.std[c])
        put("plant_week_water_budget", self.plant_budget[c, u])
        put("price_minus_unit_water_threshold_z", (self.p[c, t] - self.unit_threshold[c, u]) / self.std[c])
        put("price_minus_plant_water_threshold_z", (self.p[c, t] - self.plant_threshold[c, u]) / self.std[c])
        put("price_minus_combined_dispatch_threshold_z",
            (self.p[c, t] - np.maximum(self.plant_threshold[c, u], self.upwv[c, u])) / self.std[c])
        put("rank_budget_distance", self.static_hour["price_week_rank"][c, t] + np.clip(self.plant_budget[c, u], 0, 1) - 1)
        for width, a in self.positive_context.items(): put("positive_margin_context_" + str(width), a[c, t, u])
        for j, r in enumerate(self.reservoirs): put("water_value_trend_" + r, (self.wv[c, j] - self.wv0[c, j]) / self.pscale[c])
        result = np.column_stack([base, *extra]); assert np.isfinite(result).all()
        self.names = names
        return result


class NetworkPressureFeatures(DispatchFeatures):
    """Adds initial reservoir levels, tunnel level differences / pressure-flow scenarios and an
    approximate upstream-release scenario. These are input-only approximations, not a hydraulic simulation."""
    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        self.tunnel_states = []
        self.tunnel_feed = np.zeros((self.n, len(units)), dtype=np.float32)
        for name, tunnel in config["model"].get("tunnel", {}).items():
            source = [e["from"] for e in config["connections"] if e["to"] == name and e["to_type"] == "tunnel" and e["from_type"] == "reservoir"]
            target = [e["to"] for e in config["connections"] if e["from"] == name and e["from_type"] == "tunnel" and e["to_type"] == "reservoir"]
            if len(source) != 1 or len(target) != 1: continue
            a, b = self.rid[source[0]], self.rid[target[0]]
            alpha = float(tunnel["loss_factor"]); ha, hb = self.heads[:, a], self.heads[:, b]
            start = float(tunnel["start_height"]); end = float(tunnel["end_height"])
            positive = np.maximum(ha - np.maximum(hb, end), 0) * (ha >= start)
            negative = np.maximum(hb - np.maximum(ha, start), 0) * (hb >= end)
            q = (np.sqrt(positive / max(alpha, 1e-9)) - np.sqrt(negative / max(alpha, 1e-9))).astype(np.float32)
            self.tunnel_states.append((name, (ha - hb).astype(np.float32), q))
            self.tunnel_feed += q[:, None] * (self.feedweights[:, b] - self.feedweights[:, a])[None, :]
        self.intake_head = self.uphead.copy()
        for k, row in enumerate(self.topology):       # Songa: two intake tunnels -> pressure balance at full flow
            incoming = [e["from"] for e in config["connections"] if e["to_type"] == "plant" and e["to"] == row["plant"] and e["from_type"] == "tunnel"]
            heads, alphas = [], []
            for name in incoming:
                res = [e["from"] for e in config["connections"] if e["to_type"] == "tunnel" and e["to"] == name and e["from_type"] == "reservoir"]
                if len(res) == 1:
                    heads.append(self.heads[:, self.rid[res[0]]]); alphas.append(float(config["model"]["tunnel"][name]["loss_factor"]))
            if len(heads) < 2: continue
            upper = np.column_stack(heads); alpha = np.array(alphas)[None, :]; qtarget = float(self.plant_q[k])
            lo = upper.min(axis=1) - max(alphas) * qtarget * qtarget * 2 - 1; hi = upper.max(axis=1)
            for _ in range(25):
                middle = (lo + hi) / 2; dh = upper - middle[:, None]
                flow = (np.sign(dh) * np.sqrt(np.abs(dh) / alpha)).sum(axis=1)
                lo = np.where(flow > qtarget, middle, lo); hi = np.where(flow > qtarget, hi, middle)
            self.intake_head[:, k] = (lo + hi) / 2
        threshold = np.maximum(self.upwv, self.plant_threshold)
        z = np.clip(2 * (self.p[:, :, None] - threshold[:, None, :]) / self.std[:, None, None], -20, 20)
        q = 1 / (1 + np.exp(-z)) * self.qmax[None, None, :]
        for plant in np.unique(self.plant_code):
            ids = np.flatnonzero(self.plant_code == plant)
            required = q[:, :, ids].sum(axis=(1, 2)) * 3600
            available = np.maximum(self.slack[:, ids[0]] + self.infweek[:, ids[0]], 0) * 1e6
            q[:, :, ids] *= np.minimum(1, available / np.maximum(required, 1))[:, None, None]
        self.upstream_release_inflow = np.einsum("ctu,ur->ctr", q, self.downweights).astype(np.float32)
        self.upstream_release_cum = np.cumsum(self.upstream_release_inflow, axis=1) * 3600 / 1e6
        self.upstream_release_week = self.upstream_release_inflow.sum(axis=1) * 3600 / 1e6

    def X(self, ids, mode="network"):
        base = DispatchFeatures.X(self, ids); names = self.names.copy()
        ids = np.asarray(ids); c, l = ids // self.L, ids % self.L; u, t = self.ku[l], self.ht[l]
        extra = []
        def put(n, v): names.append(n); extra.append(np.asarray(v, dtype=np.float32))
        for j, r in enumerate(self.reservoirs): put("initial_level_" + r, self.heads[c, j])
        for name, dh, q in self.tunnel_states:
            put("tunnel_level_difference_" + name, dh[c]); put("tunnel_pressure_flow_scenario_" + name, q[c])
        put("feeding_tunnel_flow_relative_qmax", self.tunnel_feed[c, u] / self.qmax[u])
        put("intake_head_fullflow_scenario", self.intake_head[c, u])
        effective = np.maximum(self.intake_head[c, u] - self.downhead[c, u] - self.main_loss[u] * self.plant_q[u] ** 2
                               - self.pipe_loss[u] * self.pipe_q[u] ** 2, 1)
        put("fullflow_effective_head_scenario", effective)
        put("price_minus_head_scaled_value_proxy_z", (self.p[c, t] - self.upwv[c, u] * self.nethead[c, u] / effective) / self.std[c])
        inflow = np.einsum("ij,ij->i", self.upstream_release_inflow[c, t, :], self.feedweights[u])
        cumulative = np.einsum("ij,ij->i", self.upstream_release_cum[c, t, :], self.feedweights[u])
        natural = np.einsum("ij,ij->i", self.inf_cum[c, t, :], self.feedweights[u])
        whole = np.einsum("ij,ij->i", self.upstream_release_week[c, :], self.feedweights[u])
        volume = self.feedvol[c, u] + cumulative + natural
        put("upstream_release_scenario_relative_qmax", inflow / self.qmax[u])
        put("upstream_release_scenario_cumulative_fraction", cumulative / self.feedcap[u])
        put("upstream_release_scenario_week_fraction", whole / self.feedcap[u])
        put("volume_with_upstream_release_scenario_fraction", volume / self.feedcap[u])
        put("scenario_overflow_pressure", np.maximum(volume - self.feedcap[u], 0) / self.feedcap[u])
        put("scenario_required_release_fraction", np.maximum(volume - self.feedcap[u], 0) * 1e6 / np.maximum((t + 1) * 3600 * self.qmax[u], 1))
        result = np.column_stack([base, *extra]); assert np.isfinite(result).all()
        self.names = names
        return result

def lgb_params(leaves=31):
    return dict(objective="binary", metric="binary_logloss", learning_rate=0.045, num_leaves=leaves,
                min_data_in_leaf=100, lambda_l2=10, feature_fraction=0.85, bagging_fraction=0.9, bagging_freq=1,
                num_threads=THREADS, verbosity=-1, seed=SEED, deterministic=True, force_col_wise=True, max_bin=127)

def sample_rows(cases, L, n, seed):
    """random (case, unit, hour) row ids from the given cases"""
    rng = np.random.default_rng(seed)
    s = rng.choice(len(cases) * L, min(n, len(cases) * L), replace=False)
    return np.asarray(cases)[s // L] * L + s % L

def unit_ids(cases, features, k):
    cols = np.flatnonzero(features.ku == k)
    return (np.asarray(cases)[:, None] * features.L + cols[None, :]).ravel()

def predict_rows(model, features, ids, batch=100_000):
    return np.concatenate([model.predict(features.X(ids[i:i + batch]), num_threads=THREADS)
                           for i in range(0, len(ids), batch)])

MAX_POOLED, MAX_UNIT = (5, 5) if SMOKE else (2000, 1400)

def tree_reference(train_f, train_cases, pred_f, pred_cases, stop_cases=None, rounds=None, final=False):
    """Pooled LightGBM (all units, 63 leaves) + one LightGBM per unit (31 leaves), averaged 50/50.
    With stop_cases: early stopping on those cases and the chosen numbers of rounds are returned.
    Without: the given numbers of rounds are used (no peeking at the evaluation year)."""
    n_pooled = 1_500_000 if final else 1_200_000
    if SMOKE: n_pooled = 20_000
    t0 = time.time()
    ids = sample_rows(train_cases, train_f.L, n_pooled, SEED + 80)
    X = train_f.X(ids)
    ds = lgb.Dataset(X, label=Y.ravel()[ids], feature_name=train_f.names, categorical_feature=["unit"])
    callbacks, valid = [lgb.log_evaluation(0)], []
    if stop_cases is not None:
        vi = sample_rows(stop_cases, train_f.L, 150_000, SEED + 81)
        valid = [lgb.Dataset(train_f.X(vi), label=Y.ravel()[vi], reference=ds)]
        callbacks.append(lgb.early_stopping(100, verbose=False))
    pooled_model = lgb.train(lgb_params(63), ds, MAX_POOLED if rounds is None else rounds["pooled"],
                             valid_sets=valid, callbacks=callbacks)
    chosen = {"pooled": pooled_model.best_iteration or pooled_model.current_iteration()}
    all_ids = (np.asarray(pred_cases)[:, None] * pred_f.L + np.arange(pred_f.L)).ravel()
    pooled = predict_rows(pooled_model, pred_f, all_ids).reshape(len(pred_cases), pred_f.L)
    print(f"pooled tree: {len(ids):,} rows, {chosen['pooled']} rounds, {time.time() - t0:.0f}s", flush=True)
    del X, ds, valid, pooled_model; gc.collect()
    per_unit = np.empty_like(pooled)
    for k, u in enumerate(train_f.units):
        t0 = time.time()
        ids = unit_ids(train_cases, train_f, k)
        cap = 10_000 if SMOKE else 240_000
        if not final and len(ids) > cap:
            ids = np.random.default_rng(SEED + k).choice(ids, cap, replace=False)
        X = train_f.X(ids)
        ds = lgb.Dataset(X, label=Y.ravel()[ids], feature_name=train_f.names, categorical_feature=["unit"])
        callbacks, valid = [lgb.log_evaluation(0)], []
        if stop_cases is not None:
            vi = unit_ids(stop_cases, train_f, k)
            valid = [lgb.Dataset(train_f.X(vi), label=Y.ravel()[vi], reference=ds)]
            callbacks.append(lgb.early_stopping(80, verbose=False))
        m = lgb.train(lgb_params(31), ds, MAX_UNIT if rounds is None else rounds[u], valid_sets=valid, callbacks=callbacks)
        chosen[u] = m.best_iteration or m.current_iteration()
        per_unit[:, pred_f.ku == k] = predict_rows(m, pred_f, unit_ids(pred_cases, pred_f, k)).reshape(len(pred_cases), -1)
        print(f"  unit {u}: {len(ids):,} rows, {chosen[u]} rounds, {time.time() - t0:.0f}s", flush=True)
        del X, ds, valid, m; gc.collect()
    return (0.5 * pooled + 0.5 * per_unit).ravel(), chosen

## 2. Plant and unit LP relaxations (physics-based features)
A simplified version of the scheduling problem, written as a linear program and solved with HiGHS (`scipy.optimize.linprog`).
* **Plant LP**: one week, 17 reservoirs. Water balance per reservoir and hour, plant flow limits, gates and spill with costs, soft minimum flow and minimum volume.
 Each MWh earns the hourly price and pays the end-of-week water value of the plant's own upstream reservoir. Output: flow per plant and hour, and a break-even price.
* **Unit LP**: adds an on/off variable between 0 and 1 for each unit and hour, minimum and maximum flow, and start-up costs. Output: on-state and start-ups per unit and hour.
Both LPs are solved for all 2,922 training and 725 test cases. The outputs are features for the models, not predictions.

In [ ]:
%%writefile work_nb2/feats.py
import numpy as np, pandas as pd, time
K = "data/kernel"; E = "data/extended"; H = 168

def read_hourly(p):
    df = pd.read_csv(p); df["date"] = pd.to_datetime(df["date"], utc=True)
    return df.set_index("date").resample("h").mean()
def read_daily(p):
    df = pd.read_csv(p); df["date"] = pd.to_datetime(df["date"], utc=True).dt.normalize()
    return df.set_index("date")

uc = pd.read_csv(f"{K}/Unit_commitment_decisions.csv")
sample_sub = pd.read_csv("sample_submission.csv", nrows=1)
mapping = pd.read_csv("prediction_mapping.csv")
price = read_hourly(f"{K}/Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
water_value = read_daily(f"{K}/Synthetic_water_value_2015_2024.csv")
volume = read_daily(f"{K}/Historical_volume_2015_2024.csv")
inflow = read_hourly(f"{K}/Historical_inflow_1958_2025.csv")
inflow_ltm = inflow[inflow.index < "2023-01-01"].mean()
inflow = inflow[inflow.index >= "2014-01-01"]
min_flow = read_hourly(f"{E}/Constraint_min_flow.csv")
min_vol = read_hourly(f"{E}/Constraint_min_volume.csv")
LABEL_COLS = [c for c in sample_sub.columns if c != "Run No"]
UNITS = list(dict.fromkeys(c.replace("result_committed_", "").rsplit("_t", 1)[0] for c in LABEL_COLS))

UPSTREAM = {"Hogga": "Bandak", "Tokke": "Vinjevatn", "Vinje": "Vaamarvatn", "Haukeli": "Vatjern",
            "Byrte": "Botnedalsvatn", "Lio": "Byrtevatn", "Kjela": "Foersvatn", "Vesle Kjela": "Kjelavatn", "Songa": "Songavatn"}
DOWNSTREAM = {"Hogga": None, "Tokke": "Bandak", "Vinje": "Vinjevatn", "Haukeli": "Vinjevatn", "Byrte": "Byrtevatn",
              "Lio": "Bandak", "Kjela": "Hyljelihyl", "Vesle Kjela": "Foersvatn", "Songa": "Totak"}
MAX_VOL = {"Bandak": 174.2, "Vinjevatn": 9.36, "Byrtevatn": 76.71, "Botnedalsvatn": 54.17, "Kjelavatn": 84.5,
           "Staavatn": 48, "Venemo": 23, "Langesae": 78, "Hyljelihyl": 0.26, "Bordalsvatn": 184.2, "Foersvatn": 122.3,
           "Songavatn": 640, "Bitdalsvatn": 110, "Totak": 257.8, "Vaamarvatn": 26.2, "Langeidvatn": 32.1, "Vatjern": 0.3}
FEEDERS = {"Haukeli": ["Vatjern", "Langeidvatn"], "Songa": ["Songavatn", "Bitdalsvatn"]}
QMAX = {"Hogga_G1": 168.6, "Tokke_G1": 32, "Tokke_G2": 32, "Tokke_G3": 32, "Tokke_G4": 32, "Lio_G1": 14, "Byrte_G1": 8,
        "Vinje_G1": 55, "Vinje_G2": 55, "Vinje_G3": 55, "Haukeli_G1": 2.1, "Songa_G1": 52, "Kjela_G1": 40, "Vesle Kjela_G1": 20}
# unit-specific constraints: upstream reservoir min volume, rivers the plant's discharge must help satisfy
UNIT_FLOW = {"Lio": "r_Vest_Vassdraget", "Hogga": "b_Bandak", "Tokke": "b_Bandak", "Kjela": "b_Hyljelihyl",
             "Vinje": "b_Vinjevatn", "Haukeli": "b_Vinjevatn", "Byrte": "b_Byrtevatn", "Vesle Kjela": "b_Kjelavatn"}
plant_of = lambda u: u.rsplit("_", 1)[0]
volume_frac = volume / pd.Series(MAX_VOL)[volume.columns]
unit_codes = {u: i for i, u in enumerate(UNITS)}
pv, pidx = price.values, price.index
THR = {}
PQ_FEATURES = None   # P-Q LP tensor (cases, 14, 168, 16), rows aligned with uc then mapping
LP_CASE_INDEX = {}
BASE_LP_FEATURES = None   # Baseline LP tensor, same layout as PQ_FEATURES
MIP = {}   # start -> binary on/off per unit x hour from the MILP
LPU = {}   # start -> (relaxed on-state z, start-up) per unit x hour from lp_unit.py
UNIT_ORDER_LP = None
LPC = {}   # start -> (q fraction per plant x hour, break-even price per plant x hour) from lp.py
PLANTS_LP = ['Hogga','Tokke','Lio','Byrte','Vinje','Haukeli','Songa','Kjela','Vesle Kjela']
N_UNITS_PLANT = {'Tokke':4,'Vinje':3}

def roll(x, w, f):
    return getattr(pd.Series(x).rolling(w, center=True, min_periods=1), f)().values

def run_len(mask):
    """length of the contiguous True/False block containing each hour, and position inside it"""
    n = len(mask); L = np.zeros(n); P = np.zeros(n); i = 0
    while i < n:
        j = i
        while j < n and mask[j] == mask[i]: j += 1
        L[i:j] = j - i; P[i:j] = np.arange(j - i) / max(j - i, 1); i = j
    return L, P

import yaml as _yaml
GEN_YAML = _yaml.safe_load(open("data/extended/Tokke_Vinje_topology.yaml"))["model"]["generator"]
def econ_advantage(revenue, startup, shutdown):
    """best total value with the unit forced ON at hour t minus forced OFF at hour t (forward/backward DP)"""
    Hh = len(revenue); fw = np.zeros((Hh, 2)); bw = np.zeros((Hh, 2)); fw[0, 1] = revenue[0]
    for t in range(1, Hh):
        fw[t, 0] = max(fw[t-1, 0], fw[t-1, 1] - shutdown); fw[t, 1] = revenue[t] + max(fw[t-1, 1], fw[t-1, 0] - startup)
    for t in range(Hh - 2, -1, -1):
        bw[t, 0] = max(bw[t+1, 0], bw[t+1, 1] + revenue[t+1] - startup); bw[t, 1] = max(bw[t+1, 0] - shutdown, bw[t+1, 1] + revenue[t+1])
    tot = fw + bw
    return tot[:, 1] - tot[:, 0]

def case_features(starttime, extra=()):
    s = pd.Timestamp(starttime, tz="UTC"); i0 = pidx.get_loc(s)
    p = pv[i0: i0 + H].astype(float)
    mean, std = p.mean(), p.std() + 1e-6
    day_max = p.reshape(7, 24).max(axis=1).repeat(24)
    local = pd.date_range(s, periods=H, freq="h").tz_convert("Europe/Oslo")
    sh = {"t": np.arange(H), "hour": local.hour.values, "weekday": local.weekday.values, "month": np.full(H, s.month),
          "price_rank_week": pd.Series(p).rank(pct=True).values,
          "price_rank_day": pd.DataFrame(p.reshape(7, 24)).rank(axis=1, pct=True).values.ravel(),
          "price_z": (p - mean) / std, "price_ratio_mean": p / (mean + 1e-6), "price_ratio_daymax": p / (day_max + 1e-6),
          "price_diff_prev": np.r_[0, np.diff(p)] / std, "price_diff_next": np.r_[np.diff(p), 0] / std,
          "week_price_std_ratio": np.full(H, std / (abs(mean) + 1e-6))}
    d0, d7 = s.normalize(), s.normalize() + pd.Timedelta(days=7)
    for r in water_value.columns:
        sh[f"wv7_{r}"] = np.full(H, water_value.at[d7, r] / (mean + 1e-6))
        sh[f"vol0_{r}"] = np.full(H, volume_frac.at[d0, r])
    k0 = min_flow.index.get_loc(s); mf = min_flow.iloc[k0:k0 + H]; mv = min_vol.iloc[k0:k0 + H]
    for c in min_flow.columns: sh[f"minflow_{c}"] = mf[c].values
    for r in min_vol.columns: sh[f"minvol_slack_{r}"] = np.full(H, (volume.at[d0, r] - mv[r].max()) / MAX_VOL[r])
    j0 = inflow.index.get_loc(s); iw = inflow.iloc[j0:j0 + H]
    tot_inf = iw.sum(axis=1).mean() / (inflow_ltm.sum() + 1e-6)
    if "futvol" in extra:
        for r in water_value.columns:
            for k in (1, 3, 7):
                sh[f"dvol{k}_{r}"] = np.full(H, (volume.at[d0 + pd.Timedelta(days=k), r] - volume.at[d0, r]) / MAX_VOL[r])
    if "temporal" in extra:
        z = sh["price_z"]
        for w in (5, 13, 25):
            sh[f"pz_rmean{w}"] = roll(z, w, "mean")
            sh[f"pz_rmax{w}"] = roll(z, w, "max") - z
            sh[f"pz_rmin{w}"] = z - roll(z, w, "min")
        sh["prank_win25"] = np.array([(p[max(0, t-12):t+13] < p[t]).mean() for t in range(H)])
        prev = pv[max(0, i0 - 24):i0].astype(float)
        sh["prev24_ratio"] = np.full(H, prev.mean() / (mean + 1e-6) if len(prev) else np.nan)
        sh["p_t0_over_prev"] = np.full(H, p[0] / (prev[-1] + 1e-6) if len(prev) else np.nan)
    rows = []
    for u in UNITS:
        pl = plant_of(u); up = UPSTREAM[pl]; dn = DOWNSTREAM[pl]; fe = FEEDERS.get(pl, [up])
        wv0, wv7 = water_value.at[d0, up], water_value.at[d7, up]
        f = dict(sh); f["unit"] = np.full(H, unit_codes[u])
        f["wv_change"] = np.full(H, (wv7 - wv0) / (abs(wv0) + 1e-6))
        f["price_over_wv_end"] = p / (abs(wv7) + 1e-6)
        f["price_minus_wv_end_z"] = (p - wv7) / std
        f["frac_hours_profitable"] = np.full(H, (p > wv7).mean())
        wvd0 = water_value.at[d0, dn] if dn else 0.0; wvd7 = water_value.at[d7, dn] if dn else 0.0
        net0, net7 = wv0 - wvd0, wv7 - wvd7
        f["net_wv_end"] = np.full(H, net7)
        f["price_over_net_end"] = p / (abs(net7) + 1e-6)
        mz = (p - net7) / std
        f["price_minus_net_end_z"] = mz
        f["price_minus_net_start_z"] = (p - net0) / std
        f["frac_hours_above_net"] = np.full(H, (p > net7).mean())
        f["wv_end_over_week_mean"] = np.full(H, wv7 / (mean + 1e-6))
        f["vol_up"] = np.full(H, volume.loc[d0, fe].sum() / sum(MAX_VOL[r] for r in fe))
        f["vol_down"] = np.full(H, volume_frac.at[d0, dn] if dn else np.nan)
        f["inflow_up_rel"] = np.full(H, iw[fe].mean().sum() / (inflow_ltm[fe].sum() + 1e-6))
        f["inflow_total_rel"] = np.full(H, tot_inf)
        stored = infl = 0.0
        for r in fe:
            need = mv[r].max() if r in min_vol.columns else 0.0
            stored += (volume.at[d0, r] - need) * 1e6; infl += iw[r].sum() * 3600
        full = QMAX[u] * H * 3600
        f["water_budget"] = np.full(H, (stored + infl) / full); f["stored_budget"] = np.full(H, stored / full)
        if "temporal" in extra:
            above = mz > 0
            L, P = run_len(above)
            f["run_len"] = np.where(above, L, -L); f["run_pos"] = P
            for w in (5, 13):
                f[f"mz_rmean{w}"] = roll(mz, w, "mean")
            # rank of this hour's margin among the week: with budget, unit runs the best ~k hours
            f["budget_minus_rank"] = f["water_budget"] - (1 - pd.Series(mz).rank(pct=True).values)
        if "thr" in extra:
            th = THR[starttime][unit_codes[u]]
            tz = (p - th) / std
            f["thr_hat"] = np.full(H, th / (mean + 1e-6))
            f["p_minus_thr_z"] = tz
            f["p_over_thr"] = p / (abs(th) + 1e-6)
            f["frac_above_thr"] = np.full(H, (p > th).mean())
            f["thr_margin_rank"] = pd.Series(tz).rank(pct=True).values
            f["thr_rmean5"] = roll(tz, 5, "mean"); f["thr_rmean13"] = roll(tz, 13, "mean")
        if "futvol" in extra:
            # realised (historical) storage on the days AFTER the case start: reveals how much water was actually used
            day_idx = np.arange(H) // 24
            for r in water_value.columns:
                pass
            for side, res in (("up", fe), ("dn", [dn] if dn else [])):
                if not res:
                    f[f"{side}_dvol_today"] = np.full(H, np.nan); f[f"{side}_dvol_week"] = np.full(H, np.nan); continue
                v = np.array([volume.loc[d0 + pd.Timedelta(days=k), res].sum() for k in range(8)]) / sum(MAX_VOL[r] for r in res)
                f[f"{side}_dvol_today"] = (v[1:] - v[:-1])[day_idx]           # storage change during the hour's day
                f[f"{side}_dvol_week"] = np.full(H, v[7] - v[0])
                f[f"{side}_dvol_sofar"] = v[day_idx + 1] - v[0]
        if "lp" in extra:
            Qc, Wc = LPC[starttime]; k = PLANTS_LP.index(pl)
            q = Qc[k].astype(float); w = Wc[k].astype(float); nu = N_UNITS_PLANT.get(pl, 1)
            pos = int(u.rsplit("_G", 1)[1]) - 1
            f["lp_q"] = q; f["lp_units"] = q * nu - pos            # >0: LP needs at least pos+1 units of this plant
            f["lp_q_prev"] = np.r_[q[:1], q[:-1]]; f["lp_q_next"] = np.r_[q[1:], q[-1:]]
            f["lp_q_roll5"] = pd.Series(q).rolling(5, center=True, min_periods=1).mean().values
            f["lp_q_roll25"] = pd.Series(q).rolling(25, center=True, min_periods=1).mean().values
            f["lp_q_week"] = np.full(H, q.mean())
            f["lp_breakeven_rel"] = w / (mean + 1e-6)
            f["p_minus_lp_breakeven_z"] = (p - w) / std
            for qq, nm in ((Qc[PLANTS_LP.index(x)], x) for x in ("Tokke", "Vinje", "Hogga", "Songa")):
                f[f"lp_q_{nm}"] = qq.astype(float)
        if "ulp" in extra:
            Zc, Sc = LPU[starttime]; j = list(UNIT_ORDER_LP).index(u)
            z = Zc[j].astype(float); st_ = Sc[j].astype(float)
            f["ulp_z"] = z; f["ulp_z_prev"] = np.r_[z[:1], z[:-1]]; f["ulp_z_next"] = np.r_[z[1:], z[-1:]]
            f["ulp_z_roll5"] = pd.Series(z).rolling(5, center=True, min_periods=1).mean().values
            f["ulp_z_roll25"] = pd.Series(z).rolling(25, center=True, min_periods=1).mean().values
            f["ulp_z_week"] = np.full(H, z.mean())
            f["ulp_start_near"] = pd.Series(st_).rolling(7, center=True, min_periods=1).sum().values
            f["ulp_shut_near"] = pd.Series(np.maximum(np.r_[0, -np.diff(z)], 0)).rolling(7, center=True, min_periods=1).sum().values
            same = [k for k, g in enumerate(UNIT_ORDER_LP) if plant_of(g) == pl]
            f["ulp_plant_units"] = Zc[same].sum(axis=0).astype(float)
        if "pq_lp" in extra:
            a = PQ_FEATURES[LP_CASE_INDEX[starttime]][UNITS.index(u)]                 # (168, 16)
            names16 = ["flow_frac", "commit", "plant_flow", "plant_units", "up_stock", "down_stock", "slack", "gross_ratio",
                       "opp_rel", "p_minus_opp_z", "reduced_z", "startup", "shutdown", "adv_z", "adv_cycle", "pmax_ratio"]
            for i_, nm in enumerate(names16): f["gpq_" + nm] = a[:, i_].astype(float)
        if "baseline_lp" in extra:
            a = BASE_LP_FEATURES[LP_CASE_INDEX[starttime]][UNITS.index(u)]
            for i_ in range(16): f[f"gv1_{i_}"] = a[:, i_].astype(float)
        if "econ" in extra:
            # value of being ON vs OFF at each hour when start/stop costs are paid at transitions (dynamic programming)
            g = GEN_YAML[u]; pmax = float(g["p_max"]); sc = float(g.get("startcost_const", 0)); stc = float(g.get("stopcost_const", 0))
            margin = (p - wv7) * pmax
            adv = econ_advantage(margin, sc, stc)
            den = max(pmax * std, 1.0)
            f["econ_margin_z"] = margin / den; f["econ_adv_z"] = adv / den
            f["econ_adv_signlog"] = np.sign(adv) * np.log1p(np.abs(adv)); f["econ_adv_cycle"] = adv / max(sc + stc, 1.0)
        if "mfdef" in extra:
            # how much water is still missing to meet the legal minimum flow of the river this unit discharges into,
            # after the natural inflow of that river (only r_Vest_Vassdraget has its own natural inflow)
            DOWN_RIVER = {"Lio": "r_Vest_Vassdraget", "Kjela": "b_Hyljelihyl", "Vinje": "b_Vinjevatn", "Haukeli": "b_Vinjevatn",
                          "Byrte": "b_Byrtevatn", "Tokke": "b_Bandak"}
            UP_BYPASS = {"Vesle Kjela": "b_Kjelavatn", "Tokke": "b_Vinjevatn", "Lio": "b_Byrtevatn", "Hogga": "b_Bandak"}
            qmax = QMAX[u]
            rv = DOWN_RIVER.get(pl)
            if rv:
                nat = iw[rv].values if rv in iw.columns else np.zeros(H)
                dfc = np.maximum(mf[rv].values - nat, 0)
            else:
                dfc = np.zeros(H)
            f["mf_down_deficit"] = dfc; f["mf_down_deficit_rel"] = dfc / qmax
            f["mf_down_deficit_24h"] = pd.Series(dfc).rolling(24, center=True, min_periods=1).mean().values
            f["mf_down_deficit_week"] = np.full(H, dfc.mean())
            bp = UP_BYPASS.get(pl)
            f["mf_up_bypass_rel"] = (mf[bp].values / qmax) if bp else np.zeros(H)
        if "mip" in extra:
            j = list(UNIT_ORDER_LP).index(u); z = MIP[starttime][j].astype(float)
            f["mip_z"] = z; f["mip_z_prev"] = np.r_[z[:1], z[:-1]]; f["mip_z_next"] = np.r_[z[1:], z[-1:]]
            f["mip_z_roll5"] = pd.Series(z).rolling(5, center=True, min_periods=1).mean().values
            f["mip_z_roll25"] = pd.Series(z).rolling(25, center=True, min_periods=1).mean().values
            f["mip_z_week"] = np.full(H, z.mean())
            same = [k for k, g in enumerate(UNIT_ORDER_LP) if plant_of(g) == pl]
            f["mip_plant_units"] = MIP[starttime][same].sum(axis=0).astype(float)
            # distance (hours) to the nearest MILP switch of this unit
            sw = np.flatnonzero(np.diff(z) != 0)
            f["mip_dist_switch"] = np.array([np.min(np.abs(sw - t)) if len(sw) else 168 for t in range(H)], float)
        if "unitcons" in extra:
            rv = UNIT_FLOW.get(pl)
            f["u_minflow"] = mf[rv].values / (QMAX[u] * (4 if pl == "Tokke" else 3 if pl == "Vinje" else 1)) if rv else np.zeros(H)
            f["u_minvol_slack_up"] = np.full(H, (volume.at[d0, up] - mv[up].max()) / MAX_VOL[up]) if up in min_vol.columns else np.full(H, np.nan)
            f["u_minvol_req_up"] = mv[up].values / MAX_VOL[up] if up in min_vol.columns else np.zeros(H)
        rows.append(pd.DataFrame(f))
    return pd.concat(rows, ignore_index=True)

def build(starts, extra=()):
    t0 = time.time()
    starts = list(starts); parts = []
    for i in range(0, len(starts), 100):
        Xi = pd.concat([case_features(s, extra) for s in starts[i:i+100]], ignore_index=True)
        parts.append(Xi.astype({c: "float32" for c in Xi.columns}))
    X = pd.concat(parts, ignore_index=True); del parts
    X["unit"] = X["unit"].astype(int).astype("category")
    pass  # print(f"built {X.shape} in {time.time()-t0:.0f}s", flush=True)
    return X

In [ ]:
%%writefile work_nb2/lp_plant.py
"""Simplified (linear) short-term hydro scheduling model of Tokke-Vinje, one week per case.
Maximise  sum_t price_t * production_t  +  sum_r water_value_r * energy_r * V_r,end  -  small flow/penalty costs
s.t. reservoir water balance each hour, plant/river/tunnel flow limits, min-flow / min-volume (soft).
No start costs, no integer on/off, fixed head from the initial levels -> fast LP (HiGHS)."""
import numpy as np, pandas as pd, yaml, scipy.sparse as sp
from scipy.optimize import linprog
import feats as F
H = 168; DT = 0.0036                       # m3/s for one hour -> Mm3
cfg = yaml.safe_load(open("data/extended/Tokke_Vinje_topology.yaml"))
M = cfg["model"]; CON = cfg["connections"]
RES = list(F.volume.columns); rid = {r: i for i, r in enumerate(RES)}
RIV = list(M["river"]); TUN = list(M.get("tunnel", {})); PLANTS = list(M["plant"])
gens = {p: [g for g in M["generator"] if any(c["from"] == g and c["to"] == p for c in CON)] for p in PLANTS}
qmax_unit = {g: max(max(cv["x"]) for cv in M["generator"][g]["turb_eff_curves"]) for g in M["generator"]}
QMAXP = {p: sum(qmax_unit[g] for g in gens[p]) for p in PLANTS}
out_e = lambda name: [c for c in CON if c["from"] == name]
in_e = lambda name: [c for c in CON if c["to"] == name]
def level(r, v):
    c = M["reservoir"][r]["vol_head"]; return float(np.interp(v, c["x"], c["y"]))
# plant source / destination
P_SRC = {p: [(c["from_type"], c["from"]) for c in in_e(p) if c["from_type"] in ("reservoir", "tunnel")] for p in PLANTS}
P_DST = {p: [(c["to_type"], c["to"]) for c in out_e(p)] for p in PLANTS}
def resolve(kind, name):
    """follow rivers/tunnels downstream until a reservoir (or the sea) is reached"""
    seen = 0
    while kind == "river" and seen < 10:
        nxt = [(c["to_type"], c["to"]) for c in out_e(name)]
        if not nxt: return None
        kind, name = nxt[0]; seen += 1
    if kind == "tunnel":
        nxt = [(c["to_type"], c["to"]) for c in out_e(name)]
        return nxt[0][1] if nxt and nxt[0][0] == "reservoir" else None
    return name if kind == "reservoir" else None
def plant_up_res(p):
    k, n = P_SRC[p][0]
    if k == "reservoir": return n
    return [c["from"] for c in in_e(n) if c["from_type"] == "reservoir"][0]
# energy coefficient used to convert each reservoir's water value: the first plant its water reaches
def first_plant(r, depth=0):
    for c in out_e(r):
        if c["to_type"] == "plant": return c["to"]
    for c in out_e(r):
        if c["to_type"] == "tunnel":
            for d in out_e(c["to"]):
                if d["to_type"] == "plant": return d["to"]
                if d["to_type"] == "reservoir" and depth < 4: return first_plant(d["to"], depth + 1)
    for c in out_e(r):
        if c["to_type"] == "river" and c["to"].startswith(("r_", "b_")):
            dst = resolve("river", c["to"])
            if dst and depth < 4: return first_plant(dst, depth + 1)
    return None
FIRST = {r: first_plant(r) for r in RES}

def solve_case(start, return_all=False):
    s = pd.Timestamp(start, tz="UTC"); i0 = F.pidx.get_loc(s); price = F.pv[i0:i0 + H].astype(float)
    d0 = s.normalize(); d7 = d0 + pd.Timedelta(days=7)
    v0 = F.volume.loc[d0, RES].values.astype(float); wv = F.water_value.loc[d7, RES].values.astype(float)
    j0 = F.inflow.index.get_loc(s); infl = F.inflow.iloc[j0:j0 + H]
    k0 = F.min_flow.index.get_loc(s); mf = F.min_flow.iloc[k0:k0 + H]; mv = F.min_vol.iloc[k0:k0 + H]
    lev0 = {r: level(r, v0[rid[r]]) for r in RES}
    # energy coefficient MW per m3/s for each plant from the initial head
    E = {}
    for p in PLANTS:
        up = plant_up_res(p); dst = P_DST[p][0] if P_DST[p] else ("sea", None)
        tail = M["plant"][p].get("outlet_line", 0)
        if dst[0] == "reservoir": tail = max(tail, lev0[dst[1]])
        eta = np.mean([max(max(cv["y"]) for cv in M["generator"][g]["turb_eff_curves"]) for g in gens[p]]) / 100 * 0.98
        E[p] = 9.81 * max(lev0[up] - tail, 1.0) * eta / 1000
    def plant_dst_res(p):
        if not P_DST[p]: return None
        k, n = P_DST[p][0]
        return n if k == "reservoir" else resolve(k, n)
    def ecum(r, depth=0):      # energy of 1 m3/s-hour of water in r on its way to the sea
        p = FIRST[r]
        if p is None or depth > 6: return 0.0
        d = plant_dst_res(p)
        return E[p] + (ecum(d, depth + 1) if d else 0.0)
    Er = np.array([ecum(r) for r in RES])
    # ---------------- variables
    names = []; lb = []; ub = []; cost = []
    def add(n, lo, hi, c): names.append(n); lb.append(lo); ub.append(hi); cost.append(c); return len(names) - 1
    V = np.array([[add(("V", r, t), 0, max(M["reservoir"][r]["max_vol"], v0[rid[r]]), 0) for t in range(H + 1)] for r in RES])
    wv_plant = {p: wv[rid[plant_up_res(p)]] for p in PLANTS}
    # each plant pays its own upstream water value per MWh it produces (empirical SHOP behaviour, see notebook)
    Q = {p: [add(("Q", p, t), 0, QMAXP[p], -(price[t] - wv_plant[p]) * E[p]) for t in range(H)] for p in PLANTS}
    rcost = lambda n: 1.0 if n.startswith("r_") else 2000.0   # natural rivers flow freely; gates (b_) and spill (s_) only when forced
    Fv = {n: [add(("F", n, t), 0, M["river"][n].get("max_flow_const", 1e4), rcost(n)) for t in range(H)] for n in RIV}
    T = {n: [add(("T", n, t), -1000 if n == "Totak_Vaamarvatn" else 0, 1000, 0.01) for t in range(H)] for n in TUN}
    over = {r: [add(("over", r, t), 0, 1e4, 1e6) for t in range(H)] for r in RES}      # emergency overflow
    MFS = {n: [add(("mfs", n, t), 0, 1e4, 5000.0) for t in range(H)] for n in mf.columns}  # min-flow slack
    MVS = {r: [add(("mvs", r, t), 0, 1e4, 1e5) for t in range(H)] for r in mv.columns}  # min-volume slack
    nv = len(names); cost = np.array(cost, float)
    # no terminal valuation: the opportunity cost of water is charged when it is released
    # ---------------- equality constraints
    rows, cols, vals, rhs = [], [], [], []; nrow = [0]
    def row(entries, b):
        for c, v in entries: rows.append(nrow[0]); cols.append(c); vals.append(v)
        rhs.append(b); nrow[0] += 1; return nrow[0] - 1
    # river flows fed only by other rivers/plants are pass-through: F = sum(inputs) + natural inflow
    passthrough = [n for n in RIV if not any(c["from_type"] == "reservoir" for c in in_e(n))]
    for n in passthrough:
        for t in range(H):
            ent = [(Fv[n][t], 1.0)]
            for c in in_e(n):
                if c["from_type"] == "river": ent.append((Fv[c["from"]][t], -1.0))
                if c["from_type"] == "plant": ent.append((Q[c["from"]][t], -1.0))
            nat = float(infl[n].iloc[t]) if n in infl.columns else 0.0
            row(ent, nat)
    for p in PLANTS:   # plants fed by tunnels: Q = sum of tunnel flows
        if P_SRC[p][0][0] == "tunnel":
            for t in range(H): row([(Q[p][t], 1.0)] + [(T[k][t], -1.0) for kind, k in P_SRC[p]], 0.0)
    bal = np.zeros((len(RES), H), dtype=int)
    for r in RES:
        i = rid[r]
        for t in range(H):
            ent = [(V[i, t + 1], 1.0), (V[i, t], -1.0), (over[r][t], DT)]
            for c in out_e(r):
                if c["to_type"] == "plant": ent.append((Q[c["to"]][t], DT))
                elif c["to_type"] == "river": ent.append((Fv[c["to"]][t], DT))
                elif c["to_type"] == "tunnel": ent.append((T[c["to"]][t], DT))
            for c in in_e(r):
                if c["from_type"] == "plant": ent.append((Q[c["from"]][t], -DT))
                elif c["from_type"] == "river": ent.append((Fv[c["from"]][t], -DT))
                elif c["from_type"] == "tunnel" and c["from"] in T: ent.append((T[c["from"]][t], -DT))
            bal[i, t] = row(ent, DT * float(infl[r].iloc[t]))
        row([(V[i, 0], 1.0)], v0[i])
    A_eq = sp.csr_matrix((vals, (rows, cols)), shape=(nrow[0], nv)); b_eq = np.array(rhs)
    # ---------------- inequalities (<=)
    rows, cols, vals, rhs = [], [], [], []; nrow = [0]
    for n in mf.columns:      # flow + slack >= min flow  ->  -flow - slack <= -min
        for t in range(H):
            row([(Fv[n][t], -1.0), (MFS[n][t], -1.0)], -float(mf[n].iloc[t]))
    for r in mv.columns:
        for t in range(H):
            row([(V[rid[r], t + 1], -1.0), (MVS[r][t], -1.0)], -float(mv[r].iloc[t]))
    A_ub = sp.csr_matrix((vals, (rows, cols)), shape=(nrow[0], nv)); b_ub = np.array(rhs)
    res = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=list(zip(lb, ub)), method="highs")
    assert res.status == 0, res.message
    x = res.x; mu = res.eqlin.marginals                       # dual of balance: d(cost)/d(rhs) [EUR per Mm3]
    qfrac = np.array([[x[Q[p][t]] / QMAXP[p] for t in range(H)] for p in PLANTS])
    wv_dyn = {}                                                 # LP water value of each plant's upstream water, EUR/MWh
    for p in PLANTS:
        up = plant_up_res(p)
        dst = None if not P_DST[p] else (P_DST[p][0][1] if P_DST[p][0][0] == "reservoir" else resolve(*P_DST[p][0]))
        lam_up = -mu[bal[rid[up]]]; lam_dn = -mu[bal[rid[dst]]] if dst else np.zeros(H)
        wv_dyn[p] = wv_plant[p] + (lam_up - lam_dn) * DT / E[p]   # break-even price incl. capacity effects
    if return_all: return dict(qfrac=qfrac, wv_dyn=wv_dyn, E=E, Er=Er, obj=res.fun, x=x, Q=Q, V=V, names=names)
    return qfrac.astype(np.float32), np.array([wv_dyn[p] for p in PLANTS], dtype=np.float32)

if __name__ == "__main__":
    import time
    t0 = time.time(); out = solve_case("2022-03-07", return_all=True); print("solve", round(time.time() - t0, 2), "s")
    print("E (MW per m3/s):", {k: round(v, 2) for k, v in out["E"].items()})
    uc = F.uc; row_ = uc[uc.starttime == "2022-03-07"].iloc[0]
    s = pd.Timestamp("2022-03-07", tz="UTC"); i0 = F.pidx.get_loc(s); price = F.pv[i0:i0 + H]
    for p in ["Tokke", "Vinje", "Lio", "Songa", "Kjela"]:
        k = PLANTS.index(p); u = [g for g in F.UNITS if F.plant_of(g) == p]
        on = np.array([[row_[f"result_committed_{g}_t{t}"] for t in range(H)] for g in u]).sum(0)
        print(p, "LP q/qmax mean", out["qfrac"][k].mean().round(2), "| units ON mean", (on / len(u)).mean().round(2),
              "| corr(LP q, #ON)", np.corrcoef(out["qfrac"][k], on)[0, 1].round(2) if on.std() > 0 and out["qfrac"][k].std() > 0 else "const",
              "| LP wv range", out["wv_dyn"][p].min().round(1), out["wv_dyn"][p].max().round(1), "| price", price.min().round(0), price.max().round(0))

In [ ]:
%%writefile work_nb2/lp_unit.py
"""Simplified (linear) short-term hydro scheduling model of Tokke-Vinje, one week per case.
Maximise  sum_t price_t * production_t  +  sum_r water_value_r * energy_r * V_r,end  -  small flow/penalty costs
s.t. reservoir water balance each hour, plant/river/tunnel flow limits, min-flow / min-volume (soft).
No start costs, no integer on/off, fixed head from the initial levels -> fast LP (HiGHS)."""
import numpy as np, pandas as pd, yaml, scipy.sparse as sp
from scipy.optimize import linprog, milp, LinearConstraint, Bounds
import feats as F
H = 168; DT = 0.0036                       # m3/s for one hour -> Mm3
cfg = yaml.safe_load(open("data/extended/Tokke_Vinje_topology.yaml"))
M = cfg["model"]; CON = cfg["connections"]
RES = list(F.volume.columns); rid = {r: i for i, r in enumerate(RES)}
RIV = list(M["river"]); TUN = list(M.get("tunnel", {})); PLANTS = list(M["plant"])
gens = {p: [g for g in M["generator"] if any(c["from"] == g and c["to"] == p for c in CON)] for p in PLANTS}
qmax_unit = {g: max(max(cv["x"]) for cv in M["generator"][g]["turb_eff_curves"]) for g in M["generator"]}
QMAXP = {p: sum(qmax_unit[g] for g in gens[p]) for p in PLANTS}
out_e = lambda name: [c for c in CON if c["from"] == name]
in_e = lambda name: [c for c in CON if c["to"] == name]
def level(r, v):
    c = M["reservoir"][r]["vol_head"]; return float(np.interp(v, c["x"], c["y"]))
# plant source / destination
P_SRC = {p: [(c["from_type"], c["from"]) for c in in_e(p) if c["from_type"] in ("reservoir", "tunnel")] for p in PLANTS}
P_DST = {p: [(c["to_type"], c["to"]) for c in out_e(p)] for p in PLANTS}
def resolve(kind, name):
    """follow rivers/tunnels downstream until a reservoir (or the sea) is reached"""
    seen = 0
    while kind == "river" and seen < 10:
        nxt = [(c["to_type"], c["to"]) for c in out_e(name)]
        if not nxt: return None
        kind, name = nxt[0]; seen += 1
    if kind == "tunnel":
        nxt = [(c["to_type"], c["to"]) for c in out_e(name)]
        return nxt[0][1] if nxt and nxt[0][0] == "reservoir" else None
    return name if kind == "reservoir" else None
def plant_up_res(p):
    k, n = P_SRC[p][0]
    if k == "reservoir": return n
    return [c["from"] for c in in_e(n) if c["from_type"] == "reservoir"][0]
# energy coefficient used to convert each reservoir's water value: the first plant its water reaches
def first_plant(r, depth=0):
    for c in out_e(r):
        if c["to_type"] == "plant": return c["to"]
    for c in out_e(r):
        if c["to_type"] == "tunnel":
            for d in out_e(c["to"]):
                if d["to_type"] == "plant": return d["to"]
                if d["to_type"] == "reservoir" and depth < 4: return first_plant(d["to"], depth + 1)
    for c in out_e(r):
        if c["to_type"] == "river" and c["to"].startswith(("r_", "b_")):
            dst = resolve("river", c["to"])
            if dst and depth < 4: return first_plant(dst, depth + 1)
    return None
FIRST = {r: first_plant(r) for r in RES}
# commitment order inside a plant, read from the 2015-2020 labels (Tokke: G1, G3, G4, G2; Vinje: G1, G2, G3)
PRIORITY = {"Tokke_G1": 0, "Tokke_G3": 1, "Tokke_G4": 2, "Tokke_G2": 3, "Vinje_G1": 0, "Vinje_G2": 1, "Vinje_G3": 2}

def solve_case(start, return_all=False, mip=False, time_limit=5.0):
    s = pd.Timestamp(start, tz="UTC"); i0 = F.pidx.get_loc(s); price = F.pv[i0:i0 + H].astype(float)
    d0 = s.normalize(); d7 = d0 + pd.Timedelta(days=7)
    v0 = F.volume.loc[d0, RES].values.astype(float); wv = F.water_value.loc[d7, RES].values.astype(float)
    j0 = F.inflow.index.get_loc(s); infl = F.inflow.iloc[j0:j0 + H]
    k0 = F.min_flow.index.get_loc(s); mf = F.min_flow.iloc[k0:k0 + H]; mv = F.min_vol.iloc[k0:k0 + H]
    lev0 = {r: level(r, v0[rid[r]]) for r in RES}
    # energy coefficient MW per m3/s for each plant from the initial head
    E = {}
    for p in PLANTS:
        up = plant_up_res(p); dst = P_DST[p][0] if P_DST[p] else ("sea", None)
        tail = M["plant"][p].get("outlet_line", 0)
        if dst[0] == "reservoir": tail = max(tail, lev0[dst[1]])
        eta = np.mean([max(max(cv["y"]) for cv in M["generator"][g]["turb_eff_curves"]) for g in gens[p]]) / 100 * 0.98
        E[p] = 9.81 * max(lev0[up] - tail, 1.0) * eta / 1000
    def plant_dst_res(p):
        if not P_DST[p]: return None
        k, n = P_DST[p][0]
        return n if k == "reservoir" else resolve(k, n)
    def ecum(r, depth=0):      # energy of 1 m3/s-hour of water in r on its way to the sea
        p = FIRST[r]
        if p is None or depth > 6: return 0.0
        d = plant_dst_res(p)
        return E[p] + (ecum(d, depth + 1) if d else 0.0)
    Er = np.array([ecum(r) for r in RES])
    # ---------------- variables
    names = []; lb = []; ub = []; cost = []
    def add(n, lo, hi, c): names.append(n); lb.append(lo); ub.append(hi); cost.append(c); return len(names) - 1
    V = np.array([[add(("V", r, t), 0, max(M["reservoir"][r]["max_vol"], v0[rid[r]]), 0) for t in range(H + 1)] for r in RES])
    wv_plant = {p: wv[rid[plant_up_res(p)]] for p in PLANTS}
    # each plant pays its own upstream water value per MWh it produces (empirical SHOP behaviour, see notebook)
    Q = {p: [add(("Q", p, t), 0, QMAXP[p], -(price[t] - wv_plant[p]) * E[p]) for t in range(H)] for p in PLANTS}
    rcost = lambda n: 1.0 if n.startswith("r_") else 2000.0   # natural rivers flow freely; gates (b_) and spill (s_) only when forced
    Fv = {n: [add(("F", n, t), 0, M["river"][n].get("max_flow_const", 1e4), rcost(n)) for t in range(H)] for n in RIV}
    T = {n: [add(("T", n, t), -1000 if n == "Totak_Vaamarvatn" else 0, 1000, 0.01) for t in range(H)] for n in TUN}
    over = {r: [add(("over", r, t), 0, 1e4, 1e6) for t in range(H)] for r in RES}      # emergency overflow
    MFS = {n: [add(("mfs", n, t), 0, 1e4, 5000.0) for t in range(H)] for n in mf.columns}  # min-flow slack
    MVS = {r: [add(("mvs", r, t), 0, 1e4, 1e5) for t in range(H)] for r in mv.columns}  # min-volume slack
    # ---- unit level (relaxed commitment): q_g <= qmax_g z_g, q_g >= qmin_g z_g, start-up s_g >= z_t - z_{t-1}
    UQ, UZ, US = {}, {}, {}
    for p in PLANTS:
        for g in gens[p]:
            curves = M["generator"][g]["turb_eff_curves"]
            qmin = min(min(cv["x"]) for cv in curves); rk = PRIORITY.get(g, 0)
            UQ[g] = [add(("uq", g, t), 0, qmax_unit[g], 0.0) for t in range(H)]
            UZ[g] = [add(("uz", g, t), 0, 1, 0.01 * rk) for t in range(H)]        # tiny cost = commitment order
            US[g] = [add(("us", g, t), 0, 1, float(M["generator"][g].get("startcost_const", 0))) for t in range(H)]
            UQ[g + "_qmin"] = qmin
    nv = len(names); cost = np.array(cost, float)
    # no terminal valuation: the opportunity cost of water is charged when it is released
    # ---------------- equality constraints
    rows, cols, vals, rhs = [], [], [], []; nrow = [0]
    def row(entries, b):
        for c, v in entries: rows.append(nrow[0]); cols.append(c); vals.append(v)
        rhs.append(b); nrow[0] += 1; return nrow[0] - 1
    # river flows fed only by other rivers/plants are pass-through: F = sum(inputs) + natural inflow
    passthrough = [n for n in RIV if not any(c["from_type"] == "reservoir" for c in in_e(n))]
    for n in passthrough:
        for t in range(H):
            ent = [(Fv[n][t], 1.0)]
            for c in in_e(n):
                if c["from_type"] == "river": ent.append((Fv[c["from"]][t], -1.0))
                if c["from_type"] == "plant": ent.append((Q[c["from"]][t], -1.0))
            nat = float(infl[n].iloc[t]) if n in infl.columns else 0.0
            row(ent, nat)
    for p in PLANTS:   # plant flow = sum of unit flows
        for t in range(H): row([(Q[p][t], 1.0)] + [(UQ[g][t], -1.0) for g in gens[p]], 0.0)
    for p in PLANTS:   # plants fed by tunnels: Q = sum of tunnel flows
        if P_SRC[p][0][0] == "tunnel":
            for t in range(H): row([(Q[p][t], 1.0)] + [(T[k][t], -1.0) for kind, k in P_SRC[p]], 0.0)
    bal = np.zeros((len(RES), H), dtype=int)
    for r in RES:
        i = rid[r]
        for t in range(H):
            ent = [(V[i, t + 1], 1.0), (V[i, t], -1.0), (over[r][t], DT)]
            for c in out_e(r):
                if c["to_type"] == "plant": ent.append((Q[c["to"]][t], DT))
                elif c["to_type"] == "river": ent.append((Fv[c["to"]][t], DT))
                elif c["to_type"] == "tunnel": ent.append((T[c["to"]][t], DT))
            for c in in_e(r):
                if c["from_type"] == "plant": ent.append((Q[c["from"]][t], -DT))
                elif c["from_type"] == "river": ent.append((Fv[c["from"]][t], -DT))
                elif c["from_type"] == "tunnel" and c["from"] in T: ent.append((T[c["from"]][t], -DT))
            bal[i, t] = row(ent, DT * float(infl[r].iloc[t]))
        row([(V[i, 0], 1.0)], v0[i])
    A_eq = sp.csr_matrix((vals, (rows, cols)), shape=(nrow[0], nv)); b_eq = np.array(rhs)
    # ---------------- inequalities (<=)
    rows, cols, vals, rhs = [], [], [], []; nrow = [0]
    for n in mf.columns:      # flow + slack >= min flow  ->  -flow - slack <= -min
        for t in range(H):
            row([(Fv[n][t], -1.0), (MFS[n][t], -1.0)], -float(mf[n].iloc[t]))
    for r in mv.columns:
        for t in range(H):
            row([(V[rid[r], t + 1], -1.0), (MVS[r][t], -1.0)], -float(mv[r].iloc[t]))
    for p in PLANTS:
        for g in gens[p]:
            for t in range(H):
                row([(UQ[g][t], 1.0), (UZ[g][t], -qmax_unit[g])], 0.0)
                row([(UQ[g][t], -1.0), (UZ[g][t], UQ[g + "_qmin"])], 0.0)
                if t > 0: row([(UZ[g][t], 1.0), (UZ[g][t - 1], -1.0), (US[g][t], -1.0)], 0.0)
    A_ub = sp.csr_matrix((vals, (rows, cols)), shape=(nrow[0], nv)); b_ub = np.array(rhs)
    if mip:   # true on/off decisions: z binary, real minimum flow, start-up costs
        integ = np.zeros(nv); 
        for g in UZ: integ[UZ[g]] = 1
        r = milp(cost, integrality=integ, bounds=Bounds(lb, ub),
                 constraints=[LinearConstraint(A_eq, b_eq, b_eq), LinearConstraint(A_ub, -np.inf, b_ub)],
                 options={"time_limit": time_limit, "mip_rel_gap": 0.005, "disp": False})
        assert r.x is not None, r.message
        x = r.x; z = np.array([[x[UZ[g][t]] for t in range(H)] for g in [g for p in PLANTS for g in gens[p]]])
        return z.astype(np.float32), r.status, getattr(r, "mip_gap", None)
    res = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=list(zip(lb, ub)), method="highs")
    assert res.status == 0, res.message
    x = res.x; mu = res.eqlin.marginals                       # dual of balance: d(cost)/d(rhs) [EUR per Mm3]
    qfrac = np.array([[x[Q[p][t]] / QMAXP[p] for t in range(H)] for p in PLANTS])
    wv_dyn = {}                                                 # LP water value of each plant's upstream water, EUR/MWh
    for p in PLANTS:
        up = plant_up_res(p)
        dst = None if not P_DST[p] else (P_DST[p][0][1] if P_DST[p][0][0] == "reservoir" else resolve(*P_DST[p][0]))
        lam_up = -mu[bal[rid[up]]]; lam_dn = -mu[bal[rid[dst]]] if dst else np.zeros(H)
        wv_dyn[p] = wv_plant[p] + (lam_up - lam_dn) * DT / E[p]   # break-even price incl. capacity effects
    UNITS_ORDER = [g for p in PLANTS for g in gens[p]]
    z = np.array([[x[UZ[g][t]] for t in range(H)] for g in UNITS_ORDER]); su = np.array([[x[US[g][t]] for t in range(H)] for g in UNITS_ORDER])
    if return_all: return dict(qfrac=qfrac, wv_dyn=wv_dyn, E=E, Er=Er, obj=res.fun, x=x, Q=Q, V=V, names=names)
    return qfrac.astype(np.float32), np.array([wv_dyn[p] for p in PLANTS], dtype=np.float32), z.astype(np.float32), su.astype(np.float32), UNITS_ORDER

if __name__ == "__main__":
    import time
    t0 = time.time(); out = solve_case("2022-03-07", return_all=True); print("solve", round(time.time() - t0, 2), "s")
    print("E (MW per m3/s):", {k: round(v, 2) for k, v in out["E"].items()})
    uc = F.uc; row_ = uc[uc.starttime == "2022-03-07"].iloc[0]
    s = pd.Timestamp("2022-03-07", tz="UTC"); i0 = F.pidx.get_loc(s); price = F.pv[i0:i0 + H]
    for p in ["Tokke", "Vinje", "Lio", "Songa", "Kjela"]:
        k = PLANTS.index(p); u = [g for g in F.UNITS if F.plant_of(g) == p]
        on = np.array([[row_[f"result_committed_{g}_t{t}"] for t in range(H)] for g in u]).sum(0)
        print(p, "LP q/qmax mean", out["qfrac"][k].mean().round(2), "| units ON mean", (on / len(u)).mean().round(2),
              "| corr(LP q, #ON)", np.corrcoef(out["qfrac"][k], on)[0, 1].round(2) if on.std() > 0 and out["qfrac"][k].std() > 0 else "const",
              "| LP wv range", out["wv_dyn"][p].min().round(1), out["wv_dyn"][p].max().round(1), "| price", price.min().round(0), price.max().round(0))

In [ ]:
%%writefile work_nb2/run_lp.py
"""Stage 1: the plant and unit LP relaxations for every labelled and test case (physics-informed features).
Usage: python run_lp.py plant|unit   (run from the project folder; writes into $V12C_WORK)"""
import os, sys, time, numpy as np, pandas as pd
import feats as F
WORK = os.environ["V12C_WORK"]; which = sys.argv[1]
starts = list(pd.concat([F.uc.starttime, F.mapping.starttime]).values)       # 2922 labelled + 725 test cases
t0 = time.time()
if which == "plant":
    import lp_plant as L
    Q, W = [], []
    for i, s in enumerate(starts):
        q, w = L.solve_case(s); Q.append(q); W.append(w)
        if i % 500 == 0: print("plant LP", i, "/", len(starts), round(time.time() - t0), "s", flush=True)
    np.savez_compressed(f"{WORK}/lp_feats_all.npz", starts=np.array(starts), Q=np.stack(Q), W=np.stack(W))
else:
    import lp_unit as L
    Z, S, order = [], [], None
    for i, s in enumerate(starts):
        q, w, z, su, order = L.solve_case(s); Z.append(z); S.append(su)
        if i % 500 == 0: print("unit LP", i, "/", len(starts), round(time.time() - t0), "s", flush=True)
    np.savez_compressed(f"{WORK}/lp_unit_all.npz", starts=np.array(starts), Z=np.stack(Z), S=np.stack(S), order=np.array(order))
print(which, "LP done", round(time.time() - t0), "s")

In [ ]:
run_stage(["lp_feats_all.npz", "lp_unit_all.npz"], ["run_lp.py", "plant"], ["run_lp.py", "unit"])

## 3. The 16-channel P–Q LP
A P–Q LP was also tested earlier in the project. It uses five-point power–flow (P–Q) curves for each unit and simple tunnel and spill rules.
Its 16 output channels per unit and hour are extra features for A-LightGBM. Only the LP part of that code is kept here.

In [ ]:
%%writefile work_nb2/legacy_pq.py
from sklearn.metrics import roc_auc_score
display=print


import os,sys,json,time,gc,subprocess,platform
from pathlib import Path
import numpy as np,pandas as pd,yaml,lightgbm as lgb
from scipy.special import expit,logit
from threadpoolctl import threadpool_limits


# Set SINTEF_DATA_DIR to the supplied raw-data directory before running.
PROJECT=Path(os.environ.get('SINTEF_DATA_DIR','.')).resolve()
VARIANT='confidence'
SAVE_DIR=Path(os.environ.get('SINTEF_OUTPUT_DIR','prediction_outputs')).resolve()/VARIANT
WORK_DIR=Path(os.environ.get('SINTEF_WORK_DIR','work')).resolve()/VARIANT
SAVE_DIR.mkdir(parents=True,exist_ok=True)
WORK_DIR.mkdir(parents=True,exist_ok=True)
assert (PROJECT/'data/kernel/Unit_commitment_decisions.csv').is_file(), 'Set SINTEF_DATA_DIR to the given raw-data folder.'
SEED,H=4173,168
thread_limit=threadpool_limits(limits=int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1))))
DEVICE=os.environ.get('SINTEF_DEVICE','auto')
assert DEVICE in ['auto','cpu','mps','cuda']
notebook_start=time.perf_counter()

print('Python',platform.python_version(),'NumPy',np.__version__,'LightGBM',lgb.__version__)
print('Output variant:',VARIANT,'Requested device:',DEVICE)


def load_inputs(project=PROJECT):
    uc = pd.read_csv(project / 'data/kernel/Unit_commitment_decisions.csv')
    uc['starttime'] = pd.to_datetime(uc.starttime, utc=True)
    sample = pd.read_csv(project / 'sample_submission.csv')
    mapping = pd.read_csv(project / 'prediction_mapping.csv')
    mapping['starttime'] = pd.to_datetime(mapping.starttime, utc=True)
    labels = list(sample.columns.drop('Run No'))
    units = list(dict.fromkeys((c.removeprefix('result_committed_').rsplit('_t', 1)[0] for c in labels)))
    ku = np.array([units.index(c.removeprefix('result_committed_').rsplit('_t', 1)[0]) for c in labels])
    ht = np.array([int(c.rsplit('_t', 1)[1]) for c in labels])
    assert len(labels) == len(units) * H == len(set(zip(ku, ht))) == 2352
    assert uc[labels].isin([0, 1]).all().all()
    series = {}
    for key, rel in {'price': 'kernel/Historical_day_ahead_price_2015_2025.csv', 'inflow': 'kernel/Historical_inflow_1958_2025.csv', 'volume': 'kernel/Historical_volume_2015_2024.csv', 'wv': 'kernel/Synthetic_water_value_2015_2024.csv', 'flow': 'extended/Constraint_min_flow.csv', 'minvol': 'extended/Constraint_min_volume.csv'}.items():
        df = pd.read_csv(project / 'data' / rel)
        df['date'] = pd.to_datetime(df.date, utc=True)
        assert not df.date.duplicated().any()
        df = df.set_index('date').sort_index()
        if key == 'price':
            df = df.resample('h').mean()
        series[key] = df
    config = yaml.safe_load((project / 'data/extended/Tokke_Vinje_topology.yaml').read_text())
    return (uc, sample, mapping, labels, units, ku, ht, series, config)

class CaseFeatures:

    def __init__(self, starts, series, config, units, ku, ht):
        self.units, self.ku, self.ht = (units, ku, ht)
        self.L = len(ku)
        self.n = len(starts)
        self.starts = pd.DatetimeIndex(starts)
        self.reservoirs = list(series['volume'].columns)
        self.rid = {r: i for i, r in enumerate(self.reservoirs)}
        idx = pd.to_datetime((self.starts.tz_convert('UTC').tz_localize(None).to_numpy()[:, None] + np.arange(H)[None, :] * np.timedelta64(1, 'h')).ravel(), utc=True)

        def window(key):
            a = series[key].reindex(idx).to_numpy(dtype=np.float32)
            assert np.isfinite(a).all(), f'Missing hourly data: {key}'
            return a.reshape(self.n, H, -1)
        p = window('price')[:, :, 0]
        self.p = p
        self.mean = p.mean(axis=1)
        self.std = np.maximum(p.std(axis=1), 1.0)
        self.pscale = np.maximum(np.abs(self.mean), 10.0)
        self.vol = series['volume'].reindex(self.starts)[self.reservoirs].to_numpy(dtype=np.float32)
        self.wv = series['wv'].reindex(self.starts + pd.Timedelta(days=7))[self.reservoirs].to_numpy(dtype=np.float32)
        self.wv0 = series['wv'].reindex(self.starts)[self.reservoirs].to_numpy(dtype=np.float32)
        assert np.isfinite(self.vol).all() and np.isfinite(self.wv).all()
        self.caps = np.array([config['model']['reservoir'][r]['max_vol'] for r in self.reservoirs], dtype=np.float32)
        inflow = window('inflow')
        inf_cols = list(series['inflow'].columns)
        inflow = inflow[:, :, [inf_cols.index(r) for r in self.reservoirs]]
        self.inf = inflow
        self.inf_total = inflow.sum(axis=1) * 3600 / 1000000.0
        self.inf_cum = np.cumsum(inflow, axis=1) * 3600 / 1000000.0
        self.flow = window('flow')
        self.minvol = window('minvol')
        self.mv_all = np.zeros((self.n, H, len(self.reservoirs)), dtype=np.float32)
        for j, r in enumerate(series['minvol'].columns):
            self.mv_all[:, :, self.rid[r]] = self.minvol[:, :, j]
        self.week_minvol = self.mv_all.max(axis=1)
        self.connections = config['connections']
        forward, backward = ({}, {})
        for c in self.connections:
            a, b = ((c['from_type'], c['from']), (c['to_type'], c['to']))
            forward.setdefault(a, []).append(b)
            backward.setdefault(b, []).append(a)

        def nearest(node, graph):
            result, todo, seen = (set(), list(graph.get(node, [])), set())
            while todo:
                typ, name = todo.pop()
                if (typ, name) in seen:
                    continue
                seen.add((typ, name))
                if typ == 'reservoir':
                    result.add(name)
                elif typ in ['river', 'tunnel']:
                    todo.extend(graph.get((typ, name), []))
            return sorted(result)
        self.topology = []
        self.upweights = np.zeros((len(units), len(self.reservoirs)), dtype=np.float32)
        self.downweights = self.upweights.copy()
        self.feedweights = self.upweights.copy()
        self.qmax, self.pmax, self.pmin, self.startcost = ([], [], [], [])
        for k, u in enumerate(units):
            plant = next((c['to'] for c in self.connections if c['from_type'] == 'generator' and c['from'] == u and (c['to_type'] == 'plant')))
            up = nearest(('plant', plant), backward)
            down = nearest(('plant', plant), forward)
            assert up, (u, 'no upstream')
            feed = set(up)
            for r in up:
                if config['model']['reservoir'][r]['max_vol'] < 1:
                    feed.update(nearest(('reservoir', r), backward))
            for r in up:
                self.upweights[k, self.rid[r]] = 1 / len(up)
            for r in down:
                self.downweights[k, self.rid[r]] = 1 / max(len(down), 1)
            for r in feed:
                self.feedweights[k, self.rid[r]] = 1
            g = config['model']['generator'][u]
            self.qmax.append(max((max(curve['x']) for curve in g['turb_eff_curves'])))
            self.pmax.append(g['p_max'])
            self.pmin.append(g['p_min'])
            self.startcost.append(g.get('startcost_const', 0))
            self.topology.append(dict(unit=u, plant=plant, upstream=', '.join(up), downstream=', '.join(down), storage_feeders=', '.join(sorted(feed))))
        self.qmax = np.array(self.qmax, dtype=np.float32)
        self.pmax = np.array(self.pmax, dtype=np.float32)
        self.pmin = np.array(self.pmin, dtype=np.float32)
        self.startcost = np.array(self.startcost, dtype=np.float32)
        self.feedcap = self.feedweights @ self.caps
        self.upwv = self.wv @ self.upweights.T
        self.downwv = self.wv @ self.downweights.T
        self.netwv = self.upwv - self.downwv
        self.upwv0 = self.wv0 @ self.upweights.T
        self.downwv0 = self.wv0 @ self.downweights.T
        self.feedvol = self.vol @ self.feedweights.T
        self.downvol = self.vol / self.caps @ self.downweights.T
        self.infweek = self.inf_total @ self.feedweights.T
        self.slack = (self.vol - self.week_minvol) @ self.feedweights.T
        self.profitable = (p[:, :, None] > self.upwv[:, None, :]).mean(axis=1).astype(np.float32)
        self.profitable_net = (p[:, :, None] > self.netwv[:, None, :]).mean(axis=1).astype(np.float32)
        rank = pd.DataFrame(p).rank(axis=1, pct=True).to_numpy(dtype=np.float32)
        dayrank = pd.DataFrame(p.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy(dtype=np.float32).reshape(self.n, H)
        self.static_hour = {'price_week_rank': rank, 'price_day_rank': dayrank, 'price_z': (p - self.mean[:, None]) / self.std[:, None], 'price_relative_mean': p / self.pscale[:, None], 'price_delta_prev': np.diff(p, axis=1, prepend=p[:, :1]) / self.std[:, None], 'price_delta_next': np.diff(p, axis=1, append=p[:, -1:]) / self.std[:, None], 'price_rolling_6h_z': (pd.DataFrame(p).T.rolling(6, min_periods=1, center=True).mean().T.to_numpy() - self.mean[:, None]) / self.std[:, None]}
        self.local = idx.tz_convert('Europe/Oslo')
        self.mon = self.starts.month.to_numpy()
        self.hour = np.array(self.local.hour).reshape(self.n, H)
        self.dow = np.array(self.local.dayofweek).reshape(self.n, H)

    def X(self, ids, mode='domain'):
        ids = np.asarray(ids)
        c, l = (ids // self.L, ids % self.L)
        u, t = (self.ku[l], self.ht[l])
        p = self.p[c, t]
        scale = self.pscale[c]
        std = self.std[c]
        names = []
        values = []

        def put(n, a):
            names.append(n)
            values.append(np.asarray(a, dtype=np.float32))
        put('unit', u)
        put('t', t)
        put('hour', self.hour[c, t])
        put('weekday', self.dow[c, t])
        put('month', self.mon[c])
        put('season_sin', np.sin(2 * np.pi * self.starts.dayofyear.to_numpy()[c] / 365.25))
        put('season_cos', np.cos(2 * np.pi * self.starts.dayofyear.to_numpy()[c] / 365.25))
        for n, a in self.static_hour.items():
            put(n, a[c, t])
        put('week_volatility', self.std[c] / scale)
        for j, r in enumerate(self.reservoirs):
            put('storage_' + r, self.vol[c, j] / self.caps[j])
            put('wv_relative_' + r, self.wv[c, j] / scale)
        for j in range(self.flow.shape[2]):
            put('minflow_' + str(j), self.flow[c, t, j])
        for j, r in enumerate(self.reservoirs):
            if np.any(self.week_minvol[:, j] > 0):
                put('storage_slack_' + r, (self.vol[c, j] - self.week_minvol[c, j]) / self.caps[j])
        if mode in ['domain', 'domain_raw']:
            up = self.upwv[c, u]
            down = self.downwv[c, u]
            net = self.netwv[c, u]
            put('price_minus_upwv_z', (p - up) / std)
            put('price_over_upwv', p / np.maximum(np.abs(up), 5.0))
            put('price_minus_netwv_z', (p - net) / std)
            put('price_over_netwv', p / np.maximum(np.abs(net), 5.0))
            put('price_minus_netwv0_z', (p - self.upwv0[c, u] + self.downwv0[c, u]) / std)
            put('upwv_relative', up / scale)
            put('downwv_relative', down / scale)
            put('wv_change_relative', (up - self.upwv0[c, u]) / scale)
            put('profitable_week_share', self.profitable[c, u])
            put('above_net_week_share', self.profitable_net[c, u])
            put('feed_storage_fraction', self.feedvol[c, u] / self.feedcap[u])
            put('down_storage_fraction', self.downvol[c, u])
            put('week_inflow_capacity', self.infweek[c, u] / self.feedcap[u])
            put('storage_slack_fraction', self.slack[c, u] / self.feedcap[u])
            put('water_budget_heuristic', (self.slack[c, u] + self.infweek[c, u]) * 1000000.0 / (self.qmax[u] * H * 3600))
            put('stored_budget_heuristic', self.slack[c, u] * 1000000.0 / (self.qmax[u] * H * 3600))
            put('startcost_relative', self.startcost[u] / np.maximum(self.pmax[u] * scale, 1))
            put('minload_fraction', self.pmin[u] / self.pmax[u])
        if mode == 'domain_raw':
            put('raw_price', p)
            put('raw_week_mean', self.mean[c])
            put('raw_upwv', self.upwv[c, u])
        a = np.column_stack(values)
        assert np.isfinite(a).all()
        self.names = names
        return a

def get_split(uc, year):
    b = pd.Timestamp(f'{year}-01-01', tz='UTC')
    tr = np.flatnonzero((uc.starttime + pd.Timedelta(hours=H - 1) < b).to_numpy())
    va = np.flatnonzero(((uc.starttime >= b) & (uc.starttime < f'{year + 1}-01-01')).to_numpy())
    return (tr, va)

def sample_rows(cases, L, n, seed):
    rng = np.random.default_rng(seed)
    s = rng.choice(len(cases) * L, min(n, len(cases) * L), replace=False)
    return cases[s // L] * L + s % L

def predict(model, features, ids, mode, batch=100000):
    return np.concatenate([model.predict(features.X(ids[i:i + batch], mode), num_threads=int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1)))) for i in range(0, len(ids), batch)])

class RichHydroFeatures(CaseFeatures):

    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        index = pd.to_datetime((self.starts.tz_convert('UTC').tz_localize(None).to_numpy()[:, None] + np.arange(H)[None, :] * np.timedelta64(1, 'h')).ravel(), utc=True)
        inflow = series['inflow'].reindex(index)
        rivers = [c for c in inflow if c not in self.reservoirs]
        self.river_inflow = inflow[rivers].sum(axis=1).to_numpy(dtype=np.float32).reshape(self.n, H)
        self.catchment = np.zeros_like(self.feedweights)
        backward = {}
        for edge in config['connections']:
            backward.setdefault((edge['to_type'], edge['to']), []).append((edge['from_type'], edge['from']))
        for k, row in enumerate(self.topology):
            todo = [('reservoir', r) for r in row['upstream'].split(', ')]
            seen = set()
            while todo:
                node = todo.pop()
                if node in seen:
                    continue
                seen.add(node)
                if node[0] == 'reservoir':
                    self.catchment[k, self.rid[node[1]]] = 1
                if node[0] != 'generator':
                    todo.extend(backward.get(node, []))
        self.catchcap = self.catchment @ self.caps
        self.catchvol = self.vol @ self.catchment.T
        self.catchinf = self.inf_total @ self.catchment.T
        self.week_flow_mean = self.flow.mean(axis=1)
        self.week_flow_max = self.flow.max(axis=1)
        self.heads = np.empty_like(self.vol)
        for j, r in enumerate(self.reservoirs):
            curve = config['model']['reservoir'][r]['vol_head']
            self.heads[:, j] = np.interp(self.vol[:, j], curve['x'], curve['y'])
        self.uphead = self.heads @ self.upweights.T
        self.downhead = self.heads @ self.downweights.T
        self.outlet = np.array([config['model']['plant'][r['plant']].get('outlet_line', 0) for r in self.topology], dtype=np.float32)
        for u in range(len(units)):
            if not self.downweights[u].any():
                self.downhead[:, u] = self.outlet[u]
        self.nethead = np.maximum(self.uphead - self.downhead, 1.0)
        self.day_profitable = np.zeros((self.n, 7, len(units)), dtype=np.float32)
        self.day_positive_margin = np.zeros_like(self.day_profitable)
        for k in range(len(units)):
            margin = self.p - self.upwv[:, k, None]
            self.day_profitable[:, :, k] = (margin.reshape(self.n, 7, 24) > 0).mean(axis=2)
            self.day_positive_margin[:, :, k] = np.maximum(margin, 0).reshape(self.n, 7, 24).sum(axis=2)
        self.daily_max = self.p.reshape(self.n, 7, 24).max(axis=2)
        self.daily_mean = self.p.reshape(self.n, 7, 24).mean(axis=2)
        self.daily_min = self.p.reshape(self.n, 7, 24).min(axis=2)
        self.small_reservoir_ids = np.flatnonzero(self.caps < 30)

    def X(self, ids, mode='rich'):
        base = super().X(ids, 'domain')
        names = self.names.copy()
        ids = np.asarray(ids)
        c, l = (ids // self.L, ids % self.L)
        u, t = (self.ku[l], self.ht[l])
        extra = []

        def put(name, value):
            names.append(name)
            extra.append(np.asarray(value, dtype=np.float32))
        for j, r in enumerate(self.reservoirs):
            put('week_natural_inflow_capacity_' + r, self.inf_total[c, j] / self.caps[j])
        hourly = np.einsum('ij,ij->i', self.inf[c, t, :], self.feedweights[u])
        cumulative = np.einsum('ij,ij->i', self.inf_cum[c, t, :], self.feedweights[u])
        down_inflow = np.einsum('ij,ij->i', self.inf[c, t, :], self.downweights[u])
        put('hour_inflow_fraction_qmax', hourly / self.qmax[u])
        put('hour_downstream_inflow', down_inflow)
        put('cumulative_natural_inflow_fraction', cumulative / self.feedcap[u])
        put('remaining_natural_inflow_fraction', (self.infweek[c, u] - cumulative) / self.feedcap[u])
        put('volume_without_release_fraction', (self.feedvol[c, u] + cumulative) / self.feedcap[u])
        put('river_natural_inflow', self.river_inflow[c, t])
        put('river_week_natural_mean', self.river_inflow.mean(axis=1)[c])
        put('catchment_storage_fraction', self.catchvol[c, u] / self.catchcap[u])
        put('catchment_week_inflow_fraction', self.catchinf[c, u] / self.catchcap[u])
        put('catchment_budget_heuristic', (self.catchvol[c, u] + self.catchinf[c, u]) * 1000000.0 / (self.qmax[u] * H * 3600))
        put('initial_upstream_head', self.uphead[c, u])
        put('initial_net_head', self.nethead[c, u])
        put('head_implied_max_power', 9.81 * self.qmax[u] * self.nethead[c, u] / 1000)
        for j in range(self.flow.shape[2]):
            put('minflow_week_mean_' + str(j), self.week_flow_mean[c, j])
            put('minflow_week_max_' + str(j), self.week_flow_max[c, j])
        for lag in [2, 6, 12, 24]:
            put('price_prev_' + str(lag) + '_z', (self.p[c, np.maximum(t - lag, 0)] - self.mean[c]) / self.std[c])
            put('price_next_' + str(lag) + '_z', (self.p[c, np.minimum(t + lag, H - 1)] - self.mean[c]) / self.std[c])
        day = t // 24
        put('day_profitable_fraction', self.day_profitable[c, day, u])
        put('day_positive_margin_over_startcost', self.day_positive_margin[c, day, u] * self.pmax[u] / np.maximum(self.startcost[u], 1))
        put('day_max_price_z', (self.daily_max[c, day] - self.mean[c]) / self.std[c])
        put('day_mean_price_z', (self.daily_mean[c, day] - self.mean[c]) / self.std[c])
        put('day_min_price_z', (self.daily_min[c, day] - self.mean[c]) / self.std[c])
        for j in self.small_reservoir_ids:
            put('small_res_cum_inflow_' + self.reservoirs[j], self.inf_cum[c, t, j] / self.caps[j])
        result = np.column_stack([base, *extra])
        assert np.isfinite(result).all()
        self.names = names
        return result

def model_params(leaves=31, metric='binary_logloss'):
    return dict(objective='binary', metric=metric, learning_rate=0.045, num_leaves=leaves, min_data_in_leaf=100, lambda_l2=10, feature_fraction=0.85, bagging_fraction=0.9, bagging_freq=1, num_threads=int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1))), verbosity=-1, seed=SEED, deterministic=True, force_col_wise=True, max_bin=127)

def fit_pooled_rich(features, Y, train_cases, stop_cases=None, rounds=1800, n=1200000):
    ids = sample_rows(train_cases, features.L, n, SEED + 80)
    X = features.X(ids)
    ds = lgb.Dataset(X, label=Y.ravel()[ids], feature_name=features.names, categorical_feature=['unit'])
    callbacks = [lgb.log_evaluation(0)]
    validation = []
    if stop_cases is not None:
        stop_ids = sample_rows(stop_cases, features.L, 150000, SEED + 81)
        validation = [lgb.Dataset(features.X(stop_ids), label=Y.ravel()[stop_ids], reference=ds)]
        callbacks.append(lgb.early_stopping(100, verbose=False))
    tick = time.perf_counter()
    m = lgb.train(model_params(63), ds, num_boost_round=rounds, valid_sets=validation, callbacks=callbacks)
    print('Pooled fit:', len(ids), 'rows;', m.best_iteration or rounds, 'rounds;', round(time.perf_counter() - tick, 1), 's', flush=True)
    del X, ds, validation
    gc.collect()
    return m

def unit_ids(cases, features, k):
    cols = np.flatnonzero(features.ku == k)
    return (np.asarray(cases)[:, None] * features.L + cols[None, :]).ravel()

def fit_specialists(features, Y, train_cases, stop_cases=None, rounds=None, n_per_unit=240000):
    models = {}
    timings = []
    for k, u in enumerate(features.units):
        tick = time.perf_counter()
        ids = unit_ids(train_cases, features, k)
        if len(ids) > n_per_unit:
            ids = np.random.default_rng(SEED + k).choice(ids, n_per_unit, replace=False)
        X = features.X(ids)
        ds = lgb.Dataset(X, label=Y.ravel()[ids], feature_name=features.names, categorical_feature=['unit'])
        validation = []
        callbacks = [lgb.log_evaluation(0)]
        n_rounds = 1400 if rounds is None else int(rounds[k])
        if stop_cases is not None:
            stop_ids = unit_ids(stop_cases, features, k)
            validation = [lgb.Dataset(features.X(stop_ids), label=Y.ravel()[stop_ids], reference=ds)]
            callbacks.append(lgb.early_stopping(80, verbose=False))
        model = lgb.train(model_params(), ds, num_boost_round=n_rounds, valid_sets=validation, callbacks=callbacks)
        models[k] = model
        timings.append(dict(unit=u, rows=len(ids), rounds=model.best_iteration or n_rounds, seconds=time.perf_counter() - tick))
        print('Specialist:', u, timings[-1], flush=True)
        del X, ds, validation
        gc.collect()
    return (models, pd.DataFrame(timings))

def predict_specialists(models, features, cases):
    result = np.empty((len(cases), features.L), dtype=np.float64)
    for k, model in models.items():
        ids = unit_ids(cases, features, k)
        result[:, features.ku == k] = predict(model, features, ids, 'rich').reshape(len(cases), -1)
    return result.ravel()

uc,sample,mapping,labels,units,ku,ht,series,config=load_inputs(PROJECT)
Y=uc[labels].to_numpy(dtype=np.uint8)
assert len(units)==14 and len(labels)==2352
assert units==['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
    'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1']
ordered_test=sample[['Run No']].merge(mapping,on='Run No',how='left',validate='one_to_one',sort=False)
assert ordered_test.starttime.notna().all()
all_train=np.arange(len(uc));all_test=np.arange(len(sample))
print('Labelled cases:',len(uc),'Prediction cases:',len(sample),'Labels/case:',len(labels))


class DispatchFeatures(RichHydroFeatures):

    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        plants = list(dict.fromkeys((row['plant'] for row in self.topology)))
        self.plant_code = np.array([plants.index(row['plant']) for row in self.topology])
        self.penstock = np.array([config['model']['generator'][u]['penstock'] for u in units])
        self.pipe_group = np.array([100 * p + pipe for p, pipe in zip(self.plant_code, self.penstock)])
        self.pipe_position = np.empty(len(units), dtype=int)
        self.plant_q = np.empty(len(units), dtype=np.float32)
        self.pipe_q = np.empty(len(units), dtype=np.float32)
        self.main_loss = np.empty(len(units), dtype=np.float32)
        self.pipe_loss = np.empty(len(units), dtype=np.float32)
        self.eta_best = np.empty((self.n, len(units)), dtype=np.float32)
        self.eta_low = self.eta_best.copy()
        self.qbest = self.eta_best.copy()
        for k, u in enumerate(units):
            same_plant = self.plant_code == self.plant_code[k]
            same_pipe = self.pipe_group == self.pipe_group[k]
            self.pipe_position[k] = list(np.flatnonzero(same_pipe)).index(k)
            self.plant_q[k] = self.qmax[same_plant].sum()
            self.pipe_q[k] = self.qmax[same_pipe].sum()
            p = config['model']['plant'][self.topology[k]['plant']]
            assert len(p['main_loss']) == 1
            self.main_loss[k] = float(p['main_loss'][0])
            self.pipe_loss[k] = float(p['penstock_loss'][self.penstock[k] - 1])
            curves = sorted(config['model']['generator'][u]['turb_eff_curves'], key=lambda a: a['ref'])
            refs = [a['ref'] for a in curves]
            self.eta_best[:, k] = np.interp(self.nethead[:, k], refs, [max(a['y']) / 100 for a in curves])
            self.eta_low[:, k] = np.interp(self.nethead[:, k], refs, [a['y'][0] / 100 for a in curves])
            self.qbest[:, k] = np.interp(self.nethead[:, k], refs, [a['x'][int(np.argmax(a['y']))] for a in curves])
        available = self.slack + self.infweek
        self.unit_budget = available * 1000000.0 / (self.qmax[None, :] * H * 3600)
        self.plant_budget = available * 1000000.0 / (self.plant_q[None, :] * H * 3600)
        sorted_price = np.sort(self.p, axis=1)

        def threshold(budget):
            pos = (1 - np.clip(budget, 0, 1)) * (H - 1)
            low = np.floor(pos).astype(int)
            high = np.minimum(low + 1, H - 1)
            cases = np.arange(self.n)[:, None]
            return sorted_price[cases, low] + (pos - low) * (sorted_price[cases, high] - sorted_price[cases, low])
        self.unit_threshold = threshold(self.unit_budget)
        self.plant_threshold = threshold(self.plant_budget)
        self.positive_context = {}
        for width in [6, 12, 24]:
            arr = np.empty((self.n, H, len(units)), dtype=np.float32)
            for k in range(len(units)):
                margin = np.maximum(self.p - self.upwv[:, k, None], 0) / self.std[:, None]
                arr[:, :, k] = pd.DataFrame(margin).T.rolling(width, min_periods=1, center=True).mean().T.to_numpy()
            self.positive_context[width] = arr

    def X(self, ids, mode='dispatch'):
        base = super().X(ids, 'rich')
        names = self.names.copy()
        ids = np.asarray(ids)
        c, l = (ids // self.L, ids % self.L)
        u, t = (self.ku[l], self.ht[l])
        extra = []

        def put(name, value):
            names.append(name)
            extra.append(np.asarray(value, dtype=np.float32))
        put('plant_id', self.plant_code[u])
        put('penstock_group', self.pipe_group[u])
        put('position_in_pipe', self.pipe_position[u])
        put('pipe_share_of_plant_capacity', self.pipe_q[u] / self.plant_q[u])
        put('single_full_load_head_loss', self.main_loss[u] * self.qmax[u] ** 2 + self.pipe_loss[u] * self.qmax[u] ** 2)
        put('plant_full_load_head_loss', self.main_loss[u] * self.plant_q[u] ** 2 + self.pipe_loss[u] * self.pipe_q[u] ** 2)
        put('single_loss_fraction_head', (self.main_loss[u] + self.pipe_loss[u]) * self.qmax[u] ** 2 / self.nethead[c, u])
        put('best_turbine_efficiency', self.eta_best[c, u])
        put('best_flow_fraction_max', self.qbest[c, u] / self.qmax[u])
        put('low_load_water_value_proxy_z', (self.p[c, t] - self.upwv[c, u] * self.eta_best[c, u] / np.maximum(self.eta_low[c, u], 0.1)) / self.std[c])
        put('plant_week_water_budget', self.plant_budget[c, u])
        put('price_minus_unit_water_threshold_z', (self.p[c, t] - self.unit_threshold[c, u]) / self.std[c])
        put('price_minus_plant_water_threshold_z', (self.p[c, t] - self.plant_threshold[c, u]) / self.std[c])
        put('price_minus_combined_dispatch_threshold_z', (self.p[c, t] - np.maximum(self.plant_threshold[c, u], self.upwv[c, u])) / self.std[c])
        put('rank_budget_distance', self.static_hour['price_week_rank'][c, t] + np.clip(self.plant_budget[c, u], 0, 1) - 1)
        for width, a in self.positive_context.items():
            put('positive_margin_context_' + str(width), a[c, t, u])
        for j, r in enumerate(self.reservoirs):
            put('water_value_trend_' + r, (self.wv[c, j] - self.wv0[c, j]) / self.pscale[c])
        result = np.column_stack([base, *extra])
        assert np.isfinite(result).all()
        self.names = names
        return result

class NetworkPressureFeatures(DispatchFeatures):

    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        self.tunnel_states = []
        self.tunnel_feed = np.zeros((self.n, len(units)), dtype=np.float32)
        for name, tunnel in config['model'].get('tunnel', {}).items():
            source = [e['from'] for e in config['connections'] if e['to'] == name and e['to_type'] == 'tunnel' and (e['from_type'] == 'reservoir')]
            target = [e['to'] for e in config['connections'] if e['from'] == name and e['from_type'] == 'tunnel' and (e['to_type'] == 'reservoir')]
            if len(source) != 1 or len(target) != 1:
                continue
            a, b = (self.rid[source[0]], self.rid[target[0]])
            alpha = float(tunnel['loss_factor'])
            ha, hb = (self.heads[:, a], self.heads[:, b])
            start = float(tunnel['start_height'])
            end = float(tunnel['end_height'])
            positive = np.maximum(ha - np.maximum(hb, end), 0) * (ha >= start)
            negative = np.maximum(hb - np.maximum(ha, start), 0) * (hb >= end)
            q = (np.sqrt(positive / max(alpha, 1e-09)) - np.sqrt(negative / max(alpha, 1e-09))).astype(np.float32)
            self.tunnel_states.append((name, (ha - hb).astype(np.float32), q))
            self.tunnel_feed += q[:, None] * (self.feedweights[:, b] - self.feedweights[:, a])[None, :]
        self.intake_head = self.uphead.copy()
        for k, row in enumerate(self.topology):
            incoming = [e['from'] for e in config['connections'] if e['to_type'] == 'plant' and e['to'] == row['plant'] and (e['from_type'] == 'tunnel')]
            heads = []
            alphas = []
            for name in incoming:
                reservoirs = [e['from'] for e in config['connections'] if e['to_type'] == 'tunnel' and e['to'] == name and (e['from_type'] == 'reservoir')]
                if len(reservoirs) == 1:
                    heads.append(self.heads[:, self.rid[reservoirs[0]]])
                    alphas.append(float(config['model']['tunnel'][name]['loss_factor']))
            if len(heads) < 2:
                continue
            upper = np.column_stack(heads)
            alpha = np.array(alphas)[None, :]
            qtarget = float(self.plant_q[k])
            lo = upper.min(axis=1) - max(alphas) * qtarget * qtarget * 2 - 1
            hi = upper.max(axis=1)
            for _ in range(25):
                middle = (lo + hi) / 2
                dh = upper - middle[:, None]
                flow = (np.sign(dh) * np.sqrt(np.abs(dh) / alpha)).sum(axis=1)
                lo = np.where(flow > qtarget, middle, lo)
                hi = np.where(flow > qtarget, hi, middle)
            self.intake_head[:, k] = (lo + hi) / 2
        threshold = np.maximum(self.upwv, self.plant_threshold)
        z = np.clip(2 * (self.p[:, :, None] - threshold[:, None, :]) / self.std[:, None, None], -20, 20)
        q = 1 / (1 + np.exp(-z)) * self.qmax[None, None, :]
        for plant in np.unique(self.plant_code):
            ids = np.flatnonzero(self.plant_code == plant)
            required = q[:, :, ids].sum(axis=(1, 2)) * 3600
            available = np.maximum(self.slack[:, ids[0]] + self.infweek[:, ids[0]], 0) * 1000000.0
            factor = np.minimum(1, available / np.maximum(required, 1))
            q[:, :, ids] *= factor[:, None, None]
        self.upstream_release_inflow = np.einsum('ctu,ur->ctr', q, self.downweights).astype(np.float32)
        self.upstream_release_cum = np.cumsum(self.upstream_release_inflow, axis=1) * 3600 / 1000000.0
        self.upstream_release_week = self.upstream_release_inflow.sum(axis=1) * 3600 / 1000000.0

    def X(self, ids, mode='network'):
        base = super().X(ids, 'dispatch')
        names = self.names.copy()
        ids = np.asarray(ids)
        c, l = (ids // self.L, ids % self.L)
        u, t = (self.ku[l], self.ht[l])
        extra = []

        def put(n, v):
            names.append(n)
            extra.append(np.asarray(v, dtype=np.float32))
        for j, r in enumerate(self.reservoirs):
            put('initial_level_' + r, self.heads[c, j])
        for name, dh, q in self.tunnel_states:
            put('tunnel_level_difference_' + name, dh[c])
            put('tunnel_pressure_flow_scenario_' + name, q[c])
        put('feeding_tunnel_flow_relative_qmax', self.tunnel_feed[c, u] / self.qmax[u])
        put('intake_head_fullflow_scenario', self.intake_head[c, u])
        effective = np.maximum(self.intake_head[c, u] - self.downhead[c, u] - self.main_loss[u] * self.plant_q[u] ** 2 - self.pipe_loss[u] * self.pipe_q[u] ** 2, 1)
        put('fullflow_effective_head_scenario', effective)
        put('price_minus_head_scaled_value_proxy_z', (self.p[c, t] - self.upwv[c, u] * self.nethead[c, u] / effective) / self.std[c])
        inflow = np.einsum('ij,ij->i', self.upstream_release_inflow[c, t, :], self.feedweights[u])
        cumulative = np.einsum('ij,ij->i', self.upstream_release_cum[c, t, :], self.feedweights[u])
        natural = np.einsum('ij,ij->i', self.inf_cum[c, t, :], self.feedweights[u])
        whole = np.einsum('ij,ij->i', self.upstream_release_week[c, :], self.feedweights[u])
        volume = self.feedvol[c, u] + cumulative + natural
        put('upstream_release_scenario_relative_qmax', inflow / self.qmax[u])
        put('upstream_release_scenario_cumulative_fraction', cumulative / self.feedcap[u])
        put('upstream_release_scenario_week_fraction', whole / self.feedcap[u])
        put('volume_with_upstream_release_scenario_fraction', volume / self.feedcap[u])
        put('scenario_overflow_pressure', np.maximum(volume - self.feedcap[u], 0) / self.feedcap[u])
        put('scenario_required_release_fraction', np.maximum(volume - self.feedcap[u], 0) * 1000000.0 / np.maximum((t + 1) * 3600 * self.qmax[u], 1))
        result = np.column_stack([base, *extra])
        assert np.isfinite(result).all()
        self.names = names
        return result

def temporal_layout(features):
    features.X(np.arange(10))
    shared = ['t', 'hour', 'weekday', 'month', 'season_sin', 'season_cos', 'price_week_rank', 'price_day_rank', 'price_z', 'price_relative_mean', 'price_delta_prev', 'price_delta_next', 'week_volatility', 'river_natural_inflow', 'river_week_natural_mean']
    prefixes = ['storage_', 'wv_relative_', 'week_natural_inflow_capacity_', 'water_value_trend_', 'initial_level_', 'minflow_', 'tunnel_level_difference_', 'tunnel_pressure_flow_scenario_']
    shared += [n for n in features.names if any((n.startswith(p) for p in prefixes)) and n != 'storage_slack_fraction']
    shared = list(dict.fromkeys(shared))
    local = ['price_minus_upwv_z', 'price_minus_netwv_z', 'price_over_upwv', 'upwv_relative', 'downwv_relative', 'feed_storage_fraction', 'down_storage_fraction', 'week_inflow_capacity', 'cumulative_natural_inflow_fraction', 'hour_inflow_fraction_qmax', 'catchment_storage_fraction', 'plant_week_water_budget', 'price_minus_combined_dispatch_threshold_z', 'initial_net_head', 'feeding_tunnel_flow_relative_qmax', 'volume_with_upstream_release_scenario_fraction', 'intake_head_fullflow_scenario', 'scenario_overflow_pressure', 'day_profitable_fraction', 'positive_margin_context_6', 'positive_margin_context_24']
    assert set(shared + local) <= set(features.names)
    return (shared, local)

def write_temporal_inputs(features, path, layout=None):
    if layout is None:
        layout = temporal_layout(features)
    shared, local = layout
    features.X(np.arange(10))
    all_names = features.names.copy()
    shared_ids = [all_names.index(n) for n in shared]
    local_ids = [all_names.index(n) for n in local]
    U = len(features.units)
    assert np.array_equal(features.ku, np.repeat(np.arange(U), 168))
    assert np.array_equal(features.ht, np.tile(np.arange(168), U))
    output = np.lib.format.open_memmap(path, mode='w+', dtype='float32', shape=(features.n, len(shared) + U * len(local), 168))
    for a in range(0, features.n, 16):
        b = min(a + 16, features.n)
        ids = np.arange(a * features.L, b * features.L)
        x = features.X(ids).reshape(b - a, U, 168, -1)
        common = np.take(x[:, 0, :, :], shared_ids, axis=2)
        local_x = np.take(x, local_ids, axis=3).transpose(0, 1, 3, 2).reshape(b - a, -1, 168)
        output[a:b] = np.concatenate([common.transpose(0, 2, 1), local_x], axis=1)
    output.flush()
    print('Sequence input saved:', output.shape, flush=True)
    del output
    return layout

def economic_state_advantage(revenue, startup, shutdown):
    """Optimal whole-horizon objective with state ON minus state OFF at each hour.

    Initial state is free because no historical commitment is given. No artificial
    shutdown is charged after the horizon. Revenue is the operating-margin proxy
    per hour; separate startup/shutdown costs are charged at actual transitions.
    """
    revenue = np.asarray(revenue, dtype=np.float64)
    startup = np.broadcast_to(startup, revenue.shape[:-1])
    shutdown = np.broadcast_to(shutdown, revenue.shape[:-1])
    forward = np.zeros(revenue.shape + (2,), dtype=np.float64)
    forward[..., 0, 1] = revenue[..., 0]
    for hour in range(1, revenue.shape[-1]):
        previous = forward[..., hour - 1, :]
        forward[..., hour, 0] = np.maximum(previous[..., 0], previous[..., 1] - shutdown)
        forward[..., hour, 1] = revenue[..., hour] + np.maximum(previous[..., 1], previous[..., 0] - startup)
    backward = np.zeros_like(forward)
    for hour in range(revenue.shape[-1] - 2, -1, -1):
        following = backward[..., hour + 1, :]
        next_margin = revenue[..., hour + 1]
        backward[..., hour, 0] = np.maximum(following[..., 0], following[..., 1] + next_margin - startup)
        backward[..., hour, 1] = np.maximum(following[..., 0] - shutdown, following[..., 1] + next_margin)
    forced_objective = forward + backward
    return forced_objective[..., 1] - forced_objective[..., 0]

class EconomicRichFeatures(RichHydroFeatures):

    def __init__(self, starts, series, config, units, ku, ht):
        super().__init__(starts, series, config, units, ku, ht)
        self.stopcost = np.array([config['model']['generator'][unit].get('stopcost_const', 0) for unit in units])
        self.raw_margin = ((self.p[:, None, :] - self.upwv[:, :, None]) * self.pmax[None, :, None]).astype('float32')
        margin = (self.p[:, None, :] - self.upwv[:, :, None]) * self.pmax[None, :, None]
        self.economic_advantage = economic_state_advantage(margin, self.startcost[None, :], self.stopcost[None, :]).astype('float32')

    def X(self, ids, mode='rich'):
        base = super().X(ids, 'rich')
        names = self.names.copy()
        ids = np.asarray(ids)
        case, label = (ids // self.L, ids % self.L)
        unit, hour = (self.ku[label], self.ht[label])
        advantage = self.economic_advantage[case, unit, hour]
        denominator = np.maximum(self.pmax[unit] * self.std[case], 1)
        extra = [self.p[case, hour], self.upwv[case, unit], self.downwv[case, unit], self.std[case], self.startcost[unit], self.stopcost[unit], self.raw_margin[case, unit, hour] / denominator, advantage / denominator, np.sign(advantage) * np.log1p(np.abs(advantage)), advantage / np.maximum(self.startcost[unit] + self.stopcost[unit], 1)]
        names += ['raw_price', 'raw_upstream_local_value', 'raw_downstream_local_value', 'raw_price_std', 'raw_startup_cost', 'raw_shutdown_cost', 'economic_margin_z', 'forced_on_vs_off_value_z', 'signed_log_forced_value', 'forced_value_over_cycle_cost']
        self.names = names
        return np.column_stack([base, *extra]).astype('float32')

"""Auditable optimization-informed features, not a SHOP replacement or PINN.

All 17 reservoir balances and instantaneous junction balances are exact in the
surrogate. Generation uses fixed initial-head best-efficiency coefficients.
Ungated tunnels use one initial-pressure affine approximation; reservoir levels
use HRL/LRL secants. Flood storage is permitted and overflow has a convex lower
envelope, not an exact weir law. Rivers retain destinations and convex costs.
Unit commitment is a continuous relaxation, not a physical on/off schedule.
No historical within-week storage or commitment labels enter this module.
"""
from dataclasses import dataclass
import time
import warnings
import numpy as np
from scipy import sparse
from scipy.optimize import linprog, OptimizeWarning

H = 168
DT = 3600 / 1e6  # (m3/s) -> Mm3 per hourly interval
OBJECTIVE_SCALE = 1000.


@dataclass
class Arc:
    name: str
    source: int
    target: int | None
    capacity: float
    cost: float = 0.


class HydraulicSurrogate:
    """A deterministic, case-specific sparse LP built only from given inputs."""
    def __init__(self, base, config):
        self.base, self.config = base, config
        self.model, self.edges = config['model'], config['connections']
        self.reservoirs = base.reservoirs
        self.rid = base.rid
        self.R, self.U = len(self.reservoirs), len(base.units)
        assert self.R == 17 and self.U == 14
        self.nodes = self.reservoirs + ['river:r_Vest_Vassdraget', 'manifold:Songa']
        self.vest, self.songa = self.R, self.R + 1
        self.N = len(self.nodes)
        self.plants = list(self.model['plant'])
        self.unit_plant = [self._connections('generator', u, 'out')[0][1] for u in base.units]
        self.plant_units = {p: np.flatnonzero(np.array(self.unit_plant) == p) for p in self.plants}
        self.unit_arcs = []
        for u, p in zip(base.units, self.unit_plant):
            ins = self._connections('plant', p, 'in')
            sources = [n for typ, n in ins if typ == 'reservoir']
            source = self.rid[sources[0]] if sources else self.songa
            assert len(sources) == 1 or p == 'Songa'
            target = self._destination(self._connections('plant', p, 'out'))
            self.unit_arcs.append(Arc(u, source, target, 0))
        self.tunnels = []
        for name, obj in self.model['tunnel'].items():
            upstream = self._connections('tunnel', name, 'in')
            assert len(upstream) == 1 and upstream[0][0] == 'reservoir'
            out = self._connections('tunnel', name, 'out')
            target = self.songa if out == [('plant', 'Songa')] else self._destination(out)
            assert target is not None
            self.tunnels.append(Arc(name, self.rid[upstream[0][1]], target, 10000))
        self.rivers, self.river_parent, self.spill_indices = [], [], []
        for name, obj in self.model['river'].items():
            if name == 'r_Vest_Vassdraget':
                source = self.vest
            else:
                ins = self._connections('river', name, 'in')
                assert len(ins) == 1 and ins[0][0] == 'reservoir', (name, ins)
                source = self.rid[ins[0][1]]
            target = self._destination(self._connections('river', name, 'out'))
            curve = obj.get('flow_cost_curve')
            if curve:
                # y is a piecewise-constant marginal cost, not total cost.
                x, y = curve['x'], curve['y']
                cap = float(obj.get('max_flow_const', 10000))
                segments = [(float(x[j+1]-x[j]), float(y[j])) for j in range(len(x)-1)]
                segments.append((cap-float(x[-1]), float(y[-1])))
            else:
                rating = obj.get('up_head_flow_curve')
                # A rating-curve endpoint is not a hard discharge capacity.
                cap = obj.get('max_flow_const', 10000)
                segments = [(float(cap), float(obj.get('flow_cost_const', 0)))]
            for j, (capacity, cost) in enumerate(segments):
                if capacity <= 0:
                    continue
                self.rivers.append(Arc(f'{name}:segment{j}', source, target, capacity, cost))
                self.river_parent.append(name)
                if name.startswith('s_'):
                    self.spill_indices.append(len(self.rivers)-1)
        self.river_parent = np.array(self.river_parent)
        self.arcs = self.unit_arcs + self.tunnels + self.rivers
        self.A = len(self.arcs)
        self.flow = np.arange(self.A * H).reshape(self.A, H)
        offset = self.A * H
        self.storage = np.arange(offset, offset+self.R*H).reshape(self.R, H)
        offset += self.R*H
        self.commitment = np.arange(offset, offset+self.U*H).reshape(self.U, H)
        offset += self.U*H
        self.startup = np.arange(offset, offset+self.U*(H-1)).reshape(self.U, H-1)
        offset += self.U*(H-1)
        self.shutdown = np.arange(offset, offset+self.U*(H-1)).reshape(self.U, H-1)
        offset += self.U*(H-1)
        self.manifold_head = np.arange(offset, offset+H)
        self.nvar = offset + H
        self.low_level = np.array([self.model['reservoir'][r]['lrl'] for r in self.reservoirs])
        self.level_slope = np.array([(self.model['reservoir'][r]['hrl']-self.model['reservoir'][r]['lrl']) /
                                     self.model['reservoir'][r]['max_vol'] for r in self.reservoirs])
        self.first_plant, self.after_first_plant = self._valuation_routes()
        self.balance_template = self._balance_matrix()
        self.names = ['lp_unit_flow_fraction', 'lp_relaxed_commitment',
            'lp_plant_flow_fraction', 'lp_plant_equivalent_units',
            'lp_upstream_storage_fraction', 'lp_downstream_storage_fraction',
            'lp_upstream_minimum_storage_slack', 'lp_approx_gross_head_ratio',
            'lp_water_opportunity_cost_relative', 'lp_price_minus_opportunity_cost_z',
            'lp_production_reduced_value_z', 'lp_relaxed_startup', 'lp_relaxed_shutdown',
            'lp_economic_forced_advantage_z', 'lp_economic_forced_advantage_over_cycle_cost',
            'lp_unit_capacity_head_factor']

    def _connections(self, typ, name, direction):
        if direction == 'out':
            return [(e['to_type'], e['to']) for e in self.edges if (e['from_type'], e['from']) == (typ, name)]
        return [(e['from_type'], e['from']) for e in self.edges if (e['to_type'], e['to']) == (typ, name)]

    def _destination(self, connections):
        if not connections:
            return None
        assert len(connections) == 1, connections
        typ, name = connections[0]
        if typ == 'reservoir':
            return self.rid[name]
        if (typ, name) == ('river', 'r_Vest_Vassdraget'):
            return self.vest
        raise ValueError(('unsupported destination', connections))

    def _valuation_routes(self):
        # Prefer the producing/main-watercourse path; bypass/spill are alternatives.
        def find_first(node, visited):
            if node in visited:
                raise ValueError(('cycle in valuation path', node))
            if node[0] == 'plant':
                return node[1]
            choices = []
            for typ, name in self._connections(*node, 'out'):
                if typ == 'generator':
                    continue
                if typ == 'river' and not (self.model['river'][name].get('main_river') or name == 'r_Vest_Vassdraget'):
                    continue
                found = find_first((typ, name), visited | {node})
                if found is not None:
                    choices.append(found)
            choices = list(set(choices))
            assert len(choices) <= 1, (node, choices)
            return choices[0] if choices else None
        first = [find_first(('reservoir', r), set()) for r in self.reservoirs]
        assert all(p is not None for p in first)
        after = {}
        for p in self.plants:
            dest = self._connections('plant', p, 'out')
            while dest and dest[0][0] == 'river':
                assert len(dest) == 1
                dest = self._connections(*dest[0], 'out')
            assert len(dest) <= 1
            after[p] = self.rid[dest[0][1]] if dest else None
        return first, after

    def _balance_matrix(self):
        rows, cols, values = [], [], []
        hours = np.arange(H)
        def add(r, c, v):
            rows.extend(np.asarray(r).ravel()); cols.extend(np.asarray(c).ravel())
            values.extend(np.broadcast_to(v, np.shape(c)).ravel())
        for r in range(self.R):
            add(r*H+hours, self.storage[r], 1.)
            add(r*H+hours[1:], self.storage[r, :-1], -1.)
        for a, arc in enumerate(self.arcs):
            add(arc.source*H+hours, self.flow[a], DT)
            if arc.target is not None:
                add(arc.target*H+hours, self.flow[a], -DT)
        return sparse.csr_matrix((values, (rows, cols)), shape=(self.N*H, self.nvar))

    def coefficients(self, case):
        b = self.base
        levels = np.array([np.interp(b.vol[case, j], self.model['reservoir'][r]['vol_head']['x'],
                                     self.model['reservoir'][r]['vol_head']['y']) for j,r in enumerate(self.reservoirs)])
        beta, qbest, gross = np.zeros(self.U), np.zeros(self.U), np.zeros(self.U)
        for u, unit in enumerate(b.units):
            g, p = self.model['generator'][unit], self.model['plant'][self.unit_plant[u]]
            ids = self.plant_units[self.unit_plant[u]]
            pipe_count = sum(self.model['generator'][b.units[k]]['penstock'] == g['penstock'] for k in ids)
            up = float(levels @ b.upweights[u])
            down = max(float(levels @ b.downweights[u]), float(p['outlet_line']))
            gross[u] = max(up-down, .1)
            q = np.linspace(min(min(c['x']) for c in g['turb_eff_curves']), b.qmax[u], 120)
            loss = float(p['main_loss'][0])*(len(ids)*q)**2 + float(p['penstock_loss'][g['penstock']-1])*(pipe_count*q)**2
            if self.unit_plant[u] == 'Songa':
                alphas = [float(self.model['tunnel'][a.name]['loss_factor']) for a in self.tunnels if a.target == self.songa]
                loss += q*q / sum(1/np.sqrt(a) for a in alphas)**2
            head = np.maximum(gross[u]-loss, .1)
            curves = sorted(g['turb_eff_curves'], key=lambda x:x['ref'])
            eff_at_q = np.array([np.interp(q, c['x'], c['y']) for c in curves])
            refs = np.array([c['ref'] for c in curves])
            eta = np.array([np.interp(h, refs, eff_at_q[:, j]) for j,h in enumerate(head)])/100
            power = .00981 * head * q * eta
            for _ in range(2):
                power = .00981 * head * q * eta * np.interp(power, g['gen_eff_curve']['x'], g['gen_eff_curve']['y'])/100
            power = np.minimum(power, float(g['p_max']))
            best = np.argmax(power/q)
            beta[u], qbest[u] = power[best]/q[best], q[best]
        qmax = np.minimum(b.qmax, b.pmax/np.maximum(beta, 1e-6))
        local_energy = {p: float(np.mean(beta[ids]))/DT for p,ids in self.plant_units.items()}
        global_value = np.full(self.R, np.nan)
        def value(r, visited):
            if np.isfinite(global_value[r]): return global_value[r]
            assert r not in visited
            p = self.first_plant[r]
            below = self.after_first_plant[p]
            global_value[r] = float(b.wv[case, r])*local_energy[p] + (value(below, visited | {r}) if below is not None else 0.)
            return global_value[r]
        for r in range(self.R): value(r, set())
        return beta, qbest, qmax, gross, global_value

    def solve(self, case):
        tick = time.perf_counter()
        b = self.base
        beta, qbest, qmax, gross, global_value = self.coefficients(case)
        lower, upper, cost = np.zeros(self.nvar), np.full(self.nvar, np.inf), np.zeros(self.nvar)
        upper[self.flow[:self.U]] = qmax[:, None]
        cost[self.flow[:self.U]] = -beta[:, None]*b.p[case][None, :]
        upper[self.flow[self.U:self.U+len(self.tunnels)]] = 10000.
        lower[self.flow[self.U:self.U+len(self.tunnels)]] = -10000.
        for a, arc in enumerate(self.rivers, self.U+len(self.tunnels)):
            upper[self.flow[a]] = arc.capacity
            cost[self.flow[a]] = arc.cost
        # The given first-interval minimum is applied to end-of-interval storage.
        lower[self.storage] = b.mv_all[case].T
        flood_capacity = np.array([max(self.model['reservoir'][r]['vol_head']['x']) for r in self.reservoirs])
        upper[self.storage] = np.maximum(flood_capacity, b.vol[case])[:, None]
        cost[self.storage[:, -1]] = -global_value
        upper[self.commitment] = 1.
        upper[self.startup] = 1.; upper[self.shutdown] = 1.
        cost[self.startup] = b.startcost[:, None]
        stopcost = np.array([self.model['generator'][u].get('stopcost_const', 0) for u in b.units])
        cost[self.shutdown] = stopcost[:, None]
        lower[self.manifold_head] = min(float(self.model['tunnel'][a.name]['start_height']) for a in self.tunnels if a.target == self.songa)-100
        upper[self.manifold_head] = max(self.model['reservoir'][self.reservoirs[a.source]]['hrl'] for a in self.tunnels if a.target == self.songa)+10
        bounds = np.column_stack([lower, upper])
        rhs = np.zeros((self.N, H))
        rhs[:self.R] = b.inf[case].T*DT
        rhs[:self.R, 0] += b.vol[case]
        rhs[self.vest] = b.series_river_inflow[case]*DT

        # Honor each minimum river flow on its aggregate segments.
        ir, ic, iv, ib = [], [], [], []
        def inequality(columns, values, bound=0.):
            row = len(ib)
            ir.extend([row]*len(columns));ic.extend(columns);iv.extend(values);ib.append(bound)
        for j in self.spill_indices:
            arc = self.rivers[j]
            name = self.river_parent[j]
            obj = self.model['river'][name]
            curve = obj['up_head_flow_curve'][0]
            # All supplied ratings have two endpoints and are linear in level.
            assert len(curve['x']) == 2
            slope = (curve['y'][1]-curve['y'][0])/(curve['x'][1]-curve['x'][0])
            slope *= self.level_slope[arc.source]
            for h in range(H):
                inequality([self.storage[arc.source,h],self.flow[self.U+len(self.tunnels)+j,h]],
                           [slope,-1.],slope*float(b.caps[arc.source]))
        for k, name in enumerate(b.flow_names):
            ids = np.flatnonzero(self.river_parent == name)+self.U+len(self.tunnels)
            assert len(ids)
            for h in range(H):
                if b.flow[case, h, k] > 0:
                    inequality(self.flow[ids, h].tolist(), [-1.]*len(ids), -float(b.flow[case,h,k]))
        for u in range(self.U):
            qmin = float(b.pmin[u]/beta[u])
            for h in range(H):
                inequality([self.flow[u,h],self.commitment[u,h]], [1.,-qmax[u]])
                inequality([self.flow[u,h],self.commitment[u,h]], [-1.,qmin])
                if h:
                    inequality([self.commitment[u,h],self.commitment[u,h-1],self.startup[u,h-1]], [1.,-1.,-1.])
                    inequality([self.commitment[u,h-1],self.commitment[u,h],self.shutdown[u,h-1]], [1.,-1.,-1.])
        a_ub = sparse.csr_matrix((iv,(ir,ic)),shape=(len(ib),self.nvar))

        # Linearized q|q| pressure loss around the initial pressure/operating point.
        er, ec, ev, eb = [], [], [], []
        levels0 = self.low_level + self.level_slope*b.vol[case]
        tunnel_audit = []
        for t, arc in enumerate(self.tunnels):
            obj = self.model['tunnel'][arc.name]
            alpha = float(obj['loss_factor'])
            ha = levels0[arc.source]
            target_res = arc.target < self.R
            hb = levels0[arc.target] if target_res else ha - alpha*(float(qbest[self.plant_units['Songa']].sum())/2)**2
            enda, endb = float(obj['start_height']), float(obj['end_height'])
            derivative_a, derivative_b = 1., 1.
            if target_res and hb < endb:
                hb = endb; derivative_b = 0.
                lower[self.flow[self.U+t]] = 0.
            if ha < enda:
                ha = enda; derivative_a = 0.
                upper[self.flow[self.U+t]] = 0.
            dh = ha-hb
            q0 = np.sign(dh)*np.sqrt(abs(dh)/alpha)
            conductance = 1/(2*np.sqrt(alpha*max(abs(dh), .25)))
            intercept = q0-conductance*dh
            for h in range(H):
                row = len(eb)
                cols, vals = [self.flow[self.U+t,h]], [1.]
                # Mid-interval storage provides an implicit, coupled approximation.
                constant = self.low_level[arc.source] if derivative_a else enda
                if derivative_a:
                    cols.append(self.storage[arc.source,h]);vals.append(-conductance*.5*self.level_slope[arc.source])
                    if h:
                        cols.append(self.storage[arc.source,h-1]);vals.append(-conductance*.5*self.level_slope[arc.source])
                    else: constant += .5*self.level_slope[arc.source]*b.vol[case,arc.source]
                if target_res:
                    constant -= self.low_level[arc.target] if derivative_b else endb
                    if derivative_b:
                        cols.append(self.storage[arc.target,h]);vals.append(conductance*.5*self.level_slope[arc.target])
                        if h:
                            cols.append(self.storage[arc.target,h-1]);vals.append(conductance*.5*self.level_slope[arc.target])
                        else: constant -= .5*self.level_slope[arc.target]*b.vol[case,arc.target]
                else:
                    cols.append(self.manifold_head[h]);vals.append(conductance)
                er.extend([row]*len(cols));ec.extend(cols);ev.extend(vals)
                eb.append(intercept+conductance*constant)
            tunnel_audit.append((q0, conductance))
        hydraulics = sparse.csr_matrix((ev,(er,ec)),shape=(len(eb),self.nvar))
        a_eq = sparse.vstack([self.balance_template, hydraulics], format='csr')
        b_eq = np.r_[rhs.ravel(), eb]
        bounds = np.column_stack([lower, upper])
        # SciPy forwards these explicit HiGHS thread controls with a warning.
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='Unrecognized options detected.*', category=OptimizeWarning)
            result = linprog(cost/OBJECTIVE_SCALE, A_ub=a_ub, b_ub=ib,
                A_eq=a_eq, b_eq=b_eq, bounds=bounds, method='highs',
                options={'dual_feasibility_tolerance':1e-7,'primal_feasibility_tolerance':1e-7,
                         'threads':1, 'parallel':False})
        info = dict(case=int(case), success=bool(result.success), status=int(result.status),
            message=result.message, seconds=time.perf_counter()-tick)
        if not result.success:
            return None, info
        x = result.x
        balance_residual = self.balance_template@x-rhs.ravel()
        eq_residual = a_eq@x-b_eq
        bound_violation = max(float(np.maximum(lower-x,0).max()),float(np.maximum(x-upper,0).max()))
        inequality_violation = float(np.maximum(a_ub@x-np.asarray(ib),0).max())
        info.update(balance_max_abs_Mm3=float(np.abs(balance_residual).max()),
            affine_hydraulic_max_abs_m3s=float(np.abs(eq_residual[self.N*H:]).max()),
            bound_max_violation=bound_violation, inequality_max_violation=inequality_violation,
            objective_surrogate_currency=float(-result.fun*OBJECTIVE_SCALE),
            total_terminal_storage_Mm3=float(x[self.storage[:,-1]].sum()),
            has_10000m3s_cap_active=bool(np.any(x[self.flow] >= 9999.99)))
        info['initial_over_normal_capacity_reservoirs'] = int(np.count_nonzero(b.vol[case]>b.caps+1e-5))
        info['spill_below_normal_capacity_max_m3s'] = max([0.]+[
            float(np.max(np.where(x[self.storage[self.rivers[j].source]]<b.caps[self.rivers[j].source]-1e-5,
                x[self.flow[self.U+len(self.tunnels)+j]],0))) for j in self.spill_indices])
        mid_storage = .5*(x[self.storage]+np.column_stack([b.vol[case],x[self.storage[:,:-1]]]))
        mid_levels = self.low_level[:,None]+self.level_slope[:,None]*mid_storage
        nonlinear_errors = []
        for t,arc in enumerate(self.tunnels):
            qa = x[self.flow[self.U+t]]
            obj = self.model['tunnel'][arc.name]
            ha = mid_levels[arc.source]
            hb = mid_levels[arc.target] if arc.target < self.R else x[self.manifold_head]
            actual_pressure = np.where(qa>=0, ha-np.maximum(hb,float(obj['end_height'])),
                                       np.maximum(ha,float(obj['start_height']))-hb)
            nonlinear_errors.extend((actual_pressure-float(obj['loss_factor'])*qa*np.abs(qa)).tolist())
        info['nonlinear_pressure_law_mean_abs_error_m'] = float(np.mean(np.abs(nonlinear_errors)))
        info['nonlinear_pressure_law_max_abs_error_m'] = float(np.max(np.abs(nonlinear_errors)))
        assert max(info['balance_max_abs_Mm3'], bound_violation, inequality_violation) < 2e-5, info
        q, storage, y = x[self.flow[:self.U]], x[self.storage], x[self.commitment]
        value = -result.eqlin.marginals[:self.N*H].reshape(self.N,H)*OBJECTIVE_SCALE
        source = np.array([a.source for a in self.unit_arcs])
        target = np.array([a.target if a.target is not None else self.N for a in self.unit_arcs])
        value_with_sink = np.vstack([value, np.zeros((1,H))])
        opportunity = DT*(value_with_sink[source]-value_with_sink[target])/beta[:,None]
        reduced = -(result.lower.marginals[self.flow[:self.U]]+result.upper.marginals[self.flow[:self.U]])*OBJECTIVE_SCALE/beta[:,None]
        revenue = (b.p[case][None,:]-opportunity)*beta[:,None]*qmax[:,None]
        advantage = economic_state_advantage(revenue, b.startcost, stopcost)
        upstream_stock = b.upweights @ (storage/b.caps[:,None])
        downstream_stock = b.downweights @ (storage/b.caps[:,None])
        storage_slack = b.upweights @ ((storage-b.mv_all[case].T)/b.caps[:,None])
        levels = self.low_level[:,None]+self.level_slope[:,None]*storage
        up_level, down_level = b.upweights@levels, b.downweights@levels
        outlet = np.array([self.model['plant'][p]['outlet_line'] for p in self.unit_plant])
        gross_ratio = (up_level-np.maximum(down_level,outlet[:,None]))/gross[:,None]
        plant_fraction, plant_units = np.zeros_like(q), np.zeros_like(q)
        for p, ids in self.plant_units.items():
            plant_fraction[ids] = q[ids].sum(axis=0)/qmax[ids].sum()
            plant_units[ids] = y[ids].sum(axis=0)
        startup, shutdown = np.zeros_like(q), np.zeros_like(q)
        startup[:,1:] = x[self.startup];shutdown[:,1:] = x[self.shutdown]
        tensors = [q/qmax[:,None], y, plant_fraction, plant_units, upstream_stock,
            downstream_stock, storage_slack, gross_ratio, opportunity/b.pscale[case],
            (b.p[case][None,:]-opportunity)/b.std[case], reduced/b.std[case], startup, shutdown,
            advantage/(b.pmax[:,None]*b.std[case]), advantage/np.maximum(b.startcost+stopcost,1)[:,None],
            np.broadcast_to((beta*qmax/b.pmax)[:,None],q.shape)]
        tensor = np.stack(tensors, axis=-1).astype(np.float32)
        assert tensor.shape == (self.U,H,len(self.names)) and np.isfinite(tensor).all()
        return tensor, info


def attach_given_river_inputs(base, series):
    """Use only explicitly supplied exogenous inflow inside each input window."""
    import pandas as pd
    index = pd.to_datetime((base.starts.tz_localize(None).to_numpy()[:,None]+
        np.arange(H)[None,:]*np.timedelta64(1,'h')).ravel(), utc=True)
    base.series_river_inflow = series['inflow'].reindex(index)['r_Vest_Vassdraget'].to_numpy().reshape(base.n,H)
    base.flow_names = list(series['flow'].columns)
    assert np.isfinite(base.series_river_inflow).all()


class PhysicsFeatures(EconomicRichFeatures):

    def __init__(self, starts, series, config, units, ku, ht, lp_path):
        super().__init__(starts, series, config, units, ku, ht)
        self.lp = np.load(lp_path, mmap_mode='r')
        assert self.lp.shape == (len(starts), len(units), 168, 16)
        self.lp_names = HydraulicSurrogate(self, config).names

    def X(self, ids, mode='rich'):
        raw = super().X(ids)
        names = self.names.copy()
        ids = np.asarray(ids)
        c, l = (ids // self.L, ids % self.L)
        values = self.lp[c, self.ku[l], self.ht[l]]
        available = np.isfinite(values).all(axis=1).astype('float32')
        self.names = names + self.lp_names + ['lp_solver_available']
        return np.column_stack([raw, values, available]).astype('float32', copy=False)

class PQHydraulicSurrogate(HydraulicSurrogate):
    def __init__(self,base,config):
        super().__init__(base,config)
        offset=self.nvar
        self.power=np.arange(offset,offset+self.U*H).reshape(self.U,H)
        offset+=self.U*H
        self.point_weights=np.arange(offset,offset+self.U*5*H).reshape(self.U,5,H)
        previous=self.nvar
        self.nvar=offset+self.U*5*H
        self.balance_template=sparse.hstack([self.balance_template,
            sparse.csr_matrix((self.N*H,self.nvar-previous))],format='csr')
    
    
    def coefficients(self,case):
        beta,qbest,qmax,gross,global_value=super().coefficients(case)
        self.current_q_points,self.current_p_points=self.operating_points(case)
        return beta,qbest,self.current_q_points.max(axis=1),gross,global_value
    
    
    def operating_points(self, case):
        b = self.base
        levels = np.array([np.interp(b.vol[case, j], self.model['reservoir'][r]['vol_head']['x'], self.model['reservoir'][r]['vol_head']['y']) for j, r in enumerate(self.reservoirs)])
        qp, pp = ([], [])
        for u, unit in enumerate(b.units):
            g = self.model['generator'][unit]
            plant = self.model['plant'][self.unit_plant[u]]
            ids = self.plant_units[self.unit_plant[u]]
            pipes = sum((self.model['generator'][b.units[k]]['penstock'] == g['penstock'] for k in ids))
            head0 = max(float(levels @ b.upweights[u]) - max(float(levels @ b.downweights[u]), float(plant['outlet_line'])), 0.1)
            q = np.linspace(min((min(z['x']) for z in g['turb_eff_curves'])), float(b.qmax[u]), 120)
            loss = float(plant['main_loss'][0]) * (len(ids) * q) ** 2 + float(plant['penstock_loss'][g['penstock'] - 1]) * (pipes * q) ** 2
            if self.unit_plant[u] == 'Songa':
                alphas = [float(self.model['tunnel'][z.name]['loss_factor']) for z in self.tunnels if z.target == self.songa]
                loss += q * q / sum((1 / np.sqrt(a) for a in alphas)) ** 2
            head = np.maximum(head0 - loss, 0.1)
            curves = sorted(g['turb_eff_curves'], key=lambda z: z['ref'])
            refs = np.array([z['ref'] for z in curves])
            eta_at_q = np.array([np.interp(q, z['x'], z['y']) for z in curves])
            eta = np.array([np.interp(h, refs, eta_at_q[:, j]) for j, h in enumerate(head)]) / 100
            power = 0.00981 * head * q * eta
            for _ in range(2):
                power = 0.00981 * head * q * eta * np.interp(power, g['gen_eff_curve']['x'], g['gen_eff_curve']['y']) / 100
            usable = np.flatnonzero((power >= float(g['p_min']) - 1e-07) & (power <= float(g['p_max']) + 1e-07))
            if not len(usable):
                raise ValueError(('No admissible fixed-head operating point', case, unit))
            best = int(np.argmax(power[usable] / q[usable]))
            last = len(usable) - 1
            selected = usable[[0, best // 2, best, (best + last) // 2, last]]
            qp.append(q[selected])
            pp.append(power[selected])
        return (np.array(qp), np.array(pp))
    
    def solve(self, case):
        tick = time.perf_counter()
        b = self.base
        beta, qbest, qmax, gross, global_value = self.coefficients(case)
        lower, upper, cost = (np.zeros(self.nvar), np.full(self.nvar, np.inf), np.zeros(self.nvar))
        upper[self.flow[:self.U]] = qmax[:, None]
        cost[self.flow[:self.U]] = 0.0
        cost[self.power] = -b.p[case][None, :]
        upper[self.power] = b.pmax[:, None]
        upper[self.point_weights] = 1.0
        upper[self.flow[self.U:self.U + len(self.tunnels)]] = 10000.0
        lower[self.flow[self.U:self.U + len(self.tunnels)]] = -10000.0
        for a, arc in enumerate(self.rivers, self.U + len(self.tunnels)):
            upper[self.flow[a]] = arc.capacity
            cost[self.flow[a]] = arc.cost
        lower[self.storage] = b.mv_all[case].T
        flood_capacity = np.array([max(self.model['reservoir'][r]['vol_head']['x']) for r in self.reservoirs])
        upper[self.storage] = np.maximum(flood_capacity, b.vol[case])[:, None]
        cost[self.storage[:, -1]] = -global_value
        upper[self.commitment] = 1.0
        upper[self.startup] = 1.0
        upper[self.shutdown] = 1.0
        cost[self.startup] = b.startcost[:, None]
        stopcost = np.array([self.model['generator'][u].get('stopcost_const', 0) for u in b.units])
        cost[self.shutdown] = stopcost[:, None]
        lower[self.manifold_head] = min((float(self.model['tunnel'][a.name]['start_height']) for a in self.tunnels if a.target == self.songa)) - 100
        upper[self.manifold_head] = max((self.model['reservoir'][self.reservoirs[a.source]]['hrl'] for a in self.tunnels if a.target == self.songa)) + 10
        bounds = np.column_stack([lower, upper])
        rhs = np.zeros((self.N, H))
        rhs[:self.R] = b.inf[case].T * DT
        rhs[:self.R, 0] += b.vol[case]
        rhs[self.vest] = b.series_river_inflow[case] * DT
        ir, ic, iv, ib = ([], [], [], [])
    
        def inequality(columns, values, bound=0.0):
            row = len(ib)
            ir.extend([row] * len(columns))
            ic.extend(columns)
            iv.extend(values)
            ib.append(bound)
        for j in self.spill_indices:
            arc = self.rivers[j]
            name = self.river_parent[j]
            obj = self.model['river'][name]
            curve = obj['up_head_flow_curve'][0]
            assert len(curve['x']) == 2
            slope = (curve['y'][1] - curve['y'][0]) / (curve['x'][1] - curve['x'][0])
            slope *= self.level_slope[arc.source]
            for h in range(H):
                inequality([self.storage[arc.source, h], self.flow[self.U + len(self.tunnels) + j, h]], [slope, -1.0], slope * float(b.caps[arc.source]))
        for k, name in enumerate(b.flow_names):
            ids = np.flatnonzero(self.river_parent == name) + self.U + len(self.tunnels)
            assert len(ids)
            for h in range(H):
                if b.flow[case, h, k] > 0:
                    inequality(self.flow[ids, h].tolist(), [-1.0] * len(ids), -float(b.flow[case, h, k]))
        for u in range(self.U):
            qmin = float(b.pmin[u] / beta[u])
            for h in range(H):
                inequality([self.flow[u, h], self.commitment[u, h]], [1.0, -qmax[u]])
                inequality([self.flow[u, h], self.commitment[u, h]], [-1.0, qmin])
                if h:
                    inequality([self.commitment[u, h], self.commitment[u, h - 1], self.startup[u, h - 1]], [1.0, -1.0, -1.0])
                    inequality([self.commitment[u, h - 1], self.commitment[u, h], self.shutdown[u, h - 1]], [1.0, -1.0, -1.0])
        a_ub = sparse.csr_matrix((iv, (ir, ic)), shape=(len(ib), self.nvar))
        er, ec, ev, eb = ([], [], [], [])
        levels0 = self.low_level + self.level_slope * b.vol[case]
        tunnel_audit = []
        for t, arc in enumerate(self.tunnels):
            obj = self.model['tunnel'][arc.name]
            alpha = float(obj['loss_factor'])
            ha = levels0[arc.source]
            target_res = arc.target < self.R
            hb = levels0[arc.target] if target_res else ha - alpha * (float(qbest[self.plant_units['Songa']].sum()) / 2) ** 2
            enda, endb = (float(obj['start_height']), float(obj['end_height']))
            derivative_a, derivative_b = (1.0, 1.0)
            if target_res and hb < endb:
                hb = endb
                derivative_b = 0.0
                lower[self.flow[self.U + t]] = 0.0
            if ha < enda:
                ha = enda
                derivative_a = 0.0
                upper[self.flow[self.U + t]] = 0.0
            dh = ha - hb
            q0 = np.sign(dh) * np.sqrt(abs(dh) / alpha)
            conductance = 1 / (2 * np.sqrt(alpha * max(abs(dh), 0.25)))
            intercept = q0 - conductance * dh
            for h in range(H):
                row = len(eb)
                cols, vals = ([self.flow[self.U + t, h]], [1.0])
                constant = self.low_level[arc.source] if derivative_a else enda
                if derivative_a:
                    cols.append(self.storage[arc.source, h])
                    vals.append(-conductance * 0.5 * self.level_slope[arc.source])
                    if h:
                        cols.append(self.storage[arc.source, h - 1])
                        vals.append(-conductance * 0.5 * self.level_slope[arc.source])
                    else:
                        constant += 0.5 * self.level_slope[arc.source] * b.vol[case, arc.source]
                if target_res:
                    constant -= self.low_level[arc.target] if derivative_b else endb
                    if derivative_b:
                        cols.append(self.storage[arc.target, h])
                        vals.append(conductance * 0.5 * self.level_slope[arc.target])
                        if h:
                            cols.append(self.storage[arc.target, h - 1])
                            vals.append(conductance * 0.5 * self.level_slope[arc.target])
                        else:
                            constant -= 0.5 * self.level_slope[arc.target] * b.vol[case, arc.target]
                else:
                    cols.append(self.manifold_head[h])
                    vals.append(conductance)
                er.extend([row] * len(cols))
                ec.extend(cols)
                ev.extend(vals)
                eb.append(intercept + conductance * constant)
            tunnel_audit.append((q0, conductance))
        hydraulics = sparse.csr_matrix((ev, (er, ec)), shape=(len(eb), self.nvar))
        pr, pc, pv, pb = ([], [], [], [])
        for u in range(self.U):
            for h in range(H):
                for variable, point_values in [(self.flow[u, h], self.current_q_points[u]), (self.power[u, h], self.current_p_points[u]), (self.commitment[u, h], np.ones(5))]:
                    row = len(pb)
                    cols = [int(variable)] + self.point_weights[u, :, h].tolist()
                    values = [1.0] + (-np.asarray(point_values)).tolist()
                    pr.extend([row] * len(cols))
                    pc.extend(cols)
                    pv.extend(values)
                    pb.append(0.0)
        pq = sparse.csr_matrix((pv, (pr, pc)), shape=(len(pb), self.nvar))
        a_eq = sparse.vstack([self.balance_template, hydraulics, pq], format='csr')
        b_eq = np.r_[rhs.ravel(), eb, pb]
        bounds = np.column_stack([lower, upper])
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', message='Unrecognized options detected.*', category=OptimizeWarning)
            result = linprog(cost / OBJECTIVE_SCALE, A_ub=a_ub, b_ub=ib, A_eq=a_eq, b_eq=b_eq, bounds=bounds, method='highs', options={'dual_feasibility_tolerance': 1e-07, 'primal_feasibility_tolerance': 1e-07, 'threads': 1, 'parallel': False})
        info = dict(case=int(case), success=bool(result.success), status=int(result.status), message=result.message, seconds=time.perf_counter() - tick)
        if not result.success:
            return (None, info)
        x = result.x
        balance_residual = self.balance_template @ x - rhs.ravel()
        eq_residual = a_eq @ x - b_eq
        bound_violation = max(float(np.maximum(lower - x, 0).max()), float(np.maximum(x - upper, 0).max()))
        inequality_violation = float(np.maximum(a_ub @ x - np.asarray(ib), 0).max())
        info.update(balance_max_abs_Mm3=float(np.abs(balance_residual).max()), affine_hydraulic_max_abs_m3s=float(np.abs(eq_residual[self.N * H:self.N * H + len(eb)]).max()), bound_max_violation=bound_violation, inequality_max_violation=inequality_violation, objective_surrogate_currency=float(-result.fun * OBJECTIVE_SCALE), total_terminal_storage_Mm3=float(x[self.storage[:, -1]].sum()), has_10000m3s_cap_active=bool(np.any(x[self.flow] >= 9999.99)))
        info['initial_over_normal_capacity_reservoirs'] = int(np.count_nonzero(b.vol[case] > b.caps + 1e-05))
        info['spill_below_normal_capacity_max_m3s'] = max([0.0] + [float(np.max(np.where(x[self.storage[self.rivers[j].source]] < b.caps[self.rivers[j].source] - 1e-05, x[self.flow[self.U + len(self.tunnels) + j]], 0))) for j in self.spill_indices])
        mid_storage = 0.5 * (x[self.storage] + np.column_stack([b.vol[case], x[self.storage[:, :-1]]]))
        mid_levels = self.low_level[:, None] + self.level_slope[:, None] * mid_storage
        nonlinear_errors = []
        for t, arc in enumerate(self.tunnels):
            qa = x[self.flow[self.U + t]]
            obj = self.model['tunnel'][arc.name]
            ha = mid_levels[arc.source]
            hb = mid_levels[arc.target] if arc.target < self.R else x[self.manifold_head]
            actual_pressure = np.where(qa >= 0, ha - np.maximum(hb, float(obj['end_height'])), np.maximum(ha, float(obj['start_height'])) - hb)
            nonlinear_errors.extend((actual_pressure - float(obj['loss_factor']) * qa * np.abs(qa)).tolist())
        info['nonlinear_pressure_law_mean_abs_error_m'] = float(np.mean(np.abs(nonlinear_errors)))
        info['nonlinear_pressure_law_max_abs_error_m'] = float(np.max(np.abs(nonlinear_errors)))
        assert max(info['balance_max_abs_Mm3'], bound_violation, inequality_violation) < 2e-05, info
        info['pq_combination_max_residual'] = float(np.abs(eq_residual[-len(pb):]).max())
        info['actual_power_min_MW'] = float(x[self.power].min())
        q, storage, y = (x[self.flow[:self.U]], x[self.storage], x[self.commitment])
        value = -result.eqlin.marginals[:self.N * H].reshape(self.N, H) * OBJECTIVE_SCALE
        source = np.array([a.source for a in self.unit_arcs])
        target = np.array([a.target if a.target is not None else self.N for a in self.unit_arcs])
        value_with_sink = np.vstack([value, np.zeros((1, H))])
        opportunity = DT * (value_with_sink[source] - value_with_sink[target]) / beta[:, None]
        reduced = -(result.lower.marginals[self.power] + result.upper.marginals[self.power]) * OBJECTIVE_SCALE
        revenue = (b.p[case][None, :] - opportunity) * beta[:, None] * qmax[:, None]
        advantage = economic_state_advantage(revenue, b.startcost, stopcost)
        upstream_stock = b.upweights @ (storage / b.caps[:, None])
        downstream_stock = b.downweights @ (storage / b.caps[:, None])
        storage_slack = b.upweights @ ((storage - b.mv_all[case].T) / b.caps[:, None])
        levels = self.low_level[:, None] + self.level_slope[:, None] * storage
        up_level, down_level = (b.upweights @ levels, b.downweights @ levels)
        outlet = np.array([self.model['plant'][p]['outlet_line'] for p in self.unit_plant])
        gross_ratio = (up_level - np.maximum(down_level, outlet[:, None])) / gross[:, None]
        plant_fraction, plant_units = (np.zeros_like(q), np.zeros_like(q))
        for p, ids in self.plant_units.items():
            plant_fraction[ids] = q[ids].sum(axis=0) / qmax[ids].sum()
            plant_units[ids] = y[ids].sum(axis=0)
        startup, shutdown = (np.zeros_like(q), np.zeros_like(q))
        startup[:, 1:] = x[self.startup]
        shutdown[:, 1:] = x[self.shutdown]
        tensors = [q / qmax[:, None], y, plant_fraction, plant_units, upstream_stock, downstream_stock, storage_slack, gross_ratio, opportunity / b.pscale[case], (b.p[case][None, :] - opportunity) / b.std[case], reduced / b.std[case], startup, shutdown, advantage / (b.pmax[:, None] * b.std[case]), advantage / np.maximum(b.startcost + stopcost, 1)[:, None], np.broadcast_to((self.current_p_points.max(axis=1) / b.pmax)[:, None], q.shape)]
        tensor = np.stack(tensors, axis=-1).astype(np.float32)
        assert tensor.shape == (self.U, H, len(self.names)) and np.isfinite(tensor).all()
        return (tensor, info)


from concurrent.futures import ThreadPoolExecutor

def generate_tensor(starts,surrogate_class,tag):
    # All measured storage is sampled at case start. Future states below are
    # optimization outputs. The source object has no commitment target array.
    base=CaseFeatures(starts,series,config,units,ku,ht)
    attach_given_river_inputs(base,series)
    path=WORK_DIR/f'{tag}_features.npy'
    result=np.lib.format.open_memmap(path,mode='w+',dtype='float32',
        shape=(len(starts),len(units),H,16))
    result[:]=np.nan
    def compute(case):
        # A separate instance per case avoids shared mutable PQ state.
        simulator=surrogate_class(base,config)
        values,record=simulator.solve(int(case))
        return int(case),values,record
    records=[];tick=time.perf_counter()
    # Each LP explicitly uses one HiGHS thread; at most two LPs run together.
    with ThreadPoolExecutor(max_workers=max(2, min(8, os.cpu_count() or 2))) as pool:
        for case,values,record in pool.map(compute,range(len(starts))):
            if values is not None:result[case]=values
            records.append(record)
            if len(records)%200==0:
                result.flush();print(tag,len(records),'/',len(starts),round(time.perf_counter()-tick,1),'s',flush=True)
    result.flush()
    audit=pd.DataFrame(records);audit.to_csv(SAVE_DIR/f'{tag}_solver_audit.csv',index=False)
    print(tag,'feasible',int(audit.success.sum()),'/',len(audit),flush=True)
    del result,base;gc.collect()
    return path
pq_train_path=generate_tensor(uc.starttime,PQHydraulicSurrogate,'pq_train')
pq_test_path=generate_tensor(ordered_test.starttime,PQHydraulicSurrogate,'pq_test')

In [ ]:
run_stage(["legacy/confidence/pq_train_features.npy", "legacy/confidence/pq_test_features.npy"], ["legacy_pq.py"])

## 4. Pipeline B (v1) (weight 0.35)
In Pipeline B (v1), a 24-channel LP is solved for every case using P–Q curves, tunnel rules and the spill-crest volume given in the data. Economic on/off values are calculated with a dynamic program. A pooled XGBoost and per-unit LightGBM models are then trained, and their predictions are averaged with equal weight.
All settings are fixed values from earlier development. Trained on the same 2,916 strict cases as the other parts.

In [ ]:
%%writefile work_nb2/spill_onset.py
"""Independent raw-data pipeline. No imports from the previous experiments."""
import os
_T = str(max(1, os.cpu_count() or 1))
for k in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS"]: os.environ.setdefault(k, _T)
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
import yaml
from scipy import sparse
from scipy.optimize import linprog, brentq, OptimizeWarning

H = 168
DT = 3600 / 1e6


def read_table(path):
    d = pd.read_csv(path)
    d['date'] = pd.to_datetime(d['date'], utc=True)
    assert not d['date'].duplicated().any()
    return d.set_index('date').sort_index()


class RawData:
    def __init__(self, project):
        self.project = Path(project).resolve()
        kernel = self.project/'data/kernel'
        extended = self.project/'data/extended'
        self.config = yaml.safe_load((extended/'Tokke_Vinje_topology.yaml').read_text())
        self.model = self.config['model']
        self.edges = self.config['connections']
        self.uc = pd.read_csv(kernel/'Unit_commitment_decisions.csv')
        self.uc['starttime'] = pd.to_datetime(self.uc.starttime, utc=True)
        self.mapping = pd.read_csv(self.project/'prediction_mapping.csv')
        self.mapping['starttime'] = pd.to_datetime(self.mapping.starttime, utc=True)
        self.sample = pd.read_csv(self.project/'sample_submission.csv')
        self.mapping = self.sample[['Run No']].merge(self.mapping, on='Run No', validate='one_to_one', sort=False)
        self.labels = list(self.sample.columns[1:])
        self.units = list(dict.fromkeys(c.removeprefix('result_committed_').rsplit('_t',1)[0] for c in self.labels))
        assert self.labels == [f'result_committed_{u}_t{h}' for u in self.units for h in range(H)]
        assert len(self.units)==14 and len(self.labels)==2352
        self.reservoirs = list(self.model['reservoir'])
        self.rid = {r:i for i,r in enumerate(self.reservoirs)}
        self.plants = list(self.model['plant'])
        self.unit_plant = [self.outgoing('generator',u)[0][1] for u in self.units]
        self.pid = np.array([self.plants.index(p) for p in self.unit_plant])
        self.group = [np.flatnonzero(self.pid==k) for k in range(len(self.plants))]
        self.penstock = np.array([self.model['generator'][u]['penstock'] for u in self.units])
        self.capacity = np.array([self.model['reservoir'][r]['max_vol'] for r in self.reservoirs])
        self.pmin = np.array([self.model['generator'][u]['p_min'] for u in self.units])
        self.pmax = np.array([self.model['generator'][u]['p_max'] for u in self.units])
        self.startcost = np.array([self.model['generator'][u]['startcost_const'] for u in self.units])
        self.stopcost = np.array([self.model['generator'][u]['stopcost_const'] for u in self.units])
        self.tables = {
            'price':read_table(kernel/'Historical_day_ahead_price_2015_2025.csv'),
            'inflow':read_table(kernel/'Historical_inflow_1958_2025.csv'),
            'volume':read_table(kernel/'Historical_volume_2015_2024.csv'),
            'water_value':read_table(kernel/'Synthetic_water_value_2015_2024.csv'),
            'min_volume':read_table(extended/'Constraint_min_volume.csv'),
            'min_flow':read_table(extended/'Constraint_min_flow.csv')}
        # This constructor receives dates and supplied explanatory inputs only.
        # Labels are read separately by the training driver.
        self.up = np.zeros((14,17));self.down=np.zeros_like(self.up)
        for k,p in enumerate(self.unit_plant):
            sources=[]
            for typ,name in self.incoming('plant',p):
                if typ=='reservoir':sources.append(name)
                elif typ=='tunnel':sources.extend(n for t,n in self.incoming(typ,name) if t=='reservoir')
            assert sources
            for r in sources:self.up[k,self.rid[r]]=1/len(sources)
            dest=self.follow_destination('plant',p)
            if dest is not None:self.down[k,dest]=1
        self.catchment=np.zeros_like(self.up)
        for k in range(14):
            def collect(node,seen):
                if node in seen:return
                seen.add(node)
                if node[0]=='reservoir':self.catchment[k,self.rid[node[1]]]=1
                for t,n in self.incoming(*node):
                    if t!='generator':collect((t,n),seen)
            collect(('plant',self.unit_plant[k]),set())

    def outgoing(self,typ,name):
        return [(e['to_type'],e['to']) for e in self.edges if e['from_type']==typ and e['from']==name]

    def incoming(self,typ,name):
        return [(e['from_type'],e['from']) for e in self.edges if e['to_type']==typ and e['to']==name]

    def follow_destination(self,typ,name):
        dest=self.outgoing(typ,name)
        if not dest:return None
        assert len(dest)==1
        t,n=dest[0]
        if t=='reservoir':return self.rid[n]
        assert t=='river'
        return self.follow_destination(t,n)


class Cases:
    def __init__(self,raw,starts):
        self.raw=raw;self.starts=pd.DatetimeIndex(starts);self.n=len(starts)
        hourly=pd.DatetimeIndex((self.starts.tz_localize(None).to_numpy()[:,None]+
            np.arange(H)[None,:]*np.timedelta64(1,'h')).ravel()).tz_localize('UTC')
        def window(name,cols):
            v=raw.tables[name].reindex(hourly)[cols].to_numpy(dtype=float)
            assert np.isfinite(v).all(),name
            return v.reshape(self.n,H,len(cols))
        self.price=window('price',list(raw.tables['price']))[:,:,0]
        self.inflow=window('inflow',raw.reservoirs)
        self.river_inflow=window('inflow',['r_Vest_Vassdraget'])[:,:,0]
        self.volume=raw.tables['volume'].reindex(self.starts)[raw.reservoirs].to_numpy(dtype=float)
        self.wv=raw.tables['water_value'].reindex(self.starts+pd.Timedelta(days=7))[raw.reservoirs].to_numpy(dtype=float)
        self.min_volume=np.zeros((self.n,H,17))
        for j,r in enumerate(raw.reservoirs):
            if r in raw.tables['min_volume']:self.min_volume[:,:,j]=window('min_volume',[r])[:,:,0]
        self.flow_names=list(raw.tables['min_flow'])
        self.min_flow=window('min_flow',self.flow_names)
        assert np.isfinite(self.volume).all() and np.isfinite(self.wv).all()
        self.level=np.column_stack([np.interp(self.volume[:,j],raw.model['reservoir'][r]['vol_head']['x'],
            raw.model['reservoir'][r]['vol_head']['y']) for j,r in enumerate(raw.reservoirs)])
        self.std=np.maximum(self.price.std(axis=1),1.)
        self.mean=self.price.mean(axis=1)
        self.price_rank=pd.DataFrame(self.price).rank(axis=1,pct=True).to_numpy()
        self.rolling={}
        for width in [6,24]:
            self.rolling[width]=pd.DataFrame(self.price.T).rolling(width,center=True,min_periods=1).mean().to_numpy().T
        self.future_max=np.maximum.accumulate(self.price[:,::-1],axis=1)[:,::-1]
        self.past_max=np.maximum.accumulate(self.price,axis=1)
        self.up_wv=self.wv@raw.up.T
        self.up_volume=(self.volume/raw.capacity)@raw.up.T
        self.down_volume=(self.volume/raw.capacity)@raw.down.T
        self.up_level=self.level@raw.up.T
        self.down_level=self.level@raw.down.T
        outlet=np.array([raw.model['plant'][p]['outlet_line'] for p in raw.unit_plant])
        self.head=np.maximum(self.up_level-np.maximum(self.down_level,outlet),.1)

    def X(self,rows,variant='raw',physical=None,names_only=False):
        rows=np.asarray(rows,dtype=np.int64)
        c=rows//2352;u=(rows%2352)//168;h=rows%168;r=self.raw
        names=[];values=[]
        def add(name,value):names.append(name);values.append(np.asarray(value,dtype=np.float32))
        add('unit',u);add('plant',r.pid[u]);add('penstock',r.penstock[u])
        add('horizon',h/167);add('hour_sin',np.sin(2*np.pi*h/24));add('hour_cos',np.cos(2*np.pi*h/24))
        day=self.starts.dayofyear.to_numpy()[c]
        add('season_sin',np.sin(2*np.pi*day/365.25));add('season_cos',np.cos(2*np.pi*day/365.25))
        p=self.price[c,h];scale=self.std[c]
        add('price',p);add('price_z',(p-self.mean[c])/scale);add('price_rank',self.price_rank[c,h])
        add('week_price_mean',self.mean[c]);add('week_price_std',scale)
        add('price_previous',(p-self.price[c,np.maximum(h-1,0)])/scale)
        add('price_next',(self.price[c,np.minimum(h+1,167)]-p)/scale)
        for w in [6,24]:add(f'price_minus_mean_{w}',(p-self.rolling[w][c,h])/scale)
        add('remaining_price_peak',(self.future_max[c,h]-p)/scale)
        add('past_price_peak',(self.past_max[c,h]-p)/scale)
        add('upstream_water_value',self.up_wv[c,u]);add('price_minus_water_value',(p-self.up_wv[c,u])/scale)
        add('upstream_volume_fraction',self.up_volume[c,u]);add('downstream_volume_fraction',self.down_volume[c,u])
        add('initial_gross_head',self.head[c,u]);add('p_min',r.pmin[u]);add('p_max',r.pmax[u])
        add('start_cost_per_capacity',r.startcost[u]/r.pmax[u])
        # Supplied initial conditions for the full connected system. Constant
        # small reservoirs are omitted here but retained in water balances.
        for j,name in enumerate(r.reservoirs):
            if name not in ['Hyljelihyl','Vatjern']:
                add(f'initial_volume_{name}',self.volume[c,j]/r.capacity[j])
                add(f'terminal_water_value_{name}',self.wv[c,j])
        for label,weights in [('direct',r.up),('catchment',r.catchment)]:
            inflow=np.einsum('ij,ij->i',self.inflow[c,h],weights[u])
            add(f'{label}_inflow',inflow)
        for j,name in enumerate(r.tables['min_volume']):
            add(f'minimum_volume_{name}',self.min_volume[c,h,r.rid[name]]/r.capacity[r.rid[name]])
        for j,name in enumerate(self.flow_names):add(f'minimum_flow_{name}',self.min_flow[c,h,j])
        if variant!='raw':
            assert physical is not None
            end=12 if variant=='lp' else 24
            for j,name in enumerate(PHYSICAL_NAMES[:end]):add(name,physical[c,u,h,j])
        self.names=names
        if names_only:return names
        x=np.column_stack(values)
        assert np.isfinite(x).all()
        return x


PHYSICAL_NAMES=['relaxed_on','flow_over_minimum','plant_equivalent_units',
    'upstream_stock','downstream_stock','water_cost','price_minus_water_cost',
    'upstream_spill_over_minimum_flow','upstream_premature_spill_over_minimum_flow','startup_fraction','shutdown_fraction','lp_available',
    'rounded_on','resolved_flow_over_minimum','resolved_plant_units','resolved_upstream_stock',
    'resolved_downstream_stock','resolved_water_cost','resolved_price_minus_water_cost',
    'flow_change_over_capacity','upstream_stock_change','normalized_objective_change',
    'redispatch_available','near_rounding_boundary']


class Dispatch:
    """Hourly transport LP with local tangent tunnel laws and fixed-head PQ.

    It is deliberately a documented approximation, not a reimplementation of
    SHOP. Labels and previously trained predictors are never inputs.
    """
    def __init__(self,cases):
        self.cases=cases;self.raw=cases.raw;r=self.raw
        self.R=17;self.N=18;self.manifold=17
        self.arcs=[];self.river_indices={};self.tunnel_ids=[]
        for k,p in enumerate(r.unit_plant):
            src=np.flatnonzero(r.up[k]);source=int(src[0]) if len(src)==1 else self.manifold
            self.arcs.append(dict(name=r.units[k],source=source,target=r.follow_destination('plant',p),capacity=0,cost=0))
        for name,obj in r.model['tunnel'].items():
            source=r.rid[r.incoming('tunnel',name)[0][1]]
            typ,dest=r.outgoing('tunnel',name)[0]
            target=r.rid[dest] if typ=='reservoir' else self.manifold
            self.tunnel_ids.append(len(self.arcs))
            self.arcs.append(dict(name=name,source=source,target=target,capacity=10000,cost=0))
        for name,obj in r.model['river'].items():
            if name=='r_Vest_Vassdraget':continue # collapse this zero-storage river to Bandak
            source=r.rid[r.incoming('river',name)[0][1]];target=r.follow_destination('river',name)
            self.river_indices[name]=[]
            curve=obj.get('flow_cost_curve')
            if curve:
                segments=[(curve['x'][i+1]-curve['x'][i],curve['y'][i]) for i in range(len(curve['x'])-1)]
                segments.append((10000-curve['x'][-1],curve['y'][-1]))
            else:segments=[(float(obj.get('max_flow_const',10000)),float(obj.get('flow_cost_const',0)))]
            for cap,cost in segments:
                self.river_indices[name].append(len(self.arcs))
                self.arcs.append(dict(name=name,source=source,target=target,capacity=cap,cost=cost))
        self.A=len(self.arcs);offset=0
        def variables(shape):
            nonlocal offset
            a=np.arange(offset,offset+int(np.prod(shape))).reshape(shape);offset+=a.size;return a
        self.flow=variables((self.A,H));self.stock=variables((17,H));self.on=variables((14,H))
        self.start=variables((14,H-1));self.stop=variables((14,H-1));self.pressure=variables((H,))
        self.power=variables((14,H));self.mix=variables((14,3,H));self.nvar=offset
        rr=[];cc=[];vv=[]
        def add(rows,cols,val):
            rr.extend(np.asarray(rows).ravel());cc.extend(np.asarray(cols).ravel())
            vv.extend(np.broadcast_to(val,np.shape(cols)).ravel())
        for j in range(17):
            add(j*H+np.arange(H),self.stock[j],1)
            add(j*H+np.arange(1,H),self.stock[j,:-1],-1)
        for j,a in enumerate(self.arcs):
            add(a['source']*H+np.arange(H),self.flow[j],DT)
            if a['target'] is not None:add(a['target']*H+np.arange(H),self.flow[j],-DT)
        self.balance=sparse.coo_matrix((vv,(rr,cc)),shape=(self.N*H,self.nvar)).tocsr()
        # The first producing path is used solely for approximate terminal valuation.
        def first_plant(typ,name,seen):
            if (typ,name) in seen:return None
            if typ=='plant':return name
            seen=seen|{(typ,name)}
            candidates=[]
            for t,n in r.outgoing(typ,name):
                if t=='river' and not r.model['river'][n].get('main_river'):continue
                if t=='generator':continue
                p=first_plant(t,n,seen)
                if p is not None:candidates.append(p)
            assert len(set(candidates))<=1
            return candidates[0] if candidates else None
        self.first=[first_plant('reservoir',name,set()) for name in r.reservoirs]
        assert all(p is not None for p in self.first)

    def pq_points(self,c):
        r=self.raw;b=self.cases;points_q=[];points_p=[]
        for k,u in enumerate(r.units):
            g=r.model['generator'][u];p=r.model['plant'][r.unit_plant[k]]
            curves=sorted(g['turb_eff_curves'],key=lambda z:z['ref'])
            q=np.linspace(min(z['x'][0] for z in curves),max(z['x'][-1] for z in curves),96)
            companions=r.group[r.pid[k]]
            pipe_count=sum(r.penstock[companions]==r.penstock[k])
            loss=float(p['main_loss'][0])*(len(companions)*q)**2+float(p['penstock_loss'][r.penstock[k]-1])*(pipe_count*q)**2
            head=np.maximum(b.head[c,k]-loss,.1)
            eta=np.vstack([np.interp(q,z['x'],z['y']) for z in curves])
            eta=np.array([np.interp(head[i],[z['ref'] for z in curves],eta[:,i]) for i in range(len(q))])/100
            power=.00981*head*q*eta
            for _ in range(3):power=.00981*head*q*eta*np.interp(power,g['gen_eff_curve']['x'],g['gen_eff_curve']['y'])/100
            ids=np.flatnonzero((power>=r.pmin[k])&(power<=r.pmax[k]))
            if not len(ids):raise ValueError(f'No PQ operating point: {c}, {u}')
            best=ids[np.argmax(power[ids]/q[ids])]
            chosen=np.array([ids[0],best,ids[-1]])
            points_q.append(q[chosen]);points_p.append(power[chosen])
        return np.array(points_q),np.array(points_p)

    def problem(self,c):
        r=self.raw;b=self.cases;qp,pp=self.pq_points(c)
        beta=(pp/qp).max(axis=1);qmin=qp.min(axis=1);qmax=qp.max(axis=1)
        cvec=np.zeros(self.nvar);lo=np.zeros(self.nvar);hi=np.full(self.nvar,np.inf)
        hi[self.flow[:14]]=qmax[:,None];hi[self.power]=r.pmax[:,None]
        cvec[self.power]=-b.price[c][None,:]
        hi[self.on]=1;hi[self.start]=1;hi[self.stop]=1;hi[self.mix]=1
        cvec[self.start]=r.startcost[:,None];cvec[self.stop]=r.stopcost[:,None]
        lo[self.stock]=b.min_volume[c].T
        ceiling=np.array([r.model['reservoir'][n]['vol_head']['x'][-1] for n in r.reservoirs])
        hi[self.stock]=np.maximum(ceiling,b.volume[c])[:,None]
        level0=b.level[c];slope=[]
        for j,n in enumerate(r.reservoirs):
            curve=r.model['reservoir'][n]['vol_head'];x=np.array(curve['x']);y=np.array(curve['y'])
            i=np.clip(np.searchsorted(x,b.volume[c,j],side='right')-1,0,len(x)-2)
            slope.append((y[i+1]-y[i])/(x[i+1]-x[i]))
        slope=np.array(slope);intercept=level0-slope*b.volume[c]
        for j,a in enumerate(self.arcs[14:],14):
            hi[self.flow[j]]=a['capacity'];cvec[self.flow[j]]=a['cost']
            if j in self.tunnel_ids:lo[self.flow[j]]=-a['capacity']
        lo[self.pressure]=min(level0)-100;hi[self.pressure]=max(level0)+100
        local_energy={p:float(beta[r.group[j]].mean())/DT for j,p in enumerate(r.plants)}
        end_values=np.full(17,np.nan)
        def water_value(j,seen):
            if np.isfinite(end_values[j]):return end_values[j]
            assert j not in seen
            p=self.first[j];down=r.follow_destination('plant',p)
            end_values[j]=b.wv[c,j]*local_energy[p]+(water_value(down,seen|{j}) if down is not None else 0)
            return end_values[j]
        for j in range(17):water_value(j,set())
        cvec[self.stock[:,-1]]=-end_values
        rhs=np.zeros((18,H));rhs[:17]=b.inflow[c].T*DT;rhs[:17,0]+=b.volume[c]
        rhs[r.rid['Bandak']]+=b.river_inflow[c]*DT
        er=[];ec=[];ev=[];eb=[];ir=[];ic=[];iv=[];ib=[]
        def eq(cols,vals,bound):
            row=len(eb);er.extend([row]*len(cols));ec.extend(cols);ev.extend(vals);eb.append(bound)
        def ub(cols,vals,bound):
            row=len(ib);ir.extend([row]*len(cols));ic.extend(cols);iv.extend(vals);ib.append(bound)
        for k in range(14):
            for h in range(H):
                for variable,vals in [(self.flow[k,h],qp[k]),(self.power[k,h],pp[k]),(self.on[k,h],np.ones(3))]:
                    eq([variable]+self.mix[k,:,h].tolist(),[1]+(-vals).tolist(),0)
                if h:
                    ub([self.on[k,h],self.on[k,h-1],self.start[k,h-1]],[1,-1,-1],0)
                    ub([self.on[k,h-1],self.on[k,h],self.stop[k,h-1]],[1,-1,-1],0)
        for j,name in enumerate(b.flow_names):
            if name=='r_Vest_Vassdraget':
                # Flow is exogenous inflow plus named tributary/plant releases.
                contributors=[i for i,a in enumerate(self.arcs) if a['target']==r.rid['Bandak'] and a['name']!='Tokke_G1' and not a['name'].startswith('Tokke_')]
                for h in range(H):
                    if b.min_flow[c,h,j]>0:ub(self.flow[contributors,h].tolist(),[-1]*len(contributors),b.river_inflow[c,h]-b.min_flow[c,h,j])
            else:
                ids=self.river_indices[name]
                for h in range(H):
                    if b.min_flow[c,h,j]>0:ub(self.flow[ids,h].tolist(),[-1]*len(ids),-b.min_flow[c,h,j])
        # Spill has a convex lower bound and positive supplied cost. It is not
        # an exact binary overflow law; surplus bypass is still possible.
        for name,ids in self.river_indices.items():
            obj=r.model['river'][name]
            if 'up_head_flow_curve' in obj:
                curve=obj['up_head_flow_curve'][0];j=self.arcs[ids[0]]['source']
                gradient=(curve['y'][1]-curve['y'][0])/(curve['x'][1]-curve['x'][0])
                for h in range(H):
                    ub([self.stock[j,h]]+self.flow[ids,h].tolist(),[gradient*slope[j]]+[-1]*len(ids),gradient*(curve['x'][0]-intercept[j]))
        # The two Songa source branches meet at one zero-storage manifold.
        songa=np.flatnonzero(np.array(r.unit_plant)=='Songa')[0]
        source_tunnels=[j for j in self.tunnel_ids if self.arcs[j]['target']==17]
        def incoming_to_manifold(head):
            return sum(np.sign(level0[self.arcs[j]['source']]-head)*np.sqrt(abs(level0[self.arcs[j]['source']]-head)/float(r.model['tunnel'][self.arcs[j]['name']]['loss_factor'])) for j in source_tunnels)
        design=float(qp[songa,1])
        mh=brentq(lambda z:incoming_to_manifold(z)-design,min(level0)-1000,max(level0)+1000)
        for j in self.tunnel_ids:
            a=self.arcs[j];obj=r.model['tunnel'][a['name']];src=a['source'];dest=a['target']
            hs=max(level0[src],float(obj['start_height']))
            ht=mh if dest==17 else max(level0[dest],float(obj['end_height']))
            da=float(level0[src]>=float(obj['start_height']))
            db=1 if dest==17 else float(level0[dest]>=float(obj['end_height']))
            if not da:hi[self.flow[j]]=0
            if dest!=17 and not db:lo[self.flow[j]]=0
            alpha=float(obj['loss_factor']);dh=hs-ht
            q0=np.sign(dh)*np.sqrt(abs(dh)/alpha)
            resistance=1/(2*np.sqrt(alpha*max(abs(dh),.25)))
            shift=q0-resistance*dh
            for h in range(H):
                cols=[self.flow[j,h]];vals=[1.]
                constant=(intercept[src] if da else float(obj['start_height']))
                if da:
                    cols.append(self.stock[src,h]);vals.append(-.5*resistance*slope[src])
                    if h:cols.append(self.stock[src,h-1]);vals.append(-.5*resistance*slope[src])
                    else:constant+=.5*slope[src]*b.volume[c,src]
                if dest==17:
                    cols.append(self.pressure[h]);vals.append(resistance)
                else:
                    constant-=intercept[dest] if db else float(obj['end_height'])
                    if db:
                        cols.append(self.stock[dest,h]);vals.append(.5*resistance*slope[dest])
                        if h:cols.append(self.stock[dest,h-1]);vals.append(.5*resistance*slope[dest])
                        else:constant-=.5*slope[dest]*b.volume[c,dest]
                eq(cols,vals,shift+resistance*constant)
        equal=sparse.vstack([self.balance,sparse.coo_matrix((ev,(er,ec)),shape=(len(eb),self.nvar)).tocsr()],format='csr')
        unequal=sparse.coo_matrix((iv,(ir,ic)),shape=(len(ib),self.nvar)).tocsr()
        return dict(c=cvec/1000,A_eq=equal,b_eq=np.r_[rhs.ravel(),eb],A_ub=unequal,b_ub=np.array(ib),
            bounds=np.column_stack([lo,hi]),qp=qp,pp=pp,beta=beta)

    @staticmethod
    def optimize(p,bounds=None):
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore',message='Unrecognized options detected.*',category=OptimizeWarning)
            return linprog(p['c'],A_eq=p['A_eq'],b_eq=p['b_eq'],A_ub=p['A_ub'],b_ub=p['b_ub'],
                bounds=p['bounds'] if bounds is None else bounds,method='highs',options={'threads':1,'parallel':False})

    def features(self,c):
        tick=time.perf_counter();p=self.problem(c);result=self.optimize(p)
        record={'case':int(c),'success':bool(result.success),'status':int(result.status)}
        features=np.zeros((14,168,24),dtype=np.float32)
        if not result.success:
            record['seconds']=time.perf_counter()-tick;return features,record
        r=self.raw;b=self.cases;x=result.x
        def extract(sol):
            q=sol.x[self.flow[:14]];stock=sol.x[self.stock];on=sol.x[self.on]
            dual=-1000*sol.eqlin.marginals[:18*H].reshape(18,H)
            cost=np.zeros((14,H))
            for k,a in enumerate(self.arcs[:14]):
                cost[k]=DT*(dual[a['source']]-(dual[a['target']] if a['target'] is not None else 0))/p['beta'][k]
            count=np.empty_like(q)
            for group in r.group:count[group]=on[group].sum(axis=0)
            upstream=r.up@(stock/r.capacity[:,None]);downstream=r.down@(stock/r.capacity[:,None])
            return q,on,cost,count,upstream,downstream
        q,on,cost,count,up,down=extract(result)
        qmin=p['qp'].min(axis=1)[:,None];qmax=p['qp'].max(axis=1)[:,None]
        spill=np.zeros((17,H));premature=np.zeros_like(spill)
        for name,ids in self.river_indices.items():
            obj=r.model['river'][name]
            if 'up_head_flow_curve' not in obj:continue
            j=self.arcs[ids[0]]['source'];flow=x[self.flow[ids]].sum(axis=0)
            curve=r.model['reservoir'][r.reservoirs[j]]['vol_head']
            crest=obj['up_head_flow_curve'][0]['x'][0]
            crest_volume=np.interp(crest,curve['y'],curve['x'])
            spill[j]+=flow
            premature[j]+=np.where(x[self.stock[j]]<crest_volume-1e-5,flow,0)
        start=np.zeros_like(q);stop=np.zeros_like(q);start[:,1:]=x[self.start];stop[:,1:]=x[self.stop]
        features[:,:,:12]=np.stack([on,q/qmin,count,up,down,cost/b.std[c],
            (b.price[c][None,:]-cost)/b.std[c],(r.up@spill)/qmin,
            (r.up@premature)/qmin,start,stop,np.ones_like(q)],axis=-1)
        rounded=(q/qmin>=.5).astype(float)
        bounds=p['bounds'].copy();bounds[self.on,0]=rounded;bounds[self.on,1]=rounded
        resolved=self.optimize(p,bounds)
        record['redispatch_success']=bool(resolved.success);record['redispatch_status']=int(resolved.status)
        qr,yr,cr,nr,ur,dr=extract(resolved) if resolved.success else (q,on,cost,count,up,down)
        delta=(resolved.fun-result.fun)*1000/(r.pmax.sum()*H*b.std[c]) if resolved.success else 0.
        features[:,:,12:]=np.stack([rounded,qr/qmin,nr,ur,dr,cr/b.std[c],
            (b.price[c][None,:]-cr)/b.std[c],(qr-q)/qmax,ur-up,
            np.full_like(q,delta),np.full_like(q,float(resolved.success)),np.abs(q/qmin-.5)],axis=-1)
        residual=float(np.max(np.abs(p['A_eq']@x-p['b_eq'])))
        violated=float(np.maximum(p['A_ub']@x-p['b_ub'],0).max())
        if resolved.success:
            residual=max(residual,float(np.abs(p['A_eq']@resolved.x-p['b_eq']).max()))
            violated=max(violated,float(np.maximum(p['A_ub']@resolved.x-p['b_ub'],0).max()))
        record.update(seconds=time.perf_counter()-tick,equality_max_residual=residual,
            inequality_max_violation=violated,rounded_on_share=float(rounded.mean()),
            premature_spill_max_m3s=float(premature.max()),premature_spill_reservoir_hours=int((premature>1e-5).sum()))
        assert residual<1e-5 and violated<1e-5 and np.isfinite(features).all(),record
        return features,record

SEED=20261102
NAMES = ['dp_input_advantage_power_z', 'dp_input_advantage_cycle', 'dp_input_forward_cycle', 'dp_shadow_advantage_power_z', 'dp_shadow_advantage_cycle', 'dp_shadow_forward_cycle']

def two_state_value(reward,start_cost,stop_cost):
    """Optimal value(ON at t) minus value(OFF at t), with free initial state.

    The result is exact for this independent two-state economic problem, not
    for the coupled nonlinear watercourse. Suffix values exclude reward at t.
    """
    reward=np.asarray(reward,dtype=np.float64)
    start=np.broadcast_to(start_cost,reward.shape[:-1])
    stop=np.broadcast_to(stop_cost,reward.shape[:-1])
    f0=np.zeros_like(reward);f1=np.zeros_like(reward)
    f1[...,0]=reward[...,0]
    for t in range(1,reward.shape[-1]):
        f0[...,t]=np.maximum(f0[...,t-1],f1[...,t-1]-stop)
        f1[...,t]=reward[...,t]+np.maximum(f1[...,t-1],f0[...,t-1]-start)
    b0=np.zeros_like(reward);b1=np.zeros_like(reward)
    for t in range(reward.shape[-1]-2,-1,-1):
        b0[...,t]=np.maximum(b0[...,t+1],reward[...,t+1]-start+b1[...,t+1])
        b1[...,t]=np.maximum(b0[...,t+1]-stop,reward[...,t+1]+b1[...,t+1])
    return f1+b1-f0-b0,f1-f0

def build_profit(cases,physical,path):
    """Two input-only price/cost scenarios, six fixed dimensionless fields."""
    result=np.lib.format.open_memmap(path,mode='w+',dtype='float32',shape=(cases.n,14,168,6))
    r=cases.raw
    for begin in range(0,cases.n,64):
        end=min(cases.n,begin+64)
        price=cases.price[begin:end,None,:]
        std=cases.std[begin:end,None,None]
        given_cost=cases.up_wv[begin:end,:,None]
        shadow_cost=np.asarray(physical[begin:end,:,:,5],dtype=float)*std
        scale=r.pmax[None,:,None]*std
        cycle=np.maximum(r.startcost+r.stopcost,1)[None,:,None]
        values=[]
        for cost in [given_cost,shadow_cost]:
            margin=price-cost
            power=np.where(margin>=0,r.pmax[None,:,None],r.pmin[None,:,None])
            advantage,forward=two_state_value(margin*power,r.startcost[None,:],r.stopcost[None,:])
            values.extend([advantage/scale,advantage/cycle,forward/cycle])
        result[begin:end]=np.stack(values,axis=-1).astype('float32')
    result.flush();assert np.isfinite(result).all();return result

class ProfitCases(Cases):
    def __init__(self,raw,starts,physical,profit):
        super().__init__(raw,starts);self.physical=physical;self.profit=profit

    def X(self,rows,variant='redispatch',physical=None,names_only=False):
        rows=np.asarray(rows,dtype=np.int64)
        x=super().X(rows,variant,self.physical)
        names=self.names.copy()
        case=rows//2352;unit=(rows%2352)//168;hour=rows%168
        self.names=names+NAMES
        if names_only:return self.names
        return np.column_stack([x,self.profit[case,unit,hour]]).astype('float32')
def categorical_frame(x,names,raw):
    d=pd.DataFrame(x,columns=names)
    categories={'unit':list(range(14)),'plant':list(range(len(raw.plants))),
                'penstock':list(range(1,int(raw.penstock.max())+1))}
    for name,values in categories.items():d[name]=pd.Categorical(d[name],categories=values)
    return d

def sample_rows(cases,count,seed):
    # Row sampling occurs only inside an already isolated case-level train set.
    rng=np.random.default_rng(seed)
    n=len(cases)*2352
    local=rng.choice(n,size=min(count,n),replace=False)
    return cases[local//2352]*2352+local%2352

SEED=20261102
RELATIVE_NAMES=['downstream_value_relative','local_value_contrast_relative',
    'price_over_upstream_value','price_minus_local_value_contrast_relative']
CONTEXT_NAMES=['initial_upstream_level','receiving_reservoir_level','reservoir_level_difference',
    'direct_cumulative_natural_inflow_fraction','direct_week_natural_inflow_fraction',
    'catchment_week_natural_inflow_fraction','initial_minimum_volume_slack_fraction',
    'week_price_above_local_value_fraction','day_price_above_local_value_fraction',
    'positive_local_margin_context_24']


class CoordinateCases(ProfitCases):
    def __init__(self,raw,starts,physical,profit,mode):
        super().__init__(raw,starts,physical,profit)
        assert mode in ['relative','context','combined']
        self.mode=mode;r=raw
        self.scale=np.maximum(np.mean(np.abs(self.price),axis=1),1.)
        self.down_wv=self.wv@r.down.T
        self.direct_cap=np.maximum(r.up@r.capacity,1e-6)
        self.catchment_cap=np.maximum(r.catchment@r.capacity,1e-6)
        self.direct_natural=np.einsum('ntr,ur->nut',self.inflow,r.up)
        self.direct_cum=np.cumsum(self.direct_natural,axis=2)*.0036/self.direct_cap[None,:,None]
        self.direct_week=self.direct_natural.sum(axis=2)*.0036/self.direct_cap[None,:]
        self.catchment_week=(self.inflow.sum(axis=1)@r.catchment.T)*.0036/self.catchment_cap[None,:]
        slack=self.volume-self.min_volume.max(axis=1)
        self.storage_slack=(slack@r.up.T)/self.direct_cap[None,:]
        outlet=np.array([r.model['plant'][p]['outlet_line'] for p in r.unit_plant])
        self.receiving_level=np.where(r.down.sum(axis=1)[None,:]>0,self.down_level,outlet[None,:])
        comparison=self.price[:,None,:]>self.up_wv[:,:,None]
        self.week_above=comparison.mean(axis=2)
        self.day_above=comparison.reshape(self.n,14,7,24).mean(axis=3)
        margin=np.maximum(self.price[:,None,:]-self.up_wv[:,:,None],0)/self.scale[:,None,None]
        self.context24=np.empty_like(margin)
        for u in range(14):
            self.context24[:,u,:]=pd.DataFrame(margin[:,u,:].T).rolling(24,center=True,min_periods=1).mean().to_numpy().T

    def X(self,rows,variant='redispatch',physical=None,names_only=False):
        rows=np.asarray(rows,dtype=np.int64)
        x=super().X(rows,variant,physical);names=self.names.copy()
        c=rows//2352;u=(rows%2352)//168;h=rows%168
        r=self.raw;scale=self.scale[c];price=self.price[c,h]
        extra=[];extra_names=[]
        def add(name,v):extra_names.append(name);extra.append(np.asarray(v,dtype='float32'))
        if self.mode in ['relative','combined']:
            for j,name in enumerate(names):
                if name in ['price','week_price_mean','week_price_std','upstream_water_value','start_cost_per_capacity'] or name.startswith('terminal_water_value_'):
                    x[:,j]/=scale
                    names[j]='relative_'+name
            up=self.up_wv[c,u];down=self.down_wv[c,u]
            add(RELATIVE_NAMES[0],down/scale)
            add(RELATIVE_NAMES[1],(up-down)/scale)
            add(RELATIVE_NAMES[2],price/np.maximum(np.abs(up),.05*scale))
            # A statistical contrast, NOT an exact marginal generation cost.
            add(RELATIVE_NAMES[3],(price-up+down)/scale)
        if self.mode in ['context','combined']:
            add(CONTEXT_NAMES[0],self.up_level[c,u])
            add(CONTEXT_NAMES[1],self.receiving_level[c,u])
            # Reservoir level difference can differ from effective turbine head.
            add(CONTEXT_NAMES[2],self.up_level[c,u]-self.receiving_level[c,u])
            add(CONTEXT_NAMES[3],self.direct_cum[c,u,h])
            add(CONTEXT_NAMES[4],self.direct_week[c,u])
            add(CONTEXT_NAMES[5],self.catchment_week[c,u])
            add(CONTEXT_NAMES[6],self.storage_slack[c,u])
            add(CONTEXT_NAMES[7],self.week_above[c,u])
            add(CONTEXT_NAMES[8],self.day_above[c,u,h//24])
            add(CONTEXT_NAMES[9],self.context24[c,u,h])
        self.names=names+extra_names
        result=np.column_stack([x,*extra]).astype('float32')
        assert len(set(self.names))==len(self.names) and np.isfinite(result).all()
        if names_only:return self.names
        return result

def sample_unit(train,unit):
    n=len(train)*168
    local=np.random.default_rng(SEED+unit).choice(n,size=min(n,160000),replace=False)
    return train[local//168]*2352+unit*168+local%168
SELECTED_MODE='combined'
XGB_PARAMETERS={'colsample_bytree': 1.0,
 'device': 'cpu',
 'enable_categorical': True,
 'eval_metric': 'auc',
 'grow_policy': 'lossguide',
 'learning_rate': 0.04,
 'max_bin': 128,
 'max_cat_to_onehot': 4,
 'max_depth': 0,
 'max_leaves': 31,
 'min_child_weight': 25,
 'n_estimators': 1081,
 'n_jobs': int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1))),
 'objective': 'binary:logistic',
 'random_state': 20261102,
 'reg_lambda': 20,
 'subsample': 1.0,
 'tree_method': 'hist'}
EXPERT_PARAMETERS={'bagging_fraction': 1.0,
 'deterministic': True,
 'feature_fraction': 1.0,
 'force_col_wise': True,
 'lambda_l2': 20,
 'learning_rate': 0.04,
 'max_bin': 127,
 'max_depth': 6,
 'metric': 'auc',
 'min_data_in_leaf': 100,
 'num_leaves': 15,
 'num_threads': int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1))),
 'objective': 'binary',
 'seed': 20261102,
 'verbosity': -1}

def reproduce_coordinates(project,workspace,output):
    from concurrent.futures import ThreadPoolExecutor
    from threadpoolctl import threadpool_limits
    import xgboost as xgb
    import lightgbm as lgb
    import gc
    assert SELECTED_MODE != 'baseline', 'No coordinate candidate passed the gate.'
    threadpool_limits(int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1))))
    workspace=Path(workspace).resolve();workspace.mkdir(parents=True,exist_ok=True)
    output=Path(output).resolve();output.parent.mkdir(parents=True,exist_ok=True)
    raw=RawData(project);training=Cases(raw,raw.uc.starttime);target=Cases(raw,raw.mapping.starttime)
    def extract(cases,name):
        simulator=Dispatch(cases)
        values=np.lib.format.open_memmap(workspace/f'{name}_physical.npy',mode='w+',dtype='float32',shape=(cases.n,14,168,24))
        records=[]
        def solve(i):return i,*simulator.features(i)
        with ThreadPoolExecutor(max_workers=max(2, min(8, os.cpu_count() or 2))) as executor:
            for i,tensor,record in executor.map(solve,range(cases.n)):
                values[i]=tensor;records.append(record)
                if (i+1)%200==0:print(name,i+1,'/',cases.n,flush=True)
        values.flush();pd.DataFrame(records).to_csv(workspace/f'{name}_solver_audit.csv',index=False)
        return values
    physical=extract(training,'train');test_physical=extract(target,'test')
    dp=build_profit(training,physical,workspace/'train_profit.npy')
    test_dp=build_profit(target,test_physical,workspace/'test_profit.npy')
    f=CoordinateCases(raw,raw.uc.starttime,physical,dp,SELECTED_MODE)
    ft=CoordinateCases(raw,raw.mapping.starttime,test_physical,test_dp,SELECTED_MODE)
    Y=raw.uc[raw.labels].to_numpy(dtype=np.uint8)
    rows=sample_rows(np.arange(len(Y)),450000,SEED);x=f.X(rows);names=f.names.copy()
    model=xgb.XGBClassifier(**XGB_PARAMETERS);model.fit(categorical_frame(x,names,raw),Y.ravel()[rows])
    model.save_model(workspace/'pool.json');ids=np.arange(len(raw.sample)*2352)
    pool=np.concatenate([model.predict_proba(categorical_frame(ft.X(ids[j:j+50000]),names,raw))[:,1]
        for j in range(0,len(ids),50000)]).reshape(len(raw.sample),14,168)
    del x,model;gc.collect()
    units=np.empty_like(pool,dtype=float)
    for unit in range(14):
        rows=sample_unit(np.arange(len(Y)),unit);x=f.X(rows);yy=Y.ravel()[rows]
        target=(np.arange(len(raw.sample))[:,None]*2352+unit*168+np.arange(168)[None,:]).ravel()
        if len(np.unique(yy))==1:
            value=float((yy.sum()+1)/(len(yy)+2));units[:,unit]=value
            (workspace/f'unit_{unit}_constant.json').write_text(json.dumps(dict(probability=value)))
        else:
            ds=lgb.Dataset(x,label=yy,feature_name=f.names,categorical_feature=['unit','plant','penstock'])
            model=lgb.train(EXPERT_PARAMETERS,ds,num_boost_round=400)
            model.save_model(str(workspace/f'unit_{unit}.txt'))
            values=np.concatenate([model.predict(ft.X(target[j:j+50000]),num_threads=int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1)))) for j in range(0,len(target),50000)])
            units[:,unit]=values.reshape(len(raw.sample),168);del model,ds
        del x;gc.collect();print('Final expert',unit+1,'/14',flush=True)
    prediction=.5*(pool.astype(float)+units)
    assert np.isfinite(prediction).all() and prediction.min()>=0 and prediction.max()<=1
    frame=pd.DataFrame(prediction.reshape(len(raw.sample),2352),columns=raw.labels)
    frame.insert(0,'Run No',raw.sample['Run No'].to_numpy())
    assert frame.shape==raw.sample.shape and list(frame)==list(raw.sample) and frame['Run No'].equals(raw.sample['Run No'])
    frame.to_csv(output,index=False);print('Saved',output,frame.shape)
    return frame



class SpillOnsetDispatch(Dispatch):
    """Use the supplied crest volume and the local above-crest head slope.

    The rest of the approximate LP is inherited unchanged. This is still a
    lower-bound relaxation and can permit discretionary spill below the crest.
    """
    def problem(self,c):
        p=super().problem(c);r=self.raw;b=self.cases
        matrix=p['A_ub'].tolil(copy=True);column=p['A_ub'].tocsc()
        corrections=[]
        for name,ids in self.river_indices.items():
            obj=r.model['river'][name]
            if 'up_head_flow_curve' not in obj:continue
            j=self.arcs[ids[0]]['source'];curve=r.model['reservoir'][r.reservoirs[j]]['vol_head']
            volume=np.array(curve['x'],dtype=float);level=np.array(curve['y'],dtype=float)
            spill=obj['up_head_flow_curve'][0];crest=float(spill['x'][0])
            assert level[0]<=crest<level[-1]
            crest_volume=float(np.interp(crest,level,volume))
            interval=int(np.clip(np.searchsorted(level,crest,side='right')-1,0,len(level)-2))
            level_slope=(level[interval+1]-level[interval])/(volume[interval+1]-volume[interval])
            discharge_slope=(spill['y'][1]-spill['y'][0])/(spill['x'][1]-spill['x'][0])
            coefficient=float(level_slope*discharge_slope)
            assert coefficient>0
            old_interval=int(np.clip(np.searchsorted(volume,b.volume[c,j],side='right')-1,0,len(volume)-2))
            old_slope=(level[old_interval+1]-level[old_interval])/(volume[old_interval+1]-volume[old_interval])
            old_intercept=b.level[c,j]-old_slope*b.volume[c,j]
            old_crest_volume=float((crest-old_intercept)/old_slope)
            for h in range(H):
                flow_col=int(self.flow[ids[0],h]);stock_col=int(self.stock[j,h])
                candidates=column.indices[column.indptr[flow_col]:column.indptr[flow_col+1]]
                expected=set([stock_col]+self.flow[ids,h].tolist());matches=[]
                for row in candidates:
                    if set(matrix.rows[row])==expected:
                        values=dict(zip(matrix.rows[row],matrix.data[row]))
                        if values[stock_col]>0 and all(values[int(q)]==-1 for q in self.flow[ids,h]):matches.append(int(row))
                assert len(matches)==1,(name,h,matches)
                row=matches[0];columns=sorted(expected)
                matrix.rows[row]=columns;matrix.data[row]=[coefficient if q==stock_col else -1. for q in columns]
                p['b_ub'][row]=coefficient*crest_volume
            corrections.append(dict(river=name,reservoir=r.reservoirs[j],crest=crest,
                supplied_crest_volume=crest_volume,old_initial_tangent_crest_volume=old_crest_volume,
                above_crest_flow_per_million_m3=coefficient))
        p['A_ub']=matrix.tocsr();p['spill_onset_corrections']=corrections
        return p

def final_training_cases(raw):
    """Labelled cases whose 168-hour target window ends before the first test day (2023-01-01).
    Drops the six cases starting 26-31 December 2022, whose windows overlap the test period."""
    end = raw.uc.starttime + pd.Timedelta(hours=167)
    return np.flatnonzero((end < pd.Timestamp("2023-01-01", tz="UTC")).to_numpy())

def reproduce_corrected(project,workspace,output):
    from concurrent.futures import ThreadPoolExecutor
    from threadpoolctl import threadpool_limits
    import xgboost as xgb
    import lightgbm as lgb
    import gc
    assert SELECTED_MODE != 'baseline', 'No coordinate candidate passed the gate.'
    threadpool_limits(int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1))))
    workspace=Path(workspace).resolve();workspace.mkdir(parents=True,exist_ok=True)
    output=Path(output).resolve();output.parent.mkdir(parents=True,exist_ok=True)
    raw=RawData(project);training=Cases(raw,raw.uc.starttime);target=Cases(raw,raw.mapping.starttime)
    def extract(cases,name):
        simulator=SpillOnsetDispatch(cases)
        values=np.lib.format.open_memmap(workspace/f'{name}_physical.npy',mode='w+',dtype='float32',shape=(cases.n,14,168,24))
        records=[]
        def solve(i):return i,*simulator.features(i)
        with ThreadPoolExecutor(max_workers=max(2, min(8, os.cpu_count() or 2))) as executor:
            for i,tensor,record in executor.map(solve,range(cases.n)):
                values[i]=tensor;records.append(record)
                if (i+1)%200==0:print(name,i+1,'/',cases.n,flush=True)
        values.flush();pd.DataFrame(records).to_csv(workspace/f'{name}_solver_audit.csv',index=False)
        return values
    physical=extract(training,'train');test_physical=extract(target,'test')
    dp=build_profit(training,physical,workspace/'train_profit.npy')
    test_dp=build_profit(target,test_physical,workspace/'test_profit.npy')
    f=CoordinateCases(raw,raw.uc.starttime,physical,dp,SELECTED_MODE)
    ft=CoordinateCases(raw,raw.mapping.starttime,test_physical,test_dp,SELECTED_MODE)
    Y=raw.uc[raw.labels].to_numpy(dtype=np.uint8)
    rows=sample_rows(final_training_cases(raw),450000,SEED);x=f.X(rows);names=f.names.copy()
    model=xgb.XGBClassifier(**XGB_PARAMETERS);model.fit(categorical_frame(x,names,raw),Y.ravel()[rows])
    model.save_model(workspace/'pool.json');ids=np.arange(len(raw.sample)*2352)
    pool=np.concatenate([model.predict_proba(categorical_frame(ft.X(ids[j:j+50000]),names,raw))[:,1]
        for j in range(0,len(ids),50000)]).reshape(len(raw.sample),14,168)
    del x,model;gc.collect()
    units=np.empty_like(pool,dtype=float)
    for unit in range(14):
        rows=sample_unit(final_training_cases(raw),unit);x=f.X(rows);yy=Y.ravel()[rows]
        target=(np.arange(len(raw.sample))[:,None]*2352+unit*168+np.arange(168)[None,:]).ravel()
        if len(np.unique(yy))==1:
            value=float((yy.sum()+1)/(len(yy)+2));units[:,unit]=value
            (workspace/f'unit_{unit}_constant.json').write_text(json.dumps(dict(probability=value)))
        else:
            ds=lgb.Dataset(x,label=yy,feature_name=f.names,categorical_feature=['unit','plant','penstock'])
            model=lgb.train(EXPERT_PARAMETERS,ds,num_boost_round=400)
            model.save_model(str(workspace/f'unit_{unit}.txt'))
            values=np.concatenate([model.predict(ft.X(target[j:j+50000]),num_threads=int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1)))) for j in range(0,len(target),50000)])
            units[:,unit]=values.reshape(len(raw.sample),168);del model,ds
        del x;gc.collect();print('Final expert',unit+1,'/14',flush=True)
    prediction=.5*(pool.astype(float)+units)
    assert np.isfinite(prediction).all() and prediction.min()>=0 and prediction.max()<=1
    frame=pd.DataFrame(prediction.reshape(len(raw.sample),2352),columns=raw.labels)
    frame.insert(0,'Run No',raw.sample['Run No'].to_numpy())
    assert frame.shape==raw.sample.shape and list(frame)==list(raw.sample) and frame['Run No'].equals(raw.sample['Run No'])
    frame.to_csv(output,index=False);print('Saved',output,frame.shape)
    return frame

SPILL_RECIPE='corrected'

def reproduce_spill_selected(project,workspace,output):
    workspace=Path(workspace).resolve();workspace.mkdir(parents=True,exist_ok=True)
    output=Path(output).resolve();output.parent.mkdir(parents=True,exist_ok=True)
    assert SPILL_RECIPE in ['corrected','old_corrected_half'], 'No new candidate passed the fixed gate.'
    if SPILL_RECIPE=='corrected':
        return reproduce_corrected(project,workspace/'corrected',output)
    reproduce_coordinates(project,workspace/'baseline',workspace/'baseline.csv')
    reproduce_corrected(project,workspace/'corrected',workspace/'corrected.csv')
    old=pd.read_csv(workspace/'baseline.csv');new=pd.read_csv(workspace/'corrected.csv')
    assert list(old)==list(new) and old['Run No'].equals(new['Run No'])
    old.iloc[:,1:]=.5*(old.iloc[:,1:].to_numpy()+new.iloc[:,1:].to_numpy())
    old.to_csv(output,index=False);return old

In [ ]:
%%writefile work_nb2/run_spill.py
"""Stage: Pipeline B (v1) , trained on the 2,916 strict cases."""
import os
from pathlib import Path
from spill_onset import *
WORK = Path(os.environ["V12C_WORK"])
reproduce_spill_selected(Path(".").resolve(), WORK / "spill", WORK / "test_pred_spill.csv")

In [ ]:
run_stage(["test_pred_spill.csv"], ["run_spill.py"])

## Pipeline A: A-LightGBM (weight 0.55)
Features = hydro features + plant LP + unit LP + 16 P–Q LP channels (172 columns).
A pooled LightGBM (all units, 63 leaves, 1.5M sampled rows) and one LightGBM per unit (31 leaves) are averaged 50/50.
The numbers of boosting rounds were found by early stopping on 2021 (trained on cases that end before 2021) and are fixed here.
**Training cases:** the 2,916 cases whose 168-hour window ends before 1 January 2023. The six cases that start on 26–31 December 2022 are left out,
because their labels reach into the test period.

In [ ]:
%%writefile work_nb2/run_trees.py
"""Stage 3: pooled + per-unit LightGBM on rich hydro features + both LP feature sets. Fixed rounds (chosen on 2021)."""
import os, sys, json
from hydro_base import *
WORK = Path(os.environ["V12C_WORK"])
z = np.load(WORK / "lp_feats_all.npz", allow_pickle=True)
LPC = {pd.Timestamp(str(s), tz="UTC"): (q, w) for s, q, w in zip(z["starts"], z["Q"], z["W"])}
u_ = np.load(WORK / "lp_unit_all.npz", allow_pickle=True)
LPU = {pd.Timestamp(str(s), tz="UTC"): (a, b) for s, a, b in zip(u_["starts"], u_["Z"], u_["S"])}; UORD = list(u_["order"])
USE_ULP = True
USE_GPQ = True
USE_ECON = False
def econ_adv(revenue, startup, shutdown):     # vectorised over the leading axes; DP over hours
    fw = np.zeros(revenue.shape + (2,)); bw = np.zeros_like(fw); fw[..., 0, 1] = revenue[..., 0]
    for t in range(1, revenue.shape[-1]):
        fw[..., t, 0] = np.maximum(fw[..., t-1, 0], fw[..., t-1, 1] - shutdown); fw[..., t, 1] = revenue[..., t] + np.maximum(fw[..., t-1, 1], fw[..., t-1, 0] - startup)
    for t in range(revenue.shape[-1] - 2, -1, -1):
        bw[..., t, 0] = np.maximum(bw[..., t+1, 0], bw[..., t+1, 1] + revenue[..., t+1] - startup); bw[..., t, 1] = np.maximum(bw[..., t+1, 0] - shutdown, bw[..., t+1, 1] + revenue[..., t+1])
    tot = fw + bw; return tot[..., 1] - tot[..., 0]
if USE_GPQ:
    _tr = np.load(WORK / "legacy/confidence/pq_train_features.npy", mmap_mode="r"); _te = np.load(WORK / "legacy/confidence/pq_test_features.npy", mmap_mode="r")
    GPQ_ALL = np.concatenate([np.asarray(_tr), np.asarray(_te)])
    GPQ_IDX = {pd.Timestamp(str(s_)) if pd.Timestamp(str(s_)).tzinfo else pd.Timestamp(str(s_), tz="UTC"): i for i, s_ in enumerate(list(uc.starttime) + list(sample[["Run No"]].merge(mapping, on="Run No", how="left").starttime))}
PL = ['Hogga', 'Tokke', 'Lio', 'Byrte', 'Vinje', 'Haukeli', 'Songa', 'Kjela', 'Vesle Kjela']; NU = {'Tokke': 4, 'Vinje': 3}
class RichLP(RichHydroFeatures):
    def __init__(self, *a):
        super().__init__(*a)
        Q = np.stack([LPC[s][0] for s in self.starts]); W = np.stack([LPC[s][1] for s in self.starts])   # (n, 9, H)
        pk = np.array([PL.index(r["plant"]) for r in self.topology]); self.pk = pk
        self.nu = np.array([NU.get(r["plant"], 1) for r in self.topology]); self.pos = np.array([int(u.rsplit("_G", 1)[1]) - 1 for u in self.units])
        self.lq = Q; self.lw = W
        Zs = np.stack([LPU[s][0] for s in self.starts]); Ss = np.stack([LPU[s][1] for s in self.starts])
        ui = np.array([UORD.index(u) for u in self.units]); self.uz = Zs[:, ui]; self.us = Ss[:, ui]   # (n, 14, H) in unit order
        r = lambda a, w: pd.DataFrame(a.reshape(-1, H)).T.rolling(w, center=True, min_periods=1).mean().T.values.reshape(a.shape)
        self.uz5, self.uz25 = r(self.uz, 5), r(self.uz, 25)
        self.ustart7 = r(self.us, 7) * 7; self.ushut7 = r(np.maximum(np.concatenate([np.zeros_like(self.uz[:, :, :1]), -np.diff(self.uz, axis=2)], 2), 0), 7) * 7
        if USE_GPQ: self.gpq = GPQ_ALL[[GPQ_IDX[s_] for s_ in self.starts]]      # (n, 14, H, 16), unit order = labels
        if USE_ECON:
            stop = np.array([config["model"]["generator"][u_]["stopcost_const"] if "stopcost_const" in config["model"]["generator"][u_] else 0 for u_ in self.units], float)
            margin = (self.p[:, None, :] - self.upwv[:, :, None]) * self.pmax[None, :, None]
            self.e_margin = margin.astype(np.float32); self.e_adv = econ_adv(margin, self.startcost[None, :], stop[None, :]).astype(np.float32); self.e_cyc = np.maximum(self.startcost + stop, 1)
        plant_of_unit = np.array([r_["plant"] for r_ in self.topology])
        self.uplant = np.stack([self.uz[:, plant_of_unit == plant_of_unit[k]].sum(1) for k in range(len(self.units))], 1)
        self.lq5 = pd.DataFrame(Q.reshape(-1, H)).T.rolling(5, center=True, min_periods=1).mean().T.values.reshape(Q.shape)
        self.lq25 = pd.DataFrame(Q.reshape(-1, H)).T.rolling(25, center=True, min_periods=1).mean().T.values.reshape(Q.shape)
    def X(self, ids, mode="rich"):
        base = super().X(ids); names = self.names.copy()
        ids = np.asarray(ids); c, l = ids // self.L, ids % self.L; u, t = self.ku[l], self.ht[l]; k = self.pk[u]
        q = self.lq[c, k, t]; w = self.lw[c, k, t]
        extra = [q, q * self.nu[u] - self.pos[u], self.lq5[c, k, t], self.lq25[c, k, t], self.lq[c, k].mean(axis=1),
                 w / self.pscale[c], (self.p[c, t] - w) / self.std[c]]
        names += ["lp_q", "lp_units", "lp_q_roll5", "lp_q_roll25", "lp_q_week", "lp_breakeven_rel", "p_minus_lp_breakeven_z"]
        for nm in ("Tokke", "Vinje", "Hogga", "Songa"):
            extra.append(self.lq[c, PL.index(nm), t]); names.append("lp_q_" + nm)
        if USE_ULP:
            zz = self.uz[c, u]
            extra += [zz[np.arange(len(t)), t], zz[np.arange(len(t)), np.maximum(t - 1, 0)], zz[np.arange(len(t)), np.minimum(t + 1, H - 1)],
                      self.uz5[c, u, t], self.uz25[c, u, t], zz.mean(axis=1), self.ustart7[c, u, t], self.ushut7[c, u, t], self.uplant[c, u, t]]
            names += ["ulp_z", "ulp_z_prev", "ulp_z_next", "ulp_z_roll5", "ulp_z_roll25", "ulp_z_week", "ulp_start_near", "ulp_shut_near", "ulp_plant_units"]
        if USE_ECON:
            den = np.maximum(self.pmax[u] * self.std[c], 1); a_ = self.e_adv[c, u, t]
            extra += [self.e_margin[c, u, t] / den, a_ / den, np.sign(a_) * np.log1p(np.abs(a_)), a_ / self.e_cyc[u]]
            names += ["econ_margin_z", "econ_adv_z", "econ_adv_signlog", "econ_adv_cycle"]
        if USE_GPQ:
            g_ = self.gpq[c, u, t]                                   # (rows, 16)
            for i_ in range(16): extra.append(g_[:, i_]); names.append(f"gpq_{i_}")
        self.names = names
        return np.column_stack([base] + [np.asarray(e, dtype=np.float32) for e in extra])
tf = RichLP(uc.starttime, series, config, units, ku, ht); tf.X(np.arange(5)); print("features:", len(tf.names), flush=True)
ROUNDS = {"pooled": 1251, "Hogga_G1": 489, "Tokke_G1": 481, "Tokke_G2": 437, "Tokke_G3": 379, "Tokke_G4": 341, "Lio_G1": 192, "Byrte_G1": 204, "Vinje_G1": 483, "Vinje_G2": 562, "Vinje_G3": 562, "Haukeli_G1": 140, "Songa_G1": 409, "Kjela_G1": 590, "Vesle Kjela_G1": 626}   # chosen by early stopping on 2021 (train = cases ending before 2021)
test_starts = sample[["Run No"]].merge(mapping, on="Run No", how="left").starttime
tt = RichLP(test_starts, series, config, units, ku, ht)
def ending_before(y): return np.flatnonzero((uc.starttime + pd.Timedelta(hours=H - 1) < pd.Timestamp(f"{y}-01-01", tz="UTC")).to_numpy())
FINAL_TRAIN = ending_before(2023)   # 2,916 cases: drops the 6 cases (26-31 Dec 2022) whose windows reach into the test period
pt, _ = tree_reference(tf, FINAL_TRAIN, tt, np.arange(len(test_starts)), rounds=ROUNDS, final=True)
np.save(WORK / "test_pred_trees.npy", pt); print("saved trees", pt.shape)

In [ ]:
run_stage(["test_pred_trees.npy"], ["run_trees.py"])

## Pipeline A: A-TCN (weight 0.10)
Each case becomes a tensor: shared channels (price, calendar, system state) and 26 channels per unit (including 5 LP channels), over 168 hours.
One network with dilated 1-D convolutions (dilations 1 to 32) is shared by all units; a unit embedding tells it which unit it is looking at.
Scaling is fitted on the training cases only. 18 epochs (chosen on 2021). Same 2,916 training cases as the trees.

In [ ]:
%%writefile work_nb2/tcn_model.py
"""Shared-unit temporal convolutional network (TCN).
Runs as its own process so that PyTorch and LightGBM never load two OpenMP runtimes in one process (macOS).
Usage: python tcn_model.py --root <folder with train_X.npy ...> --config <json>"""
import os, sys, json, time, random, argparse
from pathlib import Path
# Thread count is inherited from the parent (run_tcn.py); fall back to all CPUs.
_T = str(max(1, os.cpu_count() or 1))
os.environ.setdefault("OMP_NUM_THREADS", _T)
os.environ.setdefault("MKL_NUM_THREADS", _T)
os.environ.setdefault("OPENBLAS_NUM_THREADS", _T)
import numpy as np, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

class Block(nn.Module):
    """residual block of two dilated convolutions over the 168 hours"""
    def __init__(self, width, d):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv1d(width, width, 3, padding=d, dilation=d), nn.GroupNorm(8, width), nn.GELU(),
                                    nn.Dropout(0.15), nn.Conv1d(width, width, 3, padding=d, dilation=d),
                                    nn.GroupNorm(8, width), nn.GELU())
    def forward(self, x): return x + 0.35 * self.layers(x)

class SharedUnitTCN(nn.Module):
    """one network for all units; unit identity enters through an embedding and a per-unit bias.
    Dilations 1..32 give every hour a receptive field over the whole week."""
    def __init__(self, shared, local, units=14):
        super().__init__(); self.shared = shared
        self.context = nn.Sequential(nn.Conv1d(shared, 32, 1), nn.GELU())
        self.local = nn.Sequential(nn.Conv1d(local, 32, 1), nn.GELU())
        self.unit = nn.Embedding(units, 8); self.bias = nn.Embedding(units, 1)
        self.stem = nn.Sequential(nn.Conv1d(72, 64, 1), nn.GELU())
        self.temporal = nn.Sequential(*[Block(64, d) for d in [1, 2, 4, 8, 16, 32]])
        self.head = nn.Conv1d(64, 1, 1); self.direct = nn.Conv1d(shared + local + 8, 1, 1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)
        nn.init.normal_(self.direct.weight, std=0.01); nn.init.zeros_(self.direct.bias)
    def forward(self, x, unit):
        common, local = x[:, :self.shared], x[:, self.shared:]
        embed = self.unit(unit)[:, :, None].expand(-1, -1, x.shape[2])
        hidden = self.stem(torch.cat([self.context(common), self.local(local), embed], dim=1))
        return (self.direct(torch.cat([common, local, embed], dim=1)) + self.head(self.temporal(hidden))
                + self.bias(unit)[:, :, None])

def signed_log(x): return np.sign(x) * np.log1p(np.abs(x))

def normalizer(X, indices, shared, local, units):
    """mean / std of every input channel, computed on the training cases only"""
    a = np.zeros(shared); aa = a.copy(); b = np.zeros(local); bb = b.copy(); na = nb = 0
    for i in range(0, len(indices), 32):
        x = np.asarray(X[indices[i:i + 32]], dtype=np.float64)
        c = x[:, :shared]; l = signed_log(x[:, shared:].reshape(len(x), units, local, x.shape[2]))
        a += c.sum(axis=(0, 2)); aa += (c * c).sum(axis=(0, 2)); na += len(x) * x.shape[2]
        b += l.sum(axis=(0, 1, 3)); bb += (l * l).sum(axis=(0, 1, 3)); nb += len(x) * units * x.shape[2]
    mean = np.r_[a / na, b / nb]; std = np.sqrt(np.maximum(np.r_[aa / na, bb / nb] - mean * mean, 1e-6))
    return mean.astype(np.float32), std.astype(np.float32)

class UnitCases(Dataset):
    """item = one (case, unit): shared channels + that unit's local channels, 168 hours"""
    def __init__(self, X, indices, mean, std, shared, local, units, Y=None):
        self.X, self.indices, self.mean, self.std = X, np.asarray(indices), mean, std
        self.shared, self.local, self.units, self.Y = shared, local, units, Y
    def __len__(self): return len(self.indices) * self.units
    def __getitem__(self, i):
        case = int(self.indices[i // self.units]); u = i % self.units
        common = np.asarray(self.X[case, :self.shared])
        local = signed_log(np.asarray(self.X[case, self.shared + u * self.local:self.shared + (u + 1) * self.local]))
        x = np.clip((np.concatenate([common, local]) - self.mean[:, None]) / self.std[:, None], -8, 8).astype(np.float32)
        if self.Y is None: return torch.from_numpy(x), u
        return torch.from_numpy(x), u, torch.from_numpy(np.array(self.Y[case, u:u + 1], dtype=np.float32, copy=True))

def predict(model, X, indices, mean, std, shared, local, units, device):
    loader = DataLoader(UnitCases(X, indices, mean, std, shared, local, units), batch_size=128, shuffle=False)
    model.eval(); parts = []
    with torch.inference_mode():
        for x, u in loader: parts.append(torch.sigmoid(model(x.to(device), u.to(device))).cpu().numpy())
    return np.concatenate(parts).reshape(len(indices), units, -1)

def run():
    ap = argparse.ArgumentParser(); ap.add_argument("--root"); ap.add_argument("--config"); args = ap.parse_args()
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", max(1, os.cpu_count() or 1)))); torch.set_num_interop_threads(1)
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    SEED = int(json.loads(Path(args.config).read_text()).get("seed", 4173)); torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    print("PyTorch", torch.__version__, "device", device, flush=True)
    root = Path(args.root); cfg = json.loads(Path(args.config).read_text())
    X = np.load(root / "train_X.npy", mmap_mode="r"); Y = np.load(root / "train_Y.npy", mmap_mode="r")
    units = Y.shape[1]; layout = json.loads((root / "feature_layout.json").read_text())
    shared, local = len(layout["shared"]), len(layout["local"])
    train = np.array(cfg["train"]); valid = np.array(cfg.get("valid", []), dtype=int)
    mean, std = normalizer(X, train, shared, local, units)
    model = SharedUnitTCN(shared, local, units).to(device)
    prior = np.clip(np.asarray(Y[train].mean(axis=(0, 2)), dtype=np.float32), 0.001, 0.999)
    with torch.no_grad(): model.bias.weight.copy_(torch.from_numpy(np.log(prior / (1 - prior))[:, None]).to(device))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    epochs = int(cfg["epochs"]); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=3e-5)
    loader = DataLoader(UnitCases(X, train, mean, std, shared, local, units, Y), batch_size=128, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))
    loss_fn = nn.BCEWithLogitsLoss(); history = []; best = -1; best_epoch = 0; best_state = None; wait = 0
    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter(); model.train(); total = seen = 0
        for x, u, y in loader:
            x, u, y = x.to(device), u.to(device), y.to(device)
            opt.zero_grad(set_to_none=True); loss = loss_fn(model(x, u), y); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            total += float(loss.detach().cpu()) * len(x); seen += len(x)
        sched.step(); row = dict(epoch=epoch, train_bce=round(total / seen, 5), seconds=round(time.perf_counter() - t0, 1))
        if len(valid):     # early stopping on the selection year only
            p = predict(model, X, valid, mean, std, shared, local, units, device)
            row["validation_auc"] = round(roc_auc_score(Y[valid].ravel(), p.ravel()), 5)
            if row["validation_auc"] > best:
                best, best_epoch, wait = row["validation_auc"], epoch, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else: wait += 1
        history.append(row); print(json.dumps(row), flush=True)
        if len(valid) and wait >= int(cfg.get("patience", 10)): break
    if len(valid): model.load_state_dict(best_state)
    else: best_epoch = epochs
    target = np.load(root / cfg["predict_file"], mmap_mode="r")
    p = predict(model, target, np.array(cfg["predict"]), mean, std, shared, local, units, device)
    np.save(root / f"{cfg['name']}_prediction.npy", p)
    (root / f"{cfg['name']}_summary.json").write_text(json.dumps(dict(epoch=best_epoch, validation_auc=best, device=str(device))))
    print("FINISHED", cfg["name"], "best epoch", best_epoch, "prediction", p.shape, flush=True)

if __name__ == "__main__": run()

In [ ]:
%%writefile work_nb2/run_tcn.py
"""Stage 4: sequence tensors (127 shared + 14 x (21 + 5 LP) local channels) and the shared-unit TCN, 18 epochs on all labelled cases."""
import os, sys, json, subprocess, shutil
from hydro_base import *
ROOT = Path(os.environ["V12C_WORK"]); WORK = ROOT / "tcn"; WORK.mkdir(exist_ok=True)
def temporal_layout(features):
    """channels of the sequence tensor: 'shared' (price, calendar, whole-system state; same for every unit)
    and 'local' (per-unit economics and water state; one block per unit)"""
    features.X(np.arange(10))
    shared = ["t", "hour", "weekday", "month", "season_sin", "season_cos", "price_week_rank", "price_day_rank",
              "price_z", "price_relative_mean", "price_delta_prev", "price_delta_next", "week_volatility",
              "river_natural_inflow", "river_week_natural_mean"]
    prefixes = ["storage_", "wv_relative_", "week_natural_inflow_capacity_", "water_value_trend_",
                "initial_level_", "minflow_", "tunnel_level_difference_", "tunnel_pressure_flow_scenario_"]
    shared += [n for n in features.names if any(n.startswith(p) for p in prefixes) and n != "storage_slack_fraction"]
    shared = list(dict.fromkeys(shared))
    local = ["price_minus_upwv_z", "price_minus_netwv_z", "price_over_upwv", "upwv_relative", "downwv_relative",
             "feed_storage_fraction", "down_storage_fraction", "week_inflow_capacity", "cumulative_natural_inflow_fraction",
             "hour_inflow_fraction_qmax", "catchment_storage_fraction", "plant_week_water_budget",
             "price_minus_combined_dispatch_threshold_z", "initial_net_head", "feeding_tunnel_flow_relative_qmax",
             "volume_with_upstream_release_scenario_fraction", "intake_head_fullflow_scenario",
             "scenario_overflow_pressure", "day_profitable_fraction", "positive_margin_context_6", "positive_margin_context_24"]
    assert set(shared + local) <= set(features.names)
    return shared, local

def write_temporal_inputs(features, path, layout):
    """(cases, shared + 14 x local channels, 168 hours) float32 tensor on disk"""
    shared, local = layout
    features.X(np.arange(10)); names = features.names.copy()
    si = [names.index(n) for n in shared]; li = [names.index(n) for n in local]; U = len(features.units)
    assert np.array_equal(features.ku, np.repeat(np.arange(U), H)) and np.array_equal(features.ht, np.tile(np.arange(H), U))
    out = np.lib.format.open_memmap(path, mode="w+", dtype="float32", shape=(features.n, len(shared) + U * len(local), H))
    for a in range(0, features.n, 16):
        b = min(a + 16, features.n)
        x = features.X(np.arange(a * features.L, b * features.L)).reshape(b - a, U, H, -1)
        common = np.take(x[:, 0], si, axis=2)
        loc = np.take(x, li, axis=3).transpose(0, 1, 3, 2).reshape(b - a, -1, H)
        out[a:b] = np.concatenate([common.transpose(0, 2, 1), loc], axis=1)
    out.flush(); print("sequence tensor:", out.shape, flush=True); del out


if not (WORK / "test_X.npy").exists():
    net_features = NetworkPressureFeatures(uc.starttime, series, config, units, ku, ht)
    LAYOUT = temporal_layout(net_features)
    write_temporal_inputs(net_features, WORK / "base_train_X.npy", LAYOUT); del net_features; gc.collect()
    test_starts = sample[["Run No"]].merge(mapping, on="Run No", how="left", validate="one_to_one").starttime
    write_temporal_inputs(NetworkPressureFeatures(test_starts, series, config, units, ku, ht), WORK / "base_test_X.npy", LAYOUT)
    np.save(WORK / "train_Y.npy", Y.reshape(len(uc), len(units), H).astype("float32"))
    # add 5 LP channels per unit (plant LP flow, implied number of units on, unit LP on-state and two smoothings)
    z = np.load(ROOT / "lp_feats_all.npz", allow_pickle=True); u = np.load(ROOT / "lp_unit_all.npz", allow_pickle=True)
    PL = ['Hogga','Tokke','Lio','Byrte','Vinje','Haukeli','Songa','Kjela','Vesle Kjela']
    plant = lambda g: g.rsplit("_", 1)[0]; uord = list(u["order"]); ui = [uord.index(g) for g in units]; pk = [PL.index(plant(g)) for g in units]
    NU = {"Tokke": 4, "Vinje": 3}; pos = np.array([int(g.rsplit("_G", 1)[1]) - 1 for g in units]); nu = np.array([NU.get(plant(g), 1) for g in units])
    roll = lambda a, w: pd.DataFrame(a.reshape(-1, H)).T.rolling(w, center=True, min_periods=1).mean().T.values.reshape(a.shape)
    S, Lc = len(LAYOUT[0]), len(LAYOUT[1])
    for name, sl in (("train", slice(0, len(uc))), ("test", slice(len(uc), None))):
        X = np.load(WORK / f"base_{name}_X.npy", mmap_mode="r"); n = X.shape[0]
        q = z["Q"][sl][:, pk]; zz = u["Z"][sl][:, ui]
        new = np.stack([q, q * nu[None, :, None] - pos[None, :, None], zz, roll(zz, 5), roll(q, 25)], axis=2).astype(np.float32)
        out = np.lib.format.open_memmap(WORK / f"{name}_X.npy", mode="w+", dtype="float32", shape=(n, S + 14 * (Lc + 5), H))
        for a in range(0, n, 64):
            b = min(a + 64, n); x = np.asarray(X[a:b]); loc = x[:, S:].reshape(b - a, 14, Lc, H)
            out[a:b] = np.concatenate([x[:, :S], np.concatenate([loc, new[a:b]], axis=2).reshape(b - a, -1, H)], axis=1)
        out.flush(); print(name, out.shape, flush=True); del X, out
        os.remove(WORK / f"base_{name}_X.npy")
    (WORK / "feature_layout.json").write_text(json.dumps(dict(shared=LAYOUT[0], local=LAYOUT[1] + ["lp_q", "lp_units", "ulp_z", "ulp_z_roll5", "lp_q_roll25"])))
FINAL_TRAIN = np.flatnonzero((uc.starttime + pd.Timedelta(hours=H - 1) < pd.Timestamp("2023-01-01", tz="UTC")).to_numpy())   # 2,916 cases
SEEDS = [int(s) for s in os.environ.get("TCN_SEEDS", "4173").split(",")]
# Inherit the thread count set by run_stage; do not override.
env = os.environ.copy()
env.setdefault("OMP_NUM_THREADS", str(max(1, os.cpu_count() or 1)))
env.setdefault("MKL_NUM_THREADS", env["OMP_NUM_THREADS"])
env.setdefault("OPENBLAS_NUM_THREADS", env["OMP_NUM_THREADS"])
preds = []
for seed in SEEDS:   # same network and settings, different random initialisation and batch order
    name = f"final_s{seed}"
    if not (WORK / f"{name}_prediction.npy").exists():
        cfg = dict(name=name, train=FINAL_TRAIN.tolist(), epochs=18, patience=10, valid=[], predict=list(range(len(sample))), predict_file="test_X.npy", seed=seed)
        (WORK / f"{name}_config.json").write_text(json.dumps(cfg))
        subprocess.run([sys.executable, str(ROOT / "tcn_model.py"), "--root", str(WORK), "--config", str(WORK / f"{name}_config.json")], env=env, check=True)
    preds.append(np.load(WORK / f"{name}_prediction.npy"))
np.save(ROOT / "test_pred_tcn.npy", np.mean(preds, axis=0)); print("saved TCN, mean of seeds", SEEDS)

In [ ]:
run_stage(["test_pred_tcn.npy"], ["run_tcn.py"])

## Rank average and the submission file
Each of the three prediction branches is converted to percentile ranks. The rank vectors are combined with weights 0.55, 0.35 and 0.10. AUC evaluates the ordering of these scores.


In [ ]:
sample = pd.read_csv(PROJECT / "sample_submission.csv"); cols = sample.columns[1:]
rank = lambda x: pd.Series(np.ravel(x)).rank(pct=True).to_numpy()
trees = np.load(WORK / "test_pred_trees.npy"); tcn = np.load(WORK / "test_pred_tcn.npy")
spill = pd.read_csv(WORK / "test_pred_spill.csv").set_index("Run No").loc[sample["Run No"], cols].to_numpy()
assert trees.size == tcn.size == spill.size == len(sample) * len(cols)
for name, p in [("trees", trees), ("Pipeline B (v1)", spill), ("tcn", tcn)]:
    print(f"{name:12s} mean {np.mean(p):.4f}  rank correlation with trees {np.corrcoef(rank(p), rank(trees))[0, 1]:.4f}")
prob = (0.55 * rank(trees) + 0.35 * rank(spill) + 0.10 * rank(tcn)).reshape(len(sample), -1)
assert np.isfinite(prob).all() and prob.min() >= 0 and prob.max() <= 1
submission = pd.DataFrame(prob, columns=cols); submission.insert(0, "Run No", sample["Run No"].to_numpy())
assert list(submission.columns) == list(sample.columns)
submission.to_csv(PROJECT / "submission_nb2_v12c.csv", index=False); print("saved", submission.shape)

## Limitations
The LPs are simplified (relaxed on/off, linear head and P–Q approximations), so they give features, not decisions, and the final probabilities are not forced to obey the physical limits.
The weights were chosen on 2021–2022, which were also used during development, so the validation scores are a little optimistic.
The LP solver can return different but equally good solutions with other SciPy versions, so a rerun on another machine can differ a little in the last digits.